# Occlusion Attribution Analysis — Leukemia Cell Classification

This notebook analyses **occlusion attributions** generated by a DenseNet-121 model trained to distinguish  
AML (Acute Myeloid Leukemia) blasts from normal bone-marrow / peripheral-blood cells.

The dataset is a per-image table (`merged_predictions_occlusion_features.csv`) that merges:
- **Occlusion maps**: per-ring attribution statistics (8 radial rings, boundary→centre) and global stats
- **Cell morphology**: shape, intensity, texture, and radial-profile features extracted from segmentation masks
- **Patient annotations**: clinical diagnosis, blast percentage, genetic mutations, sample source


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import seaborn as sns
from scipy.stats import mannwhitneyu, spearmanr, pearsonr, kruskal
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from pathlib import Path

import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'
plt.rc('font', size=8) #set the default font size to be 8

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', font_scale=0.95)

CSV_PATH_AML_CT = (
    '/path/to/ainu_root/results/occlusion_analysis/ainu_split3_model_best.pth.tar/PadResize128_GD_r03_u4-32_p07_grey_blackbg/both/merged_predictions_AML_occlusion_features.csv'
)

CSV_PATH_MDS_MDR = (
    '/path/to/ainu_root/results/occlusion_analysis/ainu_split3_model_best.pth.tar/ainu_split3_MRD_MDS/MDS_AML_occlusion_features.csv'
)
COLORS   = {'AML': '#B21F2C', 'Control': '#2367AC', 'Preleukemia': '#F3A582', 'MRD': '#93C5DE'}
# Diverging colormap: Control-blue → white → AML-red; ends match class scatter colors
CMAP_OCC = LinearSegmentedColormap.from_list(
    'occ', [COLORS['Control'], 'white', COLORS['AML']])
CLASSES = ['Control',"Preleukemia",'AML']
positive_name = 'AML'
negative_name = 'Control'
N_RINGS  = 8
RING_X   = np.arange(N_RINGS)
RING_LBL = ['R0\n(boundary)'] + [f'R{i}' for i in range(1, 7)] + ['R7\n(centre)']

# ── column groups ──────────────────────────────────────────────────────────────
ATTR_RINGS     = [f'attr_ring0{i}_mean'     for i in range(N_RINGS)]
ATTR_RINGS_POS = [f'attr_pos_ring0{i}_mean' for i in range(N_RINGS)]
ATTR_RINGS_NEG = [f'attr_neg_ring0{i}_mean' for i in range(N_RINGS)]
INT_RINGS      = [f'radial_ring0{i}_mean_x' for i in range(N_RINGS)]
MORPH_COLS = [
    'area_x', 'circularity_x', 'eccentricity_x', 'solidity_x',
    'boundary_roughness_x', 'radius_gyration_normalised_x', 'radius_inscribed_ratio_x',
    'radial_inner_outer_ratio_x', 'radial_monotonicity_x',
    'intensity_mean_x', 'intensity_std_x', 'intensity_entropy_x', 'intensity_cv_x',
    'texture_contrast_x', 'texture_homogeneity_x', 'texture_correlation_x',
]
ATTR_SUMMARY = [
    'attr_mean', 'attr_std', 'attr_total_abs',
    'attr_pos_sum', 'attr_neg_sum', 'attr_pos_neg_ratio',
    'peripheral_attr_mean', 'peripheral_pos_fraction', 'peripheral_neg_fraction',
    'peripheral_frac_px', 'attr_pos_gini', 'attr_pos_centroid_offset',
    'attr_dapi_pearson', 'attr_dapi_spearman',
]
MUT_COLS = ["NPM1", "DNMT3A", "FLT3", "SETBP1", "NRAS", "TET2", "CEBPA", "IDH2", "IDH1", "CSF3R", "KIT", "SF3B1", "SRSF2", "JAK2", "STAG2", "EZH2", "CBL","ASXL1", "RUNX1", "PTPN11", "BCOR", "NF1","IGH","WT1", "ABL1", "KRAS", "TP53", "GATA2", "U2AF1", "BRC1", "KMT2A-PDT", "RUNX1::RUNX1T1"]
# NOTE: "KIT SF3B1" used to be a single (erroneous) entry here — KIT and SF3B1
# are separate columns in the source data, so that typo silently excluded both
# genes from every mutation-group analysis (§7, §7b, §7c) and the CSV export.

def mwu_stars(a, b):
    """Mann-Whitney U significance stars."""
    _, p = mannwhitneyu(a.dropna(), b.dropna(), alternative='two-sided')
    if p < 0.001: return '***', p
    if p < 0.01:  return '**',  p
    if p < 0.05:  return '*',   p
    return 'ns', p

print('Setup complete')

def cluster_bootstrap_r(x, y, clusters, n_boot=2000, ci=95, seed=0):
    """
    Spearman r with 95% CI via patient-level (cluster) bootstrap.
    Resamples whole patients, not individual cells.
    Returns (r_obs, ci_lo, ci_hi).
    """
    from scipy.stats import spearmanr
    rng = np.random.default_rng(seed)
    
    x, y, clusters = np.asarray(x), np.asarray(y), np.asarray(clusters)
    unique_patients = np.unique(clusters)
    
    # Observed r
    r_obs, _ = spearmanr(x, y)
    
    # Bootstrap
    boot_r = np.empty(n_boot)
    for i in range(n_boot):
        sampled = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        idx = np.concatenate([np.where(clusters == p)[0] for p in sampled])
        boot_r[i], _ = spearmanr(x[idx], y[idx])
    
    lo = (100 - ci) / 2
    ci_lo, ci_hi = np.percentile(boot_r, [lo, 100 - lo])
    return r_obs, ci_lo, ci_hi

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path('/path/to/ainu_root/results/occlusion_analysis/ainu_split3_model_best.pth.tar/') / 'MDS_AML_occlusion_analysis_figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Figures will be saved to: {OUTPUT_DIR}')

---
## 1 · Dataset overview

Load the merged CSV and inspect the class balance and prediction score distributions.

In [ ]:
df = pd.read_csv(CSV_PATH_MDS_MDR)
# merge with patient-level metadata
patient_metadata_path = Path('/path/to/ainu_root/data/human_data/patient_image_data.csv')
patient_metadata = pd.read_csv(patient_metadata_path)
# merge df with patient metadata on 'File'
df = df.merge(patient_metadata, left_on='image_name', right_on='File', how='left')
df
label_col = 'new_labels' if 'new_labels' in df.columns else 'condition'
df['_class'] = df[label_col].str.strip()

pos = df[df['_class'] == 'AML']
preleukemia = df[df['_class'] == 'Preleukemia']
neg = df[df['_class'] == 'Control']

print(f"Total images : {len(df)}")
print(f"  AML (positive)     : {len(pos)}")
print(f"  Preleukemia (intermediate) : {len(preleukemia)}")
print(f"  Control (negative) : {len(neg)}")
print(f"\nSample source breakdown:")
print(df.groupby(['source', '_class']).size().unstack(fill_value=0))

In [ ]:
c = 0
for col in df.columns:
    if col.endswith('_x') and col[:-2] + '_y' in df.columns:
        if df[col].equals(df[col[:-2] + '_y']):
            c+=1
            df.drop(columns=[col[:-2] + '_y'], inplace=True)
            df.rename(columns={col: col[:-2]}, inplace=True)
print(f"Dropped {c} duplicate columns after merging with patient metadata.")

In [ ]:
int_rings_sin_x = [c[:-2] if c.endswith('_x') else c for c in INT_RINGS]
df[INT_RINGS] = df[int_rings_sin_x]

# Same fix as INT_RINGS above: MORPH_COLS (circularity_x, eccentricity_x, ...)
# only exist as _x-suffixed columns in the AML_CT source. In this MDS_MDR
# source they exist under bare (unsuffixed) names — same 100% coverage, just
# no _x suffix. Without this copy, every MDS_MDR-derived row (i.e. every
# Preleukemia cell, since Preleukemia is 100% MDS_MDR-sourced) silently has
# all-NaN morphology data.
_morph_sin_x = [c[:-2] if c.endswith('_x') else c for c in MORPH_COLS]
_morph_bare_avail = [c for c in _morph_sin_x if c in df.columns]
_morph_x_avail = [c + '_x' for c in _morph_bare_avail]
df[_morph_x_avail] = df[_morph_bare_avail]

df['source_x'] = df['source']

In [ ]:
# read CSV_PATH_AML_CT and add the same column _class and then concat rows in the end of df
df_aml_ct = pd.read_csv(CSV_PATH_AML_CT)
print(df_aml_ct['new_labels'].unique())
label_col_aml_ct = 'new_labels' if 'new_labels' in df_aml_ct.columns else 'condition'
df_aml_ct['_class'] = df_aml_ct[label_col_aml_ct].str.strip()
# first check the columns of df_aml_ct and df
print("Columns in df_aml_ct:", df_aml_ct.columns)
print("Columns in df:", df.columns)
df_aml_ct
# remove duplicate columns if the values of one column are the same as the other column, for example, if 'condition' and 'new_labels' are the same, we can drop one of them
# if 'column_x' and 'column_y' are the same, we can drop one of them and keep without suffix

c = 0
for col in df_aml_ct.columns:
    if col.endswith('_x') and col[:-2] + '_y' in df_aml_ct.columns:
        if df_aml_ct[col].equals(df_aml_ct[col[:-2] + '_y']):
            c+=1
            df_aml_ct.drop(columns=[col[:-2] + '_y'], inplace=True)
            df_aml_ct.rename(columns={col: col[:-2]}, inplace=True)
print(f"Dropped {c} duplicate columns in df_aml_ct")


In [ ]:
df_aml_ct

In [ ]:
#concat df and df_aml_ct
df = pd.concat([df, df_aml_ct], ignore_index=True)
df

In [ ]:
#remove duplicates after concat, if there are any duplicate rows, we can drop them
df = df.drop_duplicates(['image_name', '_class'])
df['patientID_x'] = df['patientID'].astype(str)

In [ ]:
df_sin_mrd = df[df['MRD'].isna()]
mrd = df[df['MRD'].notna()]
df = df_sin_mrd
del df_sin_mrd
df.shape, mrd.shape



In [ ]:
mrd['_class'] = "MRD"  # merged: raw subclass detail stays in the 'MRD' column
mrd['_class'].value_counts()

In [ ]:
df = pd.concat([df, mrd], ignore_index=True)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20/2.54, 20/2.54*4/14))

# Class counts
counts = df['_class'].value_counts()
# plot barplot horisontally
axes[0].bar(counts.index, counts.values, 
            color=[COLORS[c] for c in counts.index], width=0.5, edgecolor='white')
#for i, (idx, val) in enumerate(counts.items()):
#    axes[0].text(i, val + 4, str(val), ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Class distribution', fontweight='bold')
axes[0].set_ylabel('N images')
axes[0].set_ylim(0, counts.max() * 1.15)
# rotate x-axis labels
#axes[0].tick_params(axis='x', rotation=45, ha='right')
axes[0].tick_params(axis='x', rotation=45)   # rotation only — this part is fine on its own
for label in axes[0].get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')
# Prediction score per class — violin (no overlap)
if 'aml_score' in df.columns:
    parts = df[['_class','aml_score']].dropna()
    sns.violinplot(data=parts, x='_class', y='aml_score',
                   order=CLASSES, palette=COLORS,
                   inner='quart', linewidth=0.8, ax=axes[1])
    axes[1].set_title('AML score distribution', fontweight='bold')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Score (softmax)')
else:
    axes[1].text(0.5, 0.5, 'aml_score\nnot in CSV', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=11, color='gray')
axes[1].tick_params(axis='x', rotation=45)
for label in axes[1].get_xticklabels():
    label.set_rotation(45)
    label.set_ha('right')
# Source breakdown
src_counts = df.groupby(['source', '_class']).size().unstack(fill_value=0)
src_counts.plot(kind='bar', ax=axes[2],
                color=[COLORS[c] for c in src_counts.columns], edgecolor='white')
axes[2].set_title('Sample source & class', fontweight='bold')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='class', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
counts.rename_axis('class').rename('n_images').to_csv(OUTPUT_DIR / '01_dataset_overview_class_counts.csv', header=True)
src_counts.to_csv(OUTPUT_DIR / '01_dataset_overview_source_counts.csv')

plt.savefig(OUTPUT_DIR / "01_dataset_overview.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Occlusion-analysis dataset summary: cells & patients per class ───────────
pat_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

_class_summary_rows = []
for cls in [c for c in COLORS if c in df['_class'].unique()]:
    sub = df[df['_class'] == cls]
    _class_summary_rows.append({
        'class': cls,
        'n_cells': len(sub),
        'n_patients': sub[pat_col].nunique(),
    })
class_summary_df = pd.DataFrame(_class_summary_rows)
print('Cells and patients used for the occlusion analysis, by class:')
print(class_summary_df.to_string(index=False))
print(f"\nTotal: {class_summary_df['n_cells'].sum()} cells, "
      f"{df[pat_col].nunique()} unique patients")

# Preleukemia subtype breakdown (MDS / CMML / CCUS) — the 'Preleukemia' class
# label pools these together; 'disease_type' (from patient_image_data.csv)
# has the finer diagnosis. A further ~139 originally-Preleukemia-labeled
# cells ended up in the MRD class instead (they had MRD follow-up status
# recorded), so they're listed separately here rather than folded into the
# subtype counts below.
if 'disease_type' in df.columns:
    _prelk = df[df['_class'] == 'Preleukemia']
    _subtype_rows = []
    for subtype, grp in _prelk.groupby('disease_type', dropna=False):
        _subtype_rows.append({
            'disease_type': subtype,
            'n_cells': len(grp),
            'n_patients': grp[pat_col].nunique(),
        })
    prelk_subtype_df = pd.DataFrame(_subtype_rows).sort_values('n_cells', ascending=False)
    print('\nPreleukemia subtype breakdown (disease_type column):')
    print(prelk_subtype_df.to_string(index=False))

    _mrd = df[df['_class'] == 'MRD']
    if 'disease_type' in _mrd.columns and _mrd['disease_type'].notna().any():
        print('\n(For reference) disease_type among cells now in the MRD class:')
        print(_mrd['disease_type'].value_counts(dropna=False).to_string())

    class_summary_df.to_csv(OUTPUT_DIR / '01b_class_summary_counts.csv', index=False)
    prelk_subtype_df.to_csv(OUTPUT_DIR / '01b_preleukemia_subtype_counts.csv', index=False)
else:
    print("\n'disease_type' column not found in df — cannot break down Preleukemia subtypes")
    class_summary_df.to_csv(OUTPUT_DIR / '01b_class_summary_counts.csv', index=False)

---
## 2 · Ring attribution profiles

The cell is divided into **8 concentric rings** (ring 0 = outermost boundary, ring 7 = centre nucleus).  
For each ring we compute the **mean occlusion attribution**: how much does occluding that annular region change the model's AML confidence?

> **Positive attribution** → the model relies on that region to predict AML  
> **Negative attribution** → that region *suppresses* the AML prediction  

Error bands are ± SEM. Stars mark Mann-Whitney U significance (*** p < 0.001).

In [ ]:
df.loc[df['_class']=='Control', 'patientID_x'] = df.loc[df['_class']=='Control', 'patientID.1']
df.loc[df['_class']=='AML', 'patientID_x'] = df.loc[df['_class']=='AML', 'patientID.1']

In [ ]:
# df is now fully assembled: _class finalized (incl. MRD split out of AML),
# and patientID_x corrected for Control/AML. Re-derive pos/neg/preleukemia HERE.
#
# The versions from the "Dataset overview" section (cell 4) were snapshotted
# right after loading CSV_PATH_MDS_MDR alone — before the CSV_PATH_AML_CT
# concat, before the MRD split, before the patientID_x fix. At that point:
#   - neg (Control) was EMPTY (0 rows) — MDS_MDR has no Control-labeled rows
#   - pos (AML) was missing the 1,157 AML_CT-sourced AML rows and still
#     included MRD-tracked patients that get reclassified below
# Every downstream cell that referenced the old pos/neg was silently working
# on wrong/empty data. Recompute from the final df instead.
pos = df[df['_class'] == positive_name].copy()
neg = df[df['_class'] == negative_name].copy()
preleukemia = df[df['_class'] == 'Preleukemia'].copy()
print(f'AML (pos): {len(pos)}   Control (neg): {len(neg)}   Preleukemia: {len(preleukemia)}')

In [ ]:
%matplotlib inline
avail_rings     = [c for c in ATTR_RINGS     if c in df.columns]
avail_rings_pos = [c for c in ATTR_RINGS_POS if c in df.columns]
avail_rings_neg = [c for c in ATTR_RINGS_NEG if c in df.columns]

import matplotlib as mpl
from statsmodels.stats.multitest import multipletests

# ── Journal figure dimensions (20 cm canvas, 8 pt Arial) ──────────────
FW = 20 / 2.54   # full page width in inches (7.87")
marker_class = {'AML': '^', 'Control': 'o', 'Preleukemia': 's', 'MRD': 'D'}
sns.set_theme(style='whitegrid')
mpl.rcParams.update({
    'font.family': 'Arial',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'pdf.fonttype': 42,
    'figure.dpi': 120,
    'grid.linestyle': '--',
    'grid.linewidth': 0.5,
    'axes.linewidth': 0.8,
})

pat_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

fig, axes = plt.subplots(1, 3, figsize=(FW*1.8, FW * 6 / 16))

for ax, ring_cols, title, sign in zip(
        axes,
        [avail_rings, avail_rings_pos, avail_rings_neg],
        ['Total attribution per ring',
         'AML-supporting per ring',
         'AML-suppressing per ring'],
        [1, 1, -1]):

    for cls in CLASSES:
        sub = df[df['_class'] == cls].dropna(subset=ring_cols)
        pat_means = sub.groupby(pat_col)[ring_cols].mean()
        m = sign * pat_means.mean().values
        s = pat_means.sem().values
        ax.plot(RING_X[:len(ring_cols)], m, marker_class[cls] + ':', color=COLORS[cls],
                label=f'{cls} (n={len(pat_means)} patients)', lw=1, ms=3)
        ax.fill_between(RING_X[:len(ring_cols)], m - s, m + s,
                        alpha=0.18, color=COLORS[cls])

    # Significance stars — BH-FDR corrected across rings within each subplot
    # Patient-median MWU (n=patients, not cells) — avoids pseudo-replication
    def _ring_p(col):
        a = df[df['_class']=='AML'    ].groupby(pat_col)[col].median()
        c = df[df['_class']=='Control'].groupby(pat_col)[col].median()
        
        return mannwhitneyu(a.dropna(), c.dropna(), alternative='two-sided')[1]
    raw_p = [_ring_p(col) for col in ring_cols]
    _, corrected_p, _, _ = multipletests(raw_p, method='fdr_bh')
    for i, p in enumerate(corrected_p):
        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        if star != 'ns':
            ax.text(i, ax.get_ylim()[1] * 0.97, star, ha='center',
                    va='top', color='#333')

    ax.axhline(0, color='k', lw=0.7, linestyle='--')
    ax.set_xticks(RING_X[:len(ring_cols)])
    ax.set_xticklabels(RING_LBL[:len(ring_cols)])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Ring', fontweight='bold')

axes[2].legend(fontsize=8, bbox_to_anchor=(-0.1, 1), loc='upper right')

axes[0].set_ylabel('Mean AML-class attribution')
#plt.suptitle('Radial attribution profiles \u2014 AML vs Control', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02a_ring_profiles.pdf", bbox_inches="tight")
plt.show()
# save csv of data of these plots
for i, (ax, ring_cols, title) in enumerate(zip(axes, [avail_rings, avail_rings_pos, avail_rings_neg], ['Total attribution per ring', 'AML-supporting per ring', 'AML-suppressing per ring'])):
    records = []
    for cls in CLASSES:
        sub = df[df['_class'] == cls].dropna(subset=ring_cols)
        pat_means = sub.groupby(pat_col)[ring_cols].mean()
        m = pat_means.mean().values
        s = pat_means.sem().values
        for j, col in enumerate(ring_cols):
            records.append({
                'class': cls,
                'ring': RING_LBL[j],
                'mean': m[j],
                'sem': s[j],
                'title': title
            })
    ring_df = pd.DataFrame(records)
    csv_path = OUTPUT_DIR / f"02a_ring_profiles_{i}.csv"
    ring_df.to_csv(csv_path, index=False)
    print(f"Ring profile data saved to {csv_path}")


In [ ]:
# ── 02a (all subclasses): ring attribution profiles — Control, Preleukemia, ──
# AML, and MRD (merged: positive/negative/low-positive/unspecific all collapse
# into one 'MRD' class here; raw status is still in the 'MRD' column if that
# detail is ever needed). COLORS and marker_class (cell 1 / above) already
# have an entry for all of these.
ALL_CLASSES = [c for c in COLORS if c in df['_class'].unique()]
print(f"Classes plotted: {ALL_CLASSES}")

fig, axes = plt.subplots(1, 3, figsize=(FW, FW * 6 / 16))

for ax, ring_cols, title, sign in zip(
        axes,
        [avail_rings, avail_rings_pos, avail_rings_neg],
        ['Total attribution per ring',
         'AML-supporting per ring',
         'AML-suppressing per ring'],
        [1, 1, -1]):

    for cls in ALL_CLASSES:
        sub = df[df['_class'] == cls].dropna(subset=ring_cols)
        pat_means = sub.groupby(pat_col)[ring_cols].mean()
        if pat_means.empty:
            continue
        m = sign * pat_means.mean().values
        s = pat_means.sem().values
        ax.plot(RING_X[:len(ring_cols)], m, marker_class.get(cls, 'o') + ':', color=COLORS[cls],
                label=f'{cls} (n={len(pat_means)} patients)', lw=1, ms=3)
        ax.fill_between(RING_X[:len(ring_cols)], m - s, m + s,
                        alpha=0.18, color=COLORS[cls])

    # Significance stars — AML vs Control, BH-FDR corrected across rings
    # (kept as the primary reference comparison; MRD subclasses shown for
    # context but not included in this particular test)
    def _ring_p(col):
        a = df[df['_class']=='AML'    ].groupby(pat_col)[col].median()
        c = df[df['_class']=='Control'].groupby(pat_col)[col].median()
        return mannwhitneyu(a.dropna(), c.dropna(), alternative='two-sided')[1]
    raw_p = [_ring_p(col) for col in ring_cols]
    _, corrected_p, _, _ = multipletests(raw_p, method='fdr_bh')
    for i, p in enumerate(corrected_p):
        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        if star != 'ns':
            ax.text(i, ax.get_ylim()[1] * 0.97, star, ha='center',
                    va='top', color='#333')

    ax.axhline(0, color='k', lw=0.7, linestyle='--')
    ax.set_xticks(RING_X[:len(ring_cols)])
    ax.set_xticklabels(RING_LBL[:len(ring_cols)])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Ring', fontweight='bold')

axes[2].legend(bbox_to_anchor=(1.02, 0.5), loc='center left', ncol=1, frameon=True, fontsize=6.5)

axes[0].set_ylabel('Mean AML-class attribution')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02a_ring_profiles_all_classes.pdf", bbox_inches="tight")
plt.show()
# save csv of data of these plots
for i, (ax, ring_cols, title) in enumerate(zip(axes, [avail_rings, avail_rings_pos, avail_rings_neg], ['Total attribution per ring', 'AML-supporting per ring', 'AML-suppressing per ring'])):
    records = []
    for cls in ALL_CLASSES:
        sub = df[df['_class'] == cls].dropna(subset=ring_cols)
        pat_means = sub.groupby(pat_col)[ring_cols].mean()
        if pat_means.empty:
            continue
        m = pat_means.mean().values
        s = pat_means.sem().values
        for j, col in enumerate(ring_cols):
            records.append({
                'class': cls,
                'ring': RING_LBL[j],
                'mean': m[j],
                'sem': s[j],
                'title': title
            })
    ring_df = pd.DataFrame(records)
    csv_path = OUTPUT_DIR / f"02a_ring_profiles_all_classes_{i}.csv"
    ring_df.to_csv(csv_path, index=False)
    print(f"Ring profile data saved to {csv_path}")

In [ ]:
df['follow-up'].value_counts()

In [ ]:
df_worsen = df[df['follow-up']=='worsen']

In [ ]:
%matplotlib inline
avail_rings     = [c for c in ATTR_RINGS     if c in df.columns]
avail_rings_pos = [c for c in ATTR_RINGS_POS if c in df.columns]
avail_rings_neg = [c for c in ATTR_RINGS_NEG if c in df.columns]

import matplotlib as mpl
from statsmodels.stats.multitest import multipletests

# ── Journal figure dimensions (20 cm canvas, 8 pt Arial) ──────────────
FW = 20 / 2.54   # full page width in inches (7.87")
marker_class = {'AML': '^', 'Control': 'o', 'Preleukemia': 's', 'MRD': 'D'}
sns.set_theme(style='whitegrid')
mpl.rcParams.update({
    'font.family': 'Arial',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'pdf.fonttype': 42,
    'figure.dpi': 120,
    'grid.linestyle': '--',
    'grid.linewidth': 0.5,
    'axes.linewidth': 0.8,
})

pat_col = 'patientID_x' if 'patientID_x' in df_worsen.columns else 'patient_ID'

fig, axes = plt.subplots(1, 3, figsize=(FW, FW * 6 / 16))

for ax, ring_cols, title, sign in zip(
        axes,
        [avail_rings, avail_rings_pos, avail_rings_neg],
        ['Total attribution per ring',
         'AML-supporting per ring',
         'AML-suppressing per ring'],
        [1, 1, -1]):

    for cls in CLASSES:
        sub = df_worsen[df_worsen['_class'] == cls].dropna(subset=ring_cols)
        pat_means = sub.groupby(pat_col)[ring_cols].mean()
        m = sign * pat_means.mean().values
        s = pat_means.sem().values
        ax.plot(RING_X[:len(ring_cols)], m, marker_class[cls] + ':', color=COLORS[cls],
                label=f'{cls} (n={len(pat_means)} patients)', lw=1, ms=3)
        ax.fill_between(RING_X[:len(ring_cols)], m - s, m + s,
                        alpha=0.18, color=COLORS[cls])

    # Significance stars — BH-FDR corrected across rings within each subplot
    # Patient-median MWU (n=patients, not cells) — avoids pseudo-replication
    def _ring_p(col):
        a = df_worsen[df_worsen['_class']=='AML'    ].groupby(pat_col)[col].median()
        c = df_worsen[df_worsen['_class']=='Control'].groupby(pat_col)[col].median()
        
        return mannwhitneyu(a.dropna(), c.dropna(), alternative='two-sided')[1]
    raw_p = [_ring_p(col) for col in ring_cols]
    _, corrected_p, _, _ = multipletests(raw_p, method='fdr_bh')
    for i, p in enumerate(corrected_p):
        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        if star != 'ns':
            ax.text(i, ax.get_ylim()[1] * 0.97, star, ha='center',
                    va='top', color='#333')

    ax.axhline(0, color='k', lw=0.7, linestyle='--')
    ax.set_xticks(RING_X[:len(ring_cols)])
    ax.set_xticklabels(RING_LBL[:len(ring_cols)])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Ring (boundary \u2192 centre)')

axes[2].legend(bbox_to_anchor=(0.5, -0.3), loc='center right', ncol=1, frameon=True)

axes[0].set_ylabel('Mean AML-class attribution')
#plt.suptitle('Radial attribution profiles \u2014 AML vs Control', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02a_ring_profiles.pdf", bbox_inches="tight")
plt.show()
# save csv of data of these plots
for i, (ax, ring_cols, title) in enumerate(zip(axes, [avail_rings, avail_rings_pos, avail_rings_neg], ['Total attribution per ring', 'AML-supporting per ring', 'AML-suppressing per ring'])):
    records = []
    for cls in CLASSES:
        sub = df_worsen[df_worsen['_class'] == cls].dropna(subset=ring_cols)
        pat_means = sub.groupby(pat_col)[ring_cols].mean()
        m = pat_means.mean().values
        s = pat_means.sem().values
        for j, col in enumerate(ring_cols):
            records.append({
                'class': cls,
                'ring': RING_LBL[j],
                'mean': m[j],
                'sem': s[j],
                'title': title
            })
    ring_df = pd.DataFrame(records)
    csv_path = OUTPUT_DIR / f"02a_ring_profiles_{i}.csv"
    ring_df.to_csv(csv_path, index=False)
    print(f"Ring profile data saved to {csv_path}")


In [ ]:
%matplotlib inline
# ── 02b: DAPI per ring, pos/neg profiles, and positive-mass fraction ──────────
# Four panels expanding on 02a:
#   1. DAPI intensity per ring  (INT_RINGS — radial mean pixel intensity,
#      z-scored WITHIN PATIENT before averaging — see note below)
#   2. Positive attribution per ring  (mean(max(occ,0)) per ring)
#   3. Negative attribution per ring  (mean(min(occ,0)) per ring, shown as |value|)
#   4. Positive mass fraction per ring  (pos_mass / (pos_mass + |neg_mass|))
#      ≈ proxy for "what fraction of the ring's signal is positive-signed"
#      (not pixel-count area, but attribution-mass-weighted area)

int_rings = [c for c in INT_RINGS if c in df.columns]

mpl.rcParams['font.family'] = 'Arial'
plt.rc('font', size=8) #set the default font size to be 8

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', font_scale=1.0)
# mlt gridline style
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.linewidth'] = 0.5
# also the axes border width
plt.rcParams['axes.linewidth'] = 0.8
if not int_rings:
    print('INT_RINGS not found in df — skipping DAPI ring panel')
else:
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    x = RING_X[:len(avail_rings)]
    lbl = RING_LBL[:len(avail_rings)]

    # ── Panel 1: DAPI intensity per ring ─────────────────────────────────────
    ax = axes[0]
    pat_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

    # int_rings mixes raw 0-255 intensities (AML_CT-sourced rows) with
    # normalized 0-1 intensities (MDS_MDR-sourced rows) — different patients'
    # rings are on different scales purely from provenance, not biology.
    # Z-score within patient (pooling mean/std across all 8 rings per patient)
    # to put every patient on a common scale before averaging across patients.
    pat_mu  = df.groupby(pat_col)[int_rings].transform('mean').mean(axis=1)
    pat_sig = df.groupby(pat_col)[int_rings].transform('std').mean(axis=1).replace(0, np.nan)
    int_rings_z = [f'{c}_z' for c in int_rings]
    df[int_rings_z] = df[int_rings].sub(pat_mu, axis=0).div(pat_sig, axis=0)

    for cls in ALL_CLASSES:
        print(f"Processing class {cls} for DAPI intensity per ring")
        sub = df[df['_class'] == cls].dropna(subset=int_rings_z)
        pat_means = sub.groupby(pat_col)[int_rings_z].mean()
        m = pat_means.mean().values
        s = pat_means.sem().values
        ax.plot(x, m, marker_class[cls] + '-', color=COLORS[cls], lw=2, ms=6,
                label=f'{cls} (n={len(pat_means)} patients)')
        ax.fill_between(x, m - s, m + s, alpha=0.18, color=COLORS[cls])
        # remove pat_means to free memory, sub and m, s will be used later for export
        del pat_means, sub, m, s
    for i, (col_i, col_a) in enumerate(zip(int_rings_z, avail_rings)):
        # AML vs Control specifically (not just "first two classes" — CLASSES
        # order is [Control, Preleukemia, AML], so positional indices would
        # silently compare Control vs Preleukemia instead).
        _aml_med  = df[df['_class']==positive_name].groupby(pat_col)[col_i].median()
        _ctrl_med = df[df['_class']==negative_name].groupby(pat_col)[col_i].median()
        star, _ = mwu_stars(_aml_med, _ctrl_med)
        if star != 'ns':
            ax.text(i, ax.get_ylim()[1]*0.97, star,
                    ha='center', fontsize=9, va='top', color='#333')
    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=8)
    ax.set_ylabel('Relative chromatin density\n(DAPI z-score, within patient)')
    ax.set_title('Relative chromatin density per ring', fontweight='bold')
    ax.set_xlabel('Ring',
                 fontweight='bold')
    ax.legend(fontsize=8)
    ax.remove()
    # ── Panel 4: Positive mass fraction per ring ──────────────────────────────
    # Estimator: mean-of-per-cell-ratios (ratio computed per cell, then averaged
    # across cells within patient, then across patients). NOT ratio-of-ring-means.
    # pos_frac = pos_mass / (pos_mass + |neg_mass|) per cell per ring.
    # A value of 1.0 means the ring is 100% positively attributed (by mass).
    # One-sample Wilcoxon signed-rank vs 0.5 (patient-level, one-sided "less")
    # tests whether each ring is significantly suppression-dominated.
    from scipy.stats import wilcoxon as _wsr
    ax = axes[1]
    frac_export_rows = []
    for cls in ALL_CLASSES:
        sub = df[df['_class'] == cls].dropna(subset=avail_rings_pos + avail_rings_neg)
        pos_m = sub[avail_rings_pos].values
        neg_m = np.abs(sub[avail_rings_neg].values)
        total = pos_m + neg_m
        frac = pd.DataFrame(
            np.where(total > 0, pos_m / total, np.nan),
            index=sub[pat_col].values,
            columns=avail_rings_pos,
        )
        pat_frac = frac.groupby(level=0).mean()
        m = pat_frac.mean().values
        s = pat_frac.sem().values
        ax.plot(x, m, marker_class[cls] + ':', color=COLORS[cls], lw=1.5, ms=5,
                label=f'{cls} (n={len(pat_frac)} patients)')
        ax.fill_between(x, m - s, m + s, alpha=0.18, color=COLORS[cls])

        # One-sample Wilcoxon vs 0.5 per ring, annotate non-ns rings
        for ri, (pr, lbl_r) in enumerate(zip(avail_rings_pos, lbl)):
            vals  = pat_frac.iloc[:, ri].dropna()
            diffs = vals - 0.5
            diffs_nz = diffs[diffs != 0]
            if len(diffs_nz) >= 4:
                _, p_w = _wsr(diffs_nz, alternative='less')
            else:
                p_w = 1.0
            star = '***' if p_w < 0.001 else '**' if p_w < 0.01 else '*' if p_w < 0.05 else 'ns'
            frac_export_rows.append({
                'class': cls, 'ring': ri, 'ring_label': lbl_r.replace('\n', ' '),
                'n_patients': len(vals),
                'pat_mean_frac': round(vals.mean(), 3),
                'pat_sem_frac':  round(vals.sem(),  3),
                'below_0.5':     bool(vals.mean() < 0.5),
                'p_vs_0.5_less': round(p_w, 4),
                'sig':           star,
            })
            #if star != 'ns' and cls == positive_name:
            #    ax.text(x[ri], m[ri] - s[ri] - 0.03, star,
            #            ha='center', fontsize=7, color=COLORS[cls], va='top')

    ax.axhline(0.5, color='k', lw=0.8, ls='--', label='equal pos/neg')
    ax.set_ylim(0, 1)
    ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=8)
    ax.set_ylabel('AML-supporting mass fraction\n(0=all suppressing, 1=all supporting)')
    ax.set_title('AML-supporting mass fraction per ring\n'
                 'proxy for attributed area dominance',
                 fontweight='bold', fontsize=8)
    ax.set_xlabel('Ring',
                 fontweight='bold', fontsize=8)
    ax.legend(fontsize=8)

    # Export mass-fraction stats
    frac_df_out = pd.DataFrame(frac_export_rows)
    frac_csv = OUTPUT_DIR / '02b_pos_mass_frac_patient_stats.csv'
    frac_df_out.to_csv(frac_csv, index=False)
    print(frac_df_out.to_string(index=False))
    print(f'\nSaved → {frac_csv}')

    #plt.suptitle('Ring-level DAPI, attribution sign profiles, and positive-mass fraction\n'
     #            'Mean ± SEM across cells  |  * p<0.05, ** p<0.01, *** p<0.001',
     #            fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '02b_ring_dapi_pos_neg_area.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Load heatmaps + images and compute per-cell pixel-level DAPI contrast ─────
from PIL import Image
from tqdm.auto import tqdm

# df now mixes rows from two source pipelines (CSV_PATH_AML_CT and
# CSV_PATH_MDS_MDR), and their occlusion .npy heatmaps live in different
# directories with different filename patterns:
#   AML_CT  dir: {image_name}_AML_occlusion.npy
#   MDS_MDR dir: {image_name}_AML.npy
# (verified: same array shape/dtype/sign convention, just a different naming
# convention from the two extraction runs — every row resolves to exactly one).
OCL_DIR_AML_CT  = Path(CSV_PATH_AML_CT).parent
OCL_DIR_MDS_MDR = Path(CSV_PATH_MDS_MDR).parent
IMG_DIR = Path('/path/to/ainu_root/data/human_data')
patient_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

# Two occlusion maps per cell:
#   {name}_occlusion.npy     — target=pred_label_idx (predicted class)
#                              occ > 0 supports the predicted class
#   {name}_AML_occlusion.npy — target=1 (always AML class)
#                              occ > 0 promotes AML prediction in ALL cells
#                              For AML cells: identical to _occlusion.npy
#                              For Control cells: reveals AML-like features

records = []
n_skipped = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc='Loading heatmaps'):
    name = row['image_name']
    img_path = IMG_DIR / f'{name}.png'

    occ_aml_path = OCL_DIR_AML_CT / f'{name}_AML_occlusion.npy'
    if not occ_aml_path.exists():
        occ_aml_path = OCL_DIR_MDS_MDR / f'{name}_AML.npy'

    if not occ_aml_path.exists() or not img_path.exists():
        n_skipped += 1
        continue

    ocl_aml = np.load(occ_aml_path)[0].astype(np.float32)      # target=AML class
    H, W = ocl_aml.shape
    img = np.array(
        Image.open(img_path).convert('L').resize((W, H), Image.BILINEAR),
        dtype=np.float32,
    )

    def dapi_stats(occ, img):
        pos_mask = occ > 0
        neg_mask = occ < 0
        return {
            'pos': float(img[pos_mask].mean()) if pos_mask.any() else float('nan'),
            'neg': float(img[neg_mask].mean()) if neg_mask.any() else float('nan'),
            'n_pos': int(pos_mask.sum()),
            'n_neg': int(neg_mask.sum()),
            'occ_pos_mean': float(occ[pos_mask].mean()) if pos_mask.any() else float('nan'),
            'occ_neg_mean': float(occ[neg_mask].mean()) if neg_mask.any() else float('nan'),
            'occ_pos_median': float(np.median(occ[pos_mask])) if pos_mask.any() else float('nan'),
            'occ_neg_median': float(np.median(occ[neg_mask])) if neg_mask.any() else float('nan'),
            'attr_pos_neg_ratio': float(occ[pos_mask].sum() / occ[neg_mask].sum()) if neg_mask.any() else float('nan'),
            'occ_pos': float(np.abs(occ[pos_mask]).sum()),
            'occ_neg': float(np.abs(occ[neg_mask]).sum()),
            # A single combined "total attribution" metric: magnitude × area (i.e., sum of attribution scores over the region, not mean).
        }

    aml_stats = dapi_stats(ocl_aml, img)

    records.append({
        'image_name':        name,
        'cls':               row['_class'],
        'patient':           row[patient_col],
        # Predicted-class map (target=pred_label_idx)
        'dapi_raw_pos':      aml_stats['pos'],
        'dapi_raw_neg':      aml_stats['neg'],
        'n_pos_px':          aml_stats['n_pos'],
        'n_neg_px':          aml_stats['n_neg'],
        'occ_pos_mean':      aml_stats['occ_pos_mean'],
        'occ_neg_mean':      aml_stats['occ_neg_mean'],
        'occ_pos_median':    aml_stats['occ_pos_median'],
        'occ_neg_median':    aml_stats['occ_neg_median'],
        'occ_pos':           aml_stats['occ_pos'],
        'occ_neg':           aml_stats['occ_neg'],
        'attr_pos_neg_ratio':aml_stats['attr_pos_neg_ratio'],
        # AML-class map (target=1, always AML)
        'dapi_raw_aml_pos':  aml_stats['pos'],
        'dapi_raw_aml_neg':  aml_stats['neg'],
        'n_aml_pos_px':      aml_stats['n_pos'],
        'n_aml_neg_px':      aml_stats['n_neg'],
        'occ_aml_pos_mean':  aml_stats['occ_pos_mean'],
        'occ_aml_neg_mean':  aml_stats['occ_neg_mean'],
        'occ_aml_pos_median':aml_stats['occ_pos_median'],
        'occ_aml_neg_median':aml_stats['occ_neg_median'],
    })

print(f'Loaded {len(records)} cells, skipped {n_skipped} (missing heatmap or image file)')
px_df = pd.DataFrame(records)

# ── Per-patient z-score ───────────────────────────────────────────────────────
raw_cols = ['dapi_raw_pos', 'dapi_raw_neg', 'dapi_raw_aml_pos', 'dapi_raw_aml_neg']
pat_mu  = px_df.groupby('patient')[raw_cols].transform('mean').mean(axis=1)
pat_sig = px_df.groupby('patient')[raw_cols].transform('std').mean(axis=1).replace(0, float('nan'))

# Predicted-class attribution
px_df['dapi_pos']        = (px_df['dapi_raw_pos']     - pat_mu) / pat_sig
px_df['dapi_neg']        = (px_df['dapi_raw_neg']     - pat_mu) / pat_sig
px_df['dapi_diff']       = px_df['dapi_pos'] - px_df['dapi_neg']
px_df['dapi_support']    = px_df['dapi_pos']   # occ>0 = promotes AML prediction (AML-class map)
px_df['dapi_oppose']     = px_df['dapi_neg']   # occ<0 = suppresses AML prediction (AML-class map)
px_df['dapi_class_diff'] = px_df['dapi_diff']

# AML-class attribution
px_df['dapi_aml_pos']    = (px_df['dapi_raw_aml_pos'] - pat_mu) / pat_sig
px_df['dapi_aml_neg']    = (px_df['dapi_raw_aml_neg'] - pat_mu) / pat_sig
px_df['dapi_aml_diff']   = px_df['dapi_aml_pos'] - px_df['dapi_aml_neg']


px_merged = px_df.merge(   # always rebuild — remove the "if not in dir()" check
    df[['image_name','_class','attr_pos_sum','attr_neg_sum',
        'attr_total_abs','attr_pos_gini']].rename(columns={'_class':'cls_df'}),
    on='image_name', how='left')

# ── Area-weighted DAPI metrics ────────────────────────────────────────────────
# n_pos_px / n_neg_px = number of pixels with occ>0 / occ<0 (already in px_df).
# Plain dapi_diff weights both groups equally regardless of their size.
# Area-weighted variants account for how much of the nucleus each sign covers.

total_px = px_df['n_pos_px'] + px_df['n_neg_px']

# Fraction of nucleus pixels that are positively attributed
px_df['pos_area_frac'] = px_df['n_pos_px'] / total_px

# Area-weighted DAPI contrast:
#   (mean_DAPI_pos × n_pos  −  mean_DAPI_neg × n_neg)  /  (n_pos + n_neg)
# = total DAPI "mass" in positive region minus negative region, per pixel
px_df['dapi_diff_weighted'] = (
    px_df['dapi_pos'] * px_df['n_pos_px']
    - px_df['dapi_neg'] * px_df['n_neg_px']
) / total_px

# Attribution mass = mean_attr × n_px (total signed signal, not just density)
px_df['occ_pos_mass'] = px_df['occ_pos_mean'] * px_df['n_pos_px']
px_df['occ_neg_mass'] = px_df['occ_neg_mean'] * px_df['n_neg_px']

def resolve_occ_path(name):
    """AML-class occlusion heatmap for `name`, whichever source it came from."""
    p = OCL_DIR_AML_CT / f'{name}_AML_occlusion.npy'
    if p.exists():
        return p
    p2 = OCL_DIR_MDS_MDR / f'{name}_AML.npy'
    if p2.exists():
        return p2
    raise FileNotFoundError(f'No occlusion heatmap found for {name}')

In [ ]:
# ── 02c: Pairplot — attribution rings, DAPI (sum/mean inside cell), area frac ─
# Build a per-cell feature table merging df (ring features, DAPI rings) with
# px_df (pixel-level DAPI stats, pos_area_frac) for a seaborn pairplot.

int_rings = [c for c in INT_RINGS if c in df.columns]

# ── Derive DAPI sum and mean inside nucleus from radial ring means ─────────────
# INT_RING_i = mean DAPI in ring i; rings have approximately equal pixel areas,
# so sum(INT_RINGS) ∝ total nuclear DAPI (scaled by constant ring pixel count).
pairplot_df = df[['image_name', '_class',
                   'attr_pos_neg_ratio', 'attr_pos_gini',
                   'attr_pos_sum', 'attr_neg_sum',
                   avail_rings[0],   # R0 boundary
                   avail_rings[-1],  # R7 centre
                   avail_rings_pos[-1],  # positive centre
                   ]].copy()
pairplot_df.columns = [c.replace('attr_ring0','ring').replace('_mean','')
                        .replace('attr_pos_ring0','pos_ring').replace('_mean','')
                        for c in pairplot_df.columns]

if int_rings:
    pairplot_df['dapi_sum_inside']  = df[int_rings].sum(axis=1)
    pairplot_df['dapi_mean_inside'] = df[int_rings].mean(axis=1)
    # Centre-to-boundary DAPI gradient (positive = brighter at centre)
    pairplot_df['dapi_gradient'] = df[int_rings[-1]] - df[int_rings[0]]

# Positive mass fraction per ring (panel 4 of 02b), condensed to one per-cell number:
# mean of the per-ring positive mass fraction across all rings
pos_m  = df[avail_rings_pos].values
neg_m  = np.abs(df[avail_rings_neg].values)
total  = pos_m + neg_m
frac   = np.where(total > 0, pos_m / total, np.nan)
pairplot_df['pos_mass_frac_mean'] = np.nanmean(frac, axis=1)

# Merge px_df columns: dapi_diff, pos_area_frac
px_sub = px_df[['image_name','dapi_diff','pos_area_frac',
                    'occ_pos_mean','occ_neg_mean']].copy()
if 'pos_area_frac' not in px_sub.columns:
    total_px = px_df['n_pos_px'] + px_df['n_neg_px']
    px_sub['pos_area_frac'] = px_df['n_pos_px'] / total_px
pairplot_df = pairplot_df.merge(px_sub, on='image_name', how='left')

pairplot_df = pairplot_df.rename(columns={'_class': 'Class'})
pairplot_df = pairplot_df.dropna()
print(f'pairplot_df: {pairplot_df.shape}, classes: {pairplot_df["Class"].value_counts().to_dict()}')

# ── Feature sets ──────────────────────────────────────────────────────────────
# Keep 6 features for readability; choose the most independent ones.
PAIR_COLS = [c for c in [
    'dapi_sum_inside',    # total nuclear DAPI (size × brightness)
    'dapi_mean_inside',   # mean nuclear DAPI (brightness alone)
    'dapi_gradient',      # DAPI centre-to-boundary gradient
    'dapi_diff',          # DAPI contrast between pos and neg attribution regions
    'pos_area_frac',      # fraction of attributed pixels that are positive
    'pos_mass_frac_mean', # mean pos-mass fraction across rings
    'attr_pos_neg_ratio', # attribution balance
    'attr_pos_gini',      # concentration of positive signal
] if c in pairplot_df.columns]

# ── Pairplot ──────────────────────────────────────────────────────────────────
pair_data = pairplot_df[PAIR_COLS + ['Class']].copy()

# Clip outliers at 1st–99th percentile per feature to avoid axis squash
for col in PAIR_COLS:
    lo, hi = pair_data[col].quantile([0.01, 0.99])
    pair_data[col] = pair_data[col].clip(lo, hi)

g = sns.pairplot(
    pair_data,
    hue='Class',
    palette=COLORS,
    diag_kind='kde',
    plot_kws=dict(alpha=0.25, s=8, rasterized=True),
    diag_kws=dict(fill=True, alpha=0.4),
    corner=True,          # lower triangle only — cleaner
)
g.figure.suptitle(
    '02c · Pairplot: attribution rings × DAPI (sum/mean inside) × positive area fraction\n'
    'Lower triangle; diagonal = per-class KDE',
    y=1.01, fontsize=11, fontweight='bold')

# Annotate each off-diagonal panel with Spearman r per class
from scipy.stats import spearmanr as _sp
for i, yi in enumerate(PAIR_COLS):
    for j, xi in enumerate(PAIR_COLS):
        if j >= i:
            continue
        ax = g.axes[i][j]
        if ax is None:
            continue
        for cls in CLASSES:
            sub = pair_data[pair_data['Class'] == cls][[xi, yi]].dropna()
            if len(sub) < 10:
                continue
            r, _ = _sp(sub[xi], sub[yi])
            y_pos = 0.97 if cls == CLASSES[0] else 0.82
            ax.text(0.04, y_pos, f'r={r:.2f}',
                    transform=ax.transAxes, fontsize=6,
                    color=COLORS[cls], va='top')

pairplot_df.to_csv(OUTPUT_DIR / '02c_pairplot_rings_dapi_area.csv', index=False)

g.figure.savefig(OUTPUT_DIR / '02c_pairplot_rings_dapi_area.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── 02d: Per-ring Spearman r — DAPI intensity vs attribution, patient-clustered ─
# For each ring i and each class, computes Spearman r between:
#   X = radial_ring0{i}_mean_x  (mean DAPI intensity in ring i, z-scored
#       within patient -- see note below; matches the "Relative chromatin
#       density" convention used in cell 02b)
#   Y = attr_ring0{i}_mean      (net attribution in ring i)
# and Y = pos_mass_frac_ring_i  (positive mass fraction in ring i)
# CIs via patient-level cluster bootstrap (same convention as Section 13 plots).

from scipy.stats import spearmanr as _sr

int_rings = [c for c in INT_RINGS if c in df.columns]
patient_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

if not int_rings or patient_col not in df.columns:
    print('INT_RINGS or patient column not available — skipping')
else:
    # int_rings mixes raw 0-255 intensities (AML_CT-sourced rows) with
    # normalized 0-1 intensities (MDS_MDR-sourced rows) -- different
    # patients' rings are on different scales purely from provenance, not
    # biology (same issue flagged in cell 02b). Z-score within patient
    # (pooling mean/std across all rings per patient) before correlating,
    # so the cross-patient Spearman r isn't distorted by this scale mix.
    pat_mu  = df.groupby(patient_col)[int_rings].transform('mean').mean(axis=1)
    pat_sig = df.groupby(patient_col)[int_rings].transform('std').mean(axis=1).replace(0, np.nan)
    int_rings_z = [f'{c}_z' for c in int_rings]
    df[int_rings_z] = df[int_rings].sub(pat_mu, axis=0).div(pat_sig, axis=0)

    # Compute per-ring positive mass fraction and add to df subset
    pos_m  = df[avail_rings_pos].values
    neg_m  = np.abs(df[avail_rings_neg].values)
    total  = pos_m + neg_m
    frac_arr = np.where(total > 0, pos_m / total, np.nan)
    frac_cols = [f'_pos_frac_r{i}' for i in range(len(avail_rings))]
    df_ring = df[[patient_col, '_class'] + int_rings_z + avail_rings].copy()
    for i, col in enumerate(frac_cols):
        df_ring[col] = frac_arr[:, i]

    N_BOOT = 2000
    rng = np.random.default_rng(42)

    def ring_bootstrap_r(sub, x_col, y_col):
        """Patient-clustered Spearman r + 95% CI for one ring × class slice."""
        valid = sub[[patient_col, x_col, y_col]].dropna()
        if len(valid) < 5:
            return np.nan, np.nan, np.nan
        r_obs, _ = _sr(valid[x_col], valid[y_col])
        patients  = valid[patient_col].unique()
        boot_r = []
        for _ in range(N_BOOT):
            samp = rng.choice(patients, size=len(patients), replace=True)
            idx  = np.concatenate([valid.index[valid[patient_col] == p].tolist()
                                   for p in samp])
            r_b, _ = _sr(valid.loc[idx, x_col], valid.loc[idx, y_col])
            boot_r.append(r_b)
        ci_lo, ci_hi = np.percentile(boot_r, [2.5, 97.5])
        return r_obs, ci_lo, ci_hi

    # Collect results for two Y targets: net attribution and pos-mass fraction
    results = {cls: {'attr': [], 'frac': []} for cls in ALL_CLASSES}

    for cls in ALL_CLASSES:
        sub = df_ring[df_ring['_class'] == cls]
        for i, (dapi_col, attr_col, frac_col) in enumerate(
                zip(int_rings_z, avail_rings, frac_cols)):
            r_a, lo_a, hi_a = ring_bootstrap_r(sub, dapi_col, attr_col)
            r_f, lo_f, hi_f = ring_bootstrap_r(sub, dapi_col, frac_col)
            results[cls]['attr'].append((r_a, lo_a, hi_a))
            results[cls]['frac'].append((r_f, lo_f, hi_f))



In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(FW, FW*0.35), sharey=False)
x = RING_X[:len(int_rings)]
lbl = RING_LBL[:len(int_rings)]

for ax, target, ylabel, title in zip(
    axes,
    ['attr', 'frac'],
    ['Spearman r\n(chromatin density vs net attribution)', 'Spearman r\n(chromatin density vs pos-mass fraction)'],
    ['Relative chromatin density × net attribution\nper ring', 'Relative chromatin density × positive-mass fraction\nper ring'],
):
    # Point + vertical error bar (95% CI) instead of a filled ribbon --
    # same information, much less ink, and small per-class x-offsets keep
    # 4 overlapping whiskers readable where 4 overlapping filled bands
    # would not be.
    n_cls = len(ALL_CLASSES)
    offset_step = 0.12
    for ci, cls in enumerate(ALL_CLASSES):
        vals = results[cls][target]
        rs   = np.array([v[0] for v in vals])
        los  = np.array([v[1] for v in vals])
        his  = np.array([v[2] for v in vals])
        yerr = np.vstack([rs - los, his - rs])
        x_off = x + (ci - (n_cls - 1) / 2) * offset_step
        ax.errorbar(x_off, rs, yerr=yerr, fmt=marker_class[cls]+':', color=COLORS[cls],
                    label=cls, lw=1, ms=5, capsize=2, elinewidth=1, zorder=3)

    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.axhline( 0.3, color='gray', lw=0.6, ls=':', alpha=0.6)
    ax.axhline(-0.3, color='gray', lw=0.6, ls=':', alpha=0.6)
    ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=8)
    ax.set_xlabel('Ring')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(-1, 1)

# One shared legend for both panels (same classes/colours in each)
_handles, _labels = axes[0].get_legend_handles_labels()
fig.legend(_handles, _labels, fontsize=9, loc='upper center',
           bbox_to_anchor=(0.5, 1.1), ncol=len(_labels), frameon=False)

# Save per-ring r table
rows = []
for cls in ALL_CLASSES:
    for i, (dapi_col, attr_col, frac_col) in enumerate(
            zip(int_rings, avail_rings, frac_cols)):
        r_a, lo_a, hi_a = results[cls]['attr'][i]
        r_f, lo_f, hi_f = results[cls]['frac'][i]
        n_pat = df_ring[df_ring['_class'] == cls][patient_col].nunique()
        rows.append({
            'class': cls, 'ring': i, 'ring_label': lbl[i].replace('\n',' '),
            'n_patients': n_pat,
            'r_dapi_vs_attr':     round(r_a, 3),
            'ci_lo_attr':         round(lo_a, 3),
            'ci_hi_attr':         round(hi_a, 3),
            'r_dapi_vs_posfrac':  round(r_f, 3),
            'ci_lo_frac':         round(lo_f, 3),
            'ci_hi_frac':         round(hi_f, 3),
        })
ring_r_df = pd.DataFrame(rows)
ring_r_df.to_csv(OUTPUT_DIR / 'Table S2.csv', index=False)  # S6B source data
ring_r_df.to_excel(OUTPUT_DIR / 'Table S2.xlsx', index=False)

#plt.suptitle('Figure S6B — Per-ring relative chromatin density vs attribution (patient-clustered Spearman r)\n'
#             'Echo of main finding in raw DAPI units: the support/oppose asymmetry pattern\n'
#             '(present in Control, absent in AML) is consistent with the chromatin-density\n'
#             'interpretation; not a confound-elimination test. Bands = 95% CI (patient bootstrap)',
#             fontsize=11, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'Figure_6SB.pdf', bbox_inches='tight')
plt.show()

print(ring_r_df.to_string(index=False))

In [ ]:
# ── Table S5+S6 (merged): per-ring AML-vs-Control MWU + DAPI-vs-attribution r ─
# 8 rows, one per ring. Two question types in one table:
#   S5 columns: AML vs Control MWU (3 attribution types × stats)
#               BH-FDR within each attribution type (8 tests each) — same
#               correction that placed the *** markers in Figure 6C.
#   S6 columns: within-class Spearman r (DAPI intensity vs net attribution)
#               95% CI from patient-clustered bootstrap (2000 iter, cell 11).
#               Significance = CI excludes zero (no parametric p-value due to clustering).
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

pat_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

# ── S5: recompute MWU for all 3 attribution types ────────────────────────────
panel_specs = [
    ('total',       avail_rings,     1),
    ('supporting',  avail_rings_pos, 1),
    ('suppressing', avail_rings_neg, 1),
]

# Patient-median aggregation — MWU on n=patients, not n=cells
s5_data = {}   # key: (attr_type, ring_idx) → dict of stats
for attr_type, ring_cols, sign in panel_specs:
    aml_pat  = df[df['_class'] == 'AML'    ].groupby(pat_col)[ring_cols].median()
    ctrl_pat = df[df['_class'] == 'Control'].groupby(pat_col)[ring_cols].median()
    raw_p, u_stats = [], []
    for col in ring_cols:
        u, p = mannwhitneyu(aml_pat[col].dropna(), ctrl_pat[col].dropna(), alternative='two-sided')
        u_stats.append(u); raw_p.append(p)
    _, adj_p, _, _ = multipletests(raw_p, method='fdr_bh')
    for j, col in enumerate(ring_cols):
        s5_data[(attr_type, j)] = {
            f'{attr_type}_AML_pat_med':  round(sign * aml_pat[col].median(), 4),
            f'{attr_type}_Ctrl_pat_med': round(sign * ctrl_pat[col].median(), 4),
            f'{attr_type}_n_AML_pat':    int(aml_pat[col].notna().sum()),
            f'{attr_type}_n_Ctrl_pat':   int(ctrl_pat[col].notna().sum()),
            f'{attr_type}_MWU_U':        int(u_stats[j]),
            f'{attr_type}_p_adj_BH':     float(f'{adj_p[j]:.4e}'),
            f'{attr_type}_sig':          ('***' if adj_p[j] < 0.001 else
                                          '**'  if adj_p[j] < 0.01  else
                                          '*'   if adj_p[j] < 0.05  else 'ns'),
        }

# ── S6: load bootstrapped Spearman r from cell 11 output ─────────────────────
def _ci_sig(lo, hi):
    if lo > 0: return '+ (CI>0)'
    if hi < 0: return '- (CI<0)'
    return 'ns'

spear = pd.read_csv(OUTPUT_DIR / '02d_ring_dapi_attr_spearman.csv')
s6_data = {}   # key: (class, ring_idx) → dict
for _, row in spear.iterrows():
    cls = row['class']; ri = int(row['ring'])
    s6_data[(cls, ri)] = {
        f'r_DAPI_attr_{cls}':    row['r_dapi_vs_attr'],
        f'ci_lo_attr_{cls}':     row['ci_lo_attr'],
        f'ci_hi_attr_{cls}':     row['ci_hi_attr'],
        f'sig_DAPI_attr_{cls}':  _ci_sig(row['ci_lo_attr'], row['ci_hi_attr']),
    }

# ── Merge into one row per ring ───────────────────────────────────────────────
n_rings_used = len(avail_rings)
records = []
for j in range(n_rings_used):
    row = {'ring': RING_LBL[j].replace('\n', ' ')}
    for attr_type, _, _ in panel_specs:
        row.update(s5_data.get((attr_type, j), {}))
    for cls in CLASSES:
        row.update(s6_data.get((cls, j), {}))
    records.append(row)

table_s56 = pd.DataFrame(records)
(OUTPUT_DIR / 'exported_data').mkdir(exist_ok=True)
out_path = OUTPUT_DIR / 'exported_data' / 'TableS5S6_ring_stats_merged.csv'
table_s56.to_csv(out_path, index=False)
print(f'Merged table saved → {out_path}')
print(f'Shape: {table_s56.shape}  ({table_s56.shape[0]} rings × {table_s56.shape[1]} columns)')
print()
# Pretty-print key columns only
preview_cols = (['ring'] +
    [c for c in table_s56.columns if 'sig' in c or 'p_adj' in c or 'r_DAPI' in c])
print(table_s56[preview_cols].to_string(index=False))


In [ ]:
# Per-image heatmap sorted by class
fig, axes = plt.subplots(1, len(CLASSES), figsize=(FW * len(CLASSES) / 2, FW*5.5/14))

for ax, cls in zip(axes, CLASSES):
    sub = df[df['_class'] == cls][avail_rings].dropna()
    # Clip to 99th percentile for readability
    vabs = sub.abs().to_numpy().ravel()
    vmax = np.percentile(vabs[~np.isnan(vabs)], 99)

    im = ax.imshow(sub.values, aspect='auto', cmap=CMAP_OCC,
                   norm=TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax))
    ax.set_title(f'{cls}  (n={len(sub)})', fontweight='bold')
    ax.set_xlabel('Ring (boundary → centre)')
    ax.set_ylabel('Image (each row = one cell)')
    ax.set_xticks(range(len(avail_rings)))
    ax.set_xticklabels([f'R{i}' for i in range(len(avail_rings))])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.04, label='AML-class attribution')

plt.suptitle('Per-image ring attribution heatmap  (red = AML-supporting, blue = AML-suppressing)',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
pd.concat([
    df[df['_class'] == cls][['image_name'] + avail_rings].dropna(subset=avail_rings).assign(_class=cls)
    for cls in CLASSES
], ignore_index=True).to_csv(OUTPUT_DIR / '02b_ring_heatmap_data.csv', index=False)

plt.savefig(OUTPUT_DIR / "02b_ring_heatmap.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Statistical summary table — patient-median MWU (n=patients)
pat_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'
rows = []
for col in avail_rings:
    aml_pat  = pos.groupby(pat_col)[col].median()
    ctrl_pat = neg.groupby(pat_col)[col].median()
    star, p = mwu_stars(aml_pat, ctrl_pat)
    rows.append({
        'Ring': col.replace('attr_', '').replace('_mean', ''),
        'AML pat-median': aml_pat.median(),
        'Control pat-median': ctrl_pat.median(),
        'Difference': aml_pat.median() - ctrl_pat.median(),
        'p-value': p,
        'sig': star,
    })

stats_rings = pd.DataFrame(rows).set_index('Ring')
stats_rings.style \
    .format({'AML pat-median': '{:.4f}', 'Control pat-median': '{:.4f}', 'Difference': '{:.4f}', 'p-value': '{:.2e}'}) \
    .background_gradient(subset=['Difference'], cmap='RdBu_r', vmin=-0.3, vmax=0.3)

**Interpretation**

- AML cells show **positive attribution throughout**, growing from boundary to centre — the model uses the entire cell body to predict AML, with increasing reliance on nuclear/central signal.
- Control cells show **negative attribution throughout** — almost every region of the cell *reduces* the AML prediction score when occluded, meaning the model already has low baseline confidence and losing any region confirms that it is not AML.
- The gradient toward the centre suggests the **nucleus** is the most informative region for AML classification, consistent with the enlarged, irregular nuclei characteristic of myeloblasts.

---
## 3 · Global attribution features by class

Beyond ring profiles, we extract global summary statistics for each cell's attribution map.

In [ ]:
avail_summary = [c for c in ATTR_SUMMARY if c in df.columns]

n_cols = 4
n_rows = int(np.ceil(len(avail_summary) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(FW, n_rows * FW/n_cols * 3.2/3.4))
axes_flat = axes.flatten()

pat_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'
for i, col in enumerate(avail_summary):
    ax = axes_flat[i]
    data = [df[df['_class'] == cls][col].dropna() for cls in CLASSES]
    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    labels=CLASSES, widths=0.5,
                    medianprops=dict(color='black', lw=2))
    for patch, cls in zip(bp['boxes'], CLASSES):
        patch.set_facecolor(COLORS[cls]); patch.set_alpha(0.6)
    # AML vs Control specifically — CLASSES[0]/[1] would be Control vs
    # Preleukemia given CLASSES = [Control, Preleukemia, AML].
    aml_pat  = df[df['_class']==positive_name].groupby(pat_col)[col].median()
    ctrl_pat = df[df['_class']==negative_name].groupby(pat_col)[col].median()
    star, p = mwu_stars(aml_pat, ctrl_pat)
    label = col.replace('attr_', '').replace('_x', '').replace('_', ' ')
    ax.set_title(f'{label}\n{star}  (p={p:.1e})', fontsize=7.5)
    ax.tick_params(labelsize=8)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Attribution summary statistics — AML (red) vs Control (blue)', 
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
_rows_03 = []
for _col in avail_summary:
    for _cls in CLASSES:
        _vals = df[df['_class'] == _cls][_col].dropna()
        _rows_03.append({'feature': _col, 'class': _cls, 'mean': _vals.mean(),
                          'median': _vals.median(), 'std': _vals.std(), 'n': len(_vals)})
pd.DataFrame(_rows_03).to_csv(OUTPUT_DIR / '03_attribution_summary_stats.csv', index=False)

plt.savefig(OUTPUT_DIR / "03_attribution_summary.pdf", bbox_inches="tight")
plt.show()

**Key observations**

| Feature | AML | Control | Meaning |
|---|---|---|---|
| `attr_pos_neg_ratio` | 15.3 | 2.8 | AML attribution dominated by positive signal |
| `attr_pos_sum` | 1912 | 1010 | AML cells have ~2× more positive attribution mass |
| `attr_neg_sum` | 503 | 1413 | Control cells have ~3× more negative attribution |
| `attr_total_abs` | 2415 | 2423 | **Equal total evidence** — model uses same amount of info |
| `attr_pos_gini` | 0.58 | 0.74 | AML attribution is **more spread**; control more concentrated |
| `attr_pos_centroid_offset` | 0.19 | 0.27 | AML positive attribution centred on cell; control off-centre |
| `peripheral_neg_fraction` | 0.45 | 0.35 | Control cells suppress AML at the boundary |
| `attr_dapi_pearson` | +0.043 | −0.033 | AML attribution mildly correlates with intensity |

The model uses equal *total* attribution energy for both classes — but the *direction* differs fundamentally.

---
## 4 · Morphology × Attribution correlations

Do cells with particular shapes or intensities produce stronger attributions?  
Spearman correlation between morphological features and attribution features across all cells.

In [ ]:
avail_morph = [c for c in MORPH_COLS if c in df.columns]
attr_for_corr = [c for c in avail_rings + avail_summary if c in df.columns]

corr_matrix = np.zeros((len(avail_morph), len(attr_for_corr)))
pval_matrix = np.ones_like(corr_matrix)

for i, mc in enumerate(avail_morph):
    for j, ac in enumerate(attr_for_corr):
        valid = df[[mc, ac]].dropna()
        if len(valid) > 10:
            r, p = spearmanr(valid[mc], valid[ac])
            corr_matrix[i, j] = r
            pval_matrix[i, j] = p

xlabels = [c.replace('attr_', '').replace('_mean', '').replace('_x', '') for c in attr_for_corr]
ylabels = [c.replace('_x', '') for c in avail_morph]

fig, ax = plt.subplots(figsize=(max(14, len(attr_for_corr) * 0.55),
                                 max(6, len(avail_morph) * 0.5)))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-0.55, vmax=0.55, aspect='auto')
ax.set_xticks(range(len(attr_for_corr)))
ax.set_xticklabels(xlabels, rotation=90, fontsize=6.5)
ax.set_yticks(range(len(avail_morph)))
ax.set_yticklabels(ylabels, fontsize=8)

# Mark significant cells
for i in range(len(avail_morph)):
    for j in range(len(attr_for_corr)):
        if pval_matrix[i, j] < 0.001:
            ax.text(j, i, '●', ha='center', va='center', fontsize=5, color='white')
        elif pval_matrix[i, j] < 0.05:
            ax.text(j, i, '○', ha='center', va='center', fontsize=5, color='white')

plt.colorbar(im, ax=ax, fraction=0.025, label='Spearman r')
ax.set_title('Morphology × Attribution — Spearman correlations  (● p<0.001, ○ p<0.05)', 
             fontweight='bold')
plt.tight_layout()
pd.DataFrame(
    [{'morph_feature': avail_morph[_i].replace('_x',''), 'attr_feature': attr_for_corr[_j],
      'spearman_r': corr_matrix[_i, _j], 'p_value': pval_matrix[_i, _j]}
     for _i in range(len(avail_morph)) for _j in range(len(attr_for_corr))]
).to_csv(OUTPUT_DIR / '04a_morph_attr_heatmap_data.csv', index=False)

plt.savefig(OUTPUT_DIR / "04a_morph_attr_heatmap.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Top significant correlations sorted by |r|
corr_rows = []
for i, mc in enumerate(avail_morph):
    for j, ac in enumerate(attr_for_corr):
        if pval_matrix[i, j] < 0.05:
            corr_rows.append({'morph': mc.replace('_x',''), 
                               'attribution': ac,
                               'r': corr_matrix[i, j],
                               'p': pval_matrix[i, j]})

corr_top = (pd.DataFrame(corr_rows)
              .assign(abs_r=lambda x: x['r'].abs())
              .sort_values('abs_r', ascending=False)
              .drop(columns='abs_r')
              .head(20)
              .reset_index(drop=True))

corr_top.style \
    .format({'r': '{:.3f}', 'p': '{:.2e}'}) \
    .background_gradient(subset=['r'], cmap='RdBu_r', vmin=-0.5, vmax=0.5)

---
## 5 · Attribution ring vs intensity ring — does the model attend to bright regions?

Does the model attend to rings with higher fluorescence, or does it focus on dim regions?  
Each scatter compares the **intensity of a ring** (average pixel brightness) with the **attribution of that ring**.

In [ ]:
int_rings = [c for c in INT_RINGS if c in df.columns]

if int_rings:
    pairs = list(zip(int_rings, avail_rings))[:8]
    fig, axes = plt.subplots(2, 4, figsize=(FW, FW*7/15))
    axes_flat = axes.flatten()

    for idx, (ir, ar) in enumerate(pairs):
        ax = axes_flat[idx]
        for cls in CLASSES:
            sub = df[df['_class'] == cls][[ir, ar]].dropna()
            ax.scatter(sub[ir], sub[ar], alpha=0.35, s=8,
                       color=COLORS[cls], label=cls, rasterized=True,
                       marker=marker_class[cls])
        r_all, p_all = spearmanr(df[ir].dropna(), df[ar].dropna())
        r_pos, _    = spearmanr(pos[ir].dropna(), pos[ar].dropna())
        r_neg, _    = spearmanr(neg[ir].dropna(), neg[ar].dropna())
        ax.set_title(f'Ring {idx}\nr(all)={r_all:.2f}  AML={r_pos:.2f}  Ctrl={r_neg:.2f}',
                     fontsize=7.5)
        ax.set_xlabel('Intensity', fontsize=7)
        ax.set_ylabel('Attribution', fontsize=7)
        ax.axhline(0, color='k', lw=0.6, ls='--')
        if idx == 0:
            ax.legend(fontsize=7, markerscale=2)

    plt.suptitle('Attribution vs intensity per ring  (Spearman r)', 
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    _rows_05 = []
    for _idx, (_ir, _ar) in enumerate(pairs):
        _r_all, _ = spearmanr(df[_ir].dropna(), df[_ar].dropna())
        _r_pos, _ = spearmanr(pos[_ir].dropna(), pos[_ar].dropna())
        _r_neg, _ = spearmanr(neg[_ir].dropna(), neg[_ar].dropna())
        _rows_05.append({'ring': _idx, 'intensity_col': _ir, 'attribution_col': _ar,
                          'r_all': _r_all, 'r_AML': _r_pos, 'r_Control': _r_neg})
    pd.DataFrame(_rows_05).to_csv(OUTPUT_DIR / '05_attribution_vs_intensity_data.csv', index=False)

    plt.savefig(OUTPUT_DIR / "05_attribution_vs_intensity.pdf", bbox_inches="tight")
plt.show()

---
## 6 · Patient-level annotations

Correlate attribution with clinical measurements: **blast percentage** in bone marrow (BM) and peripheral blood (PB).

In [ ]:
# ── § 6a: blast % vs attribution — patient-level (AML only) ─────────────────
# Blast % is a patient-level measurement; correlating it at cell level inflates
# n and gives anti-conservative p-values (pseudo-replication).
# Fix: median attribution per patient, then Spearman — n = patients.

clin_cols  = [c for c in ['bm_blast_percentage', 'pb_blast_percentage'] if c in df.columns]
attr_feats = [c for c in ['attr_pos_sum', 'attr_total_abs',
                           'attr_pos_neg_ratio', 'attr_pos_gini'] if c in df.columns]
patient_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'
src_col     = 'source_x'    if 'source_x'    in df.columns else 'source'

if clin_cols and patient_col in df.columns:
    aml = df[df['_class'] == 'AML'].copy()

    # Pick blast % matched to sample source; fall back to whichever is present
    def _pick_blast(row):
        src = row.get(src_col, None)
        if src == 'BM' and 'bm_blast_percentage' in row.index and pd.notna(row['bm_blast_percentage']):
            return row['bm_blast_percentage']
        if src == 'PB' and 'pb_blast_percentage' in row.index and pd.notna(row['pb_blast_percentage']):
            return row['pb_blast_percentage']
        for c in ['bm_blast_percentage', 'pb_blast_percentage']:
            if c in row.index and pd.notna(row[c]):
                return row[c]
        return float('nan')

    aml['_blast_pct'] = aml.apply(_pick_blast, axis=1)

    # Aggregate to patient level
    pat_aml = (aml.dropna(subset=['_blast_pct'])
                  .groupby([patient_col, '_blast_pct'])[attr_feats]
                  .median()
                  .reset_index())

    fig, axes = plt.subplots(1, len(attr_feats),
                              figsize=(5.5 * len(attr_feats), 4.5))
    if len(attr_feats) == 1:
        axes = [axes]

    for ax, feat in zip(axes, attr_feats):
        sub = pat_aml[[patient_col, '_blast_pct', feat]].dropna()
        if len(sub) < 4:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', transform=ax.transAxes)
            continue

        r, p = spearmanr(sub['_blast_pct'], sub[feat])
        star  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''

        ax.scatter(sub['_blast_pct'], sub[feat],
                   s=60, alpha=0.8, color=COLORS['AML'],
                   edgecolors='white', linewidths=0.5, zorder=3)

        m, b = np.polyfit(sub['_blast_pct'], sub[feat], 1)
        xr   = np.linspace(sub['_blast_pct'].min(), sub['_blast_pct'].max(), 100)
        ax.plot(xr, m*xr + b, color=COLORS['AML'], lw=1.5, ls='--', alpha=0.9)

        ax.text(0.04, 0.96,
                f'r={r:.2f}{star}\nn={len(sub)} patients',
                transform=ax.transAxes, fontsize=8.5, va='top',
                color=COLORS['AML'])

        ax.set_xlabel('Blast % (source-matched)', fontsize=9)
        ax.set_ylabel(feat.replace('attr_', '').replace('_', ' '), fontsize=9)
        ax.set_title(feat.replace('attr_', '').replace('_', ' '), fontweight='bold')

    plt.suptitle(
        'Blast % vs attribution features\n'
        'AML only · patient-level medians · source-matched blast %',
        fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "06a_blast_correlation.pdf", bbox_inches="tight")
    plt.show()

    pat_aml.to_csv(OUTPUT_DIR / "06a_blast_patient_level.csv", index=False)
    print(f"Saved {len(pat_aml)} patient rows to 06a_blast_patient_level.csv")
else:
    print("No blast % or patientID column — skipping.")


In [ ]:
# Patient-level ring profile heatmap
patient_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

if patient_col in df.columns and avail_rings:
    pat_df = (df.groupby([patient_col, '_class'])[avail_rings]
                .mean()
                .reset_index()
                .sort_values('_class'))

    vabs = pat_df[avail_rings].abs().to_numpy()
    vmax = np.nanpercentile(vabs, 95)

    fig, ax = plt.subplots(figsize=(FW, max(6, len(pat_df) * 0.28) * FW/10))
    im = ax.imshow(pat_df[avail_rings].values, aspect='auto',
                   cmap=CMAP_OCC, norm=TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax))
    ax.set_xticks(range(len(avail_rings)))
    ax.set_xticklabels([f'R{i}' for i in range(len(avail_rings))])
    ax.set_yticks(range(len(pat_df)))
    ax.set_yticklabels(pat_df[patient_col].astype(str), fontsize=5.5)
    for tick, cls in zip(ax.get_yticklabels(), pat_df['_class']):
        tick.set_color(COLORS.get(cls, 'black'))
    plt.colorbar(im, ax=ax, fraction=0.025, label='Mean AML-class attribution')
    ax.set_title('Patient-level mean ring AML-class attribution  (label colour = class)', fontweight='bold')
    plt.tight_layout()
    pat_df.to_csv(OUTPUT_DIR / '06b_patient_heatmap_data.csv', index=False)
    plt.savefig(OUTPUT_DIR / "06b_patient_heatmap.pdf", bbox_inches="tight")
plt.show()


---
## 7 · Mutation-group analysis (AML cells only)

Within AML, do specific mutations alter how the model processes cells?  
Here we split AML cells by mutation status (NPM1, FLT3, DNMT3A, etc.) and compare attribution features.

> Only AML cells are used (positive class). Control cells lack these mutation records.

In [ ]:
# ── § 7: mutation-group analysis — patient-level (AML only) ──────────────────
# Mutation status (NPM1, FLT3, …) is a patient property, not a cell property.
# Multiple cells from the same patient share identical mutation flags, so
# cell-level MWU inflates n. Fix: aggregate to patient-level medians first.

avail_mut   = [c for c in MUT_COLS if c in df.columns]
patient_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'
feat_box    = [c for c in ['attr_pos_neg_ratio', 'attr_pos_gini',
                             'attr_pos_centroid_offset'] if c in df.columns]

if avail_mut and patient_col in pos.columns:
    # Aggregate cells → patient medians; keep mutation flags via first()
    agg = {c: 'median' for c in avail_rings + feat_box if c in pos.columns}
    agg.update({mc: 'first' for mc in avail_mut})
    pat_pos = pos.groupby(patient_col).agg(agg).reset_index()

    fig, axes = plt.subplots(len(avail_mut), 2,
                              figsize=(12, 3.2 * len(avail_mut)))
    if len(avail_mut) == 1:
        axes = [axes]

    for row_axes, mc in zip(axes, avail_mut):
        flag = pat_pos[mc].fillna(0).astype(bool)
        wt   = pat_pos[~flag]
        mut  = pat_pos[flag]

        # ── Ring profiles ─────────────────────────────────────────────────────
        ax = row_axes[0]
        for grp, lbl, col in [
            (wt,  f'WT (n={len(wt)} patients)',   '#7f7f7f'),
            (mut, f'{mc}+ (n={len(mut)} patients)', '#e377c2'),
        ]:
            if len(grp) < 3:
                continue
            m = grp[avail_rings].mean().values
            s = grp[avail_rings].sem().values
            ax.plot(RING_X[:len(avail_rings)], m, 'o-', label=lbl, lw=2, color=col)
            ax.fill_between(RING_X[:len(avail_rings)], m-s, m+s, alpha=0.2, color=col)

        ax.axhline(0, color='k', lw=0.7, ls='--')
        ax.set_xticks(RING_X[:len(avail_rings)])
        ax.set_xticklabels([f'R{i}' for i in range(len(avail_rings))])
        ax.set_title(f'{mc} — ring profile', fontweight='bold')
        ax.set_ylabel('Mean attribution (patient medians)')
        ax.legend(fontsize=8)

        # ── Boxplots of attribution features ──────────────────────────────────
        ax2   = row_axes[1]
        avail_box = [f for f in feat_box if f in pat_pos.columns]
        if avail_box and len(wt) > 2 and len(mut) > 2:
            x = np.arange(len(avail_box)); w = 0.35
            for xi, fc in enumerate(avail_box):
                col_wt  = wt[fc].dropna().values
                col_mut = mut[fc].dropna().values
                for vals, pos_x, fc_col in [
                    (col_wt,  xi - w/2, '#aec7e8'),
                    (col_mut, xi + w/2, '#e377c2'),
                ]:
                    ax2.boxplot(vals, positions=[pos_x], widths=w*0.9,
                                patch_artist=True,
                                boxprops=dict(facecolor=fc_col, alpha=0.75),
                                medianprops=dict(color='black', lw=2),
                                flierprops=dict(marker='o', ms=3, alpha=0.5))
                star, _ = mwu_stars(pd.Series(col_wt), pd.Series(col_mut))
                ymax = max(np.nanpercentile(col_wt, 90) if len(col_wt) else 0,
                           np.nanpercentile(col_mut, 90) if len(col_mut) else 0)
                ax2.text(xi, ymax * 1.05, star, ha='center', fontsize=10)

            ax2.set_xticks(x)
            ax2.set_xticklabels([f.replace('attr_', '') for f in avail_box], fontsize=8)
            ax2.set_title(f'{mc} — attribution features', fontweight='bold')
            from matplotlib.patches import Patch
            ax2.legend(handles=[Patch(color='#aec7e8', alpha=0.75, label='WT'),
                                 Patch(color='#e377c2', alpha=0.75, label=f'{mc}+')],
                       fontsize=8)

    plt.suptitle(
        'Attribution by mutation status \n'
        'Ring profiles = mean ± SEM across patients; boxplots = patient medians',
        fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "07_mutations.pdf", bbox_inches="tight")
    plt.show()

    # Save patient-level mutation table
    pat_pos[[patient_col] + avail_mut + avail_rings + feat_box].to_csv(
        OUTPUT_DIR / "07_mutations_patient_level.csv", index=False)
    print(f"Saved {len(pat_pos)} patient rows to 07_mutations_patient_level.csv")
else:
    print("No mutation columns or patientID — skipping.")


---
## 7b · Formal mutation-group shape analysis (patient-level)

**Confirmatory (n ≥ 9, formally pre-specified):** NPM1, TET2, DNMT3A, NRAS, FLT3

**Exploratory (n ≤ 4):** all other mutations — Cliff's delta / HL shift only, no p-tests

**BH-FDR families:**
- Family A: permutation functional shape tests (5 mutations × 1 = 5 tests)
- Family B: Mann-Whitney U on summary features (5 mutations × k features)

**Inferential unit:** patient-level medians throughout (avoids pseudoreplication).

In [ ]:
# ── § 7b-A: data prep + statistical helpers ──────────────────────────────────
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import warnings; warnings.filterwarnings('ignore')

_RNG   = np.random.default_rng(42)
N_PERM = 2000
N_BOOT = 2000

FORMAL_MUT  = ['NPM1', 'TET2', 'DNMT3A', 'NRAS', 'FLT3']   # n>=9 pre-specified
_avail_mut  = [c for c in MUT_COLS if c in df.columns and df[c].notna().any()]
formal_mut  = [m for m in FORMAL_MUT if m in _avail_mut]
small_n_mut = [m for m in _avail_mut  if m not in formal_mut]

MUTATION_COLORS = {          # Tableau-10: consistent across all §7 figures
    'NPM1':   '#4e79a7',
    'TET2':   '#f28e2b',
    'DNMT3A': '#59a14f',
    'NRAS':   '#e15759',
    'FLT3':   '#b07aa1',
}
_COLOR_WT   = '#7f7f7f'   # WT / baseline
_COLOR_EXPL = '#bab0ac'   # small-n exploratory

def mut_color(mc): return MUTATION_COLORS.get(mc, _COLOR_EXPL)
_pat_col    = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

_feat_cols     = [c for c in ATTR_SUMMARY    if c in pos.columns]
_ring_cols     = [c for c in avail_rings     if c in pos.columns]
_ring_cols_pos = [c for c in avail_rings_pos if c in pos.columns]
_ring_cols_neg = [c for c in avail_rings_neg if c in pos.columns]
_agg = {c: 'median' for c in _ring_cols + _ring_cols_pos + _ring_cols_neg + _feat_cols}
_agg.update({c: 'first' for c in _avail_mut if c in pos.columns})
pat_mut = pos.groupby(_pat_col).agg(_agg).reset_index()

print(f"AML patients: {len(pat_mut)}")
for m in formal_mut:
    flag = pat_mut[m].fillna(0).astype(bool)
    print(f"  {m:<10} mut={flag.sum()}  wt={(~flag).sum()}")

# Cliff's delta + bootstrap CI
def cliff_delta(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    return ((x[:,None] > y[None,:]).sum() - (x[:,None] < y[None,:]).sum()) / (len(x)*len(y))

def cliff_delta_ci(x, y, rng=_RNG, n_boot=N_BOOT):
    x, y = np.asarray(x, float), np.asarray(y, float)
    d    = cliff_delta(x, y)
    boot = [cliff_delta(rng.choice(x, len(x), replace=True),
                        rng.choice(y, len(y), replace=True)) for _ in range(n_boot)]
    return d, float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

# Hodges-Lehmann shift + bootstrap CI
def hl_shift(x, y, rng=_RNG, n_boot=N_BOOT):
    x, y  = np.asarray(x, float), np.asarray(y, float)
    diffs = (x[:,None] - y[None,:]).ravel()
    hl    = float(np.median(diffs))
    boot  = [float(np.median(rng.choice(diffs, len(diffs), replace=True)))
             for _ in range(n_boot)]
    return hl, float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

# Permutation functional shape test (IWT/GET equivalent)
def perm_shape_test(p_wt, p_mut, n_perm=N_PERM, rng=_RNG):
    """
    Two-sample permutation test on 8-ring patient profiles.
    Returns: obs_diff, p_pointwise, p_maxT (simultaneous), null_diffs
    Max-T simultaneous correction = Westfall-Young / equivalent to IWT2 envelope.
    """
    p0, p1 = np.asarray(p_wt), np.asarray(p_mut)
    n0, n1 = len(p0), len(p1)
    pool   = np.vstack([p0, p1])
    obs    = p1.mean(0) - p0.mean(0)
    nulls  = np.zeros((n_perm, pool.shape[1]))
    for b in range(n_perm):
        idx       = rng.permutation(n0 + n1)
        nulls[b]  = pool[idx[n0:]].mean(0) - pool[idx[:n0]].mean(0)
    p_pt  = np.mean(np.abs(nulls) >= np.abs(obs)[None,:], axis=0)
    maxT  = np.max(np.abs(nulls), axis=1)
    p_sim = np.array([np.mean(maxT >= abs(d)) for d in obs])
    return obs, p_pt, p_sim, nulls

print("Helpers ready.")


In [ ]:
# ── § 7b-B: Confirmatory shape tests + MWU + effect sizes + BH-FDR ───────────

# Ring columns available in pat_mut
_rc_net = [c for c in _ring_cols     if c in pat_mut.columns]
_rc_pos = [c for c in avail_rings_pos if c in pat_mut.columns]
_rc_neg = [c for c in avail_rings_neg if c in pat_mut.columns]
print(f'Ring cols — net:{len(_rc_net)}  pos:{len(_rc_pos)}  neg:{len(_rc_neg)}')

def _run_shape(rc, label):
    rows = []
    for mc in formal_mut:
        flag  = pat_mut[mc].fillna(0).astype(bool)
        wt_p  = pat_mut[~flag]; mut_p = pat_mut[flag]
        _cols = [c for c in rc if c in pat_mut.columns]
        if not _cols or len(mut_p) < 2 or len(wt_p) < 2:
            continue
        obs, p_pt, p_sim, null_d = perm_shape_test(
            wt_p[_cols].values, mut_p[_cols].values)
        best_r = int(np.argmin(p_sim))
        xm = mut_p[_cols].mean(1).values; xw = wt_p[_cols].mean(1).values
        d, lo, hi        = cliff_delta_ci(xm, xw)
        hl, hl_lo, hl_hi = hl_shift(xm, xw)
        rows.append({'attr_type': label, 'mutation': mc,
                     'n_mut': len(mut_p), 'n_wt': len(wt_p),
                     'global_p': float(p_sim[best_r]),
                     'best_ring': f'R{best_r}',
                     'min_p_pt':  float(np.min(p_pt)),
                     'cliff_d':  round(d, 3), 'cliff_lo': round(lo, 3), 'cliff_hi': round(hi, 3),
                     'HL_shift': round(hl, 4), 'HL_lo': round(hl_lo, 4), 'HL_hi': round(hl_hi, 4),
                     '_cols': _cols, '_obs': obs, '_p_pt': p_pt,
                     '_p_sim': p_sim, '_null': null_d})
    if not rows:
        return rows, pd.DataFrame()
    _df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                         for r in rows])
    _, _df['p_adj_BH'], _, _ = multipletests(_df['global_p'], method='fdr_bh')
    _df['p_adj_BH'] = _df['p_adj_BH'].round(4)
    _df['sig'] = _df['p_adj_BH'].apply(
        lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns')
    return rows, _df

shape_rows_net, shape_df_net = _run_shape(_rc_net, 'net')
shape_rows_pos, shape_df_pos = _run_shape(_rc_pos, 'pos')
shape_rows_neg, shape_df_neg = _run_shape(_rc_neg, 'neg')

# Backward-compat aliases used by downstream cells
shape_rows = shape_rows_net
shape_df   = shape_df_net.rename(columns={'sig': 'sig_A'}, errors='ignore').copy()
if 'sig_A' not in shape_df.columns and 'sig' in shape_df_net.columns:
    shape_df['sig_A'] = shape_df_net['sig'].values

# Family B: MWU on each summary feature (net attribution; unchanged)
feat_rows = []
for mc in formal_mut:
    flag  = pat_mut[mc].fillna(0).astype(bool)
    wt_p  = pat_mut[~flag]; mut_p = pat_mut[flag]
    for fc in _feat_cols:
        xm_f = mut_p[fc].dropna().values; xw_f = wt_p[fc].dropna().values
        if len(xm_f) < 3 or len(xw_f) < 3:
            continue
        _, p_mwu = mannwhitneyu(xm_f, xw_f, alternative='two-sided')
        df2, lo2, hi2   = cliff_delta_ci(xm_f, xw_f)
        hl2, lo_h, hi_h = hl_shift(xm_f, xw_f)
        feat_rows.append({'mutation': mc, 'feature': fc,
                          'n_mut': len(xm_f), 'n_wt': len(xw_f),
                          'MWU_p': round(p_mwu, 4),
                          'cliff_d':  round(df2, 3), 'cliff_lo': round(lo2, 3), 'cliff_hi': round(hi2, 3),
                          'HL_shift': round(hl2, 5), 'HL_lo': round(lo_h, 5), 'HL_hi': round(hi_h, 5)})

feat_df = pd.DataFrame(feat_rows)
_, feat_df['p_adj_BH'], _, _ = multipletests(feat_df['MWU_p'], method='fdr_bh')
feat_df['p_adj_BH'] = feat_df['p_adj_BH'].round(4)
feat_df['sig_B'] = feat_df['p_adj_BH'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns')

for lbl, _df in [('net', shape_df_net), ('pos', shape_df_pos), ('neg', shape_df_neg)]:
    if _df.empty: continue
    print(f'\n=== Family A ({lbl}) ===')
    print(_df[['mutation','global_p','p_adj_BH','sig','cliff_d','cliff_lo','cliff_hi']].to_string(index=False))

print('\n=== Family B: BH-significant MWU features ===')
sig_b = feat_df[feat_df['sig_B'] != 'ns']
print(sig_b[['mutation','feature','MWU_p','p_adj_BH','sig_B','cliff_d']].to_string(index=False)
      if len(sig_b) else '  None')

pd.concat([shape_df_net, shape_df_pos, shape_df_neg]).to_csv(
    OUTPUT_DIR / '07b_shape_tests.csv', index=False)
feat_df.to_csv(OUTPUT_DIR / '07b_feature_tests.csv', index=False)
print('\nSaved 07b_shape_tests.csv  07b_feature_tests.csv')


In [ ]:
# ── § 7b-C: Plots ───────────────────────────────────────────────────────────
import matplotlib.patches as mpatches

ATTR_SPECS = [
    ('net', 'Net attribution',       shape_rows_net, shape_df_net, _rc_net),
    ('pos', 'Supporting (+) attr.',  shape_rows_pos, shape_df_pos, _rc_pos),
    ('neg', 'Suppressing (−) attr.', shape_rows_neg, shape_df_neg, _rc_neg),
]

# -- Figure 8A: net attribution ring profiles (main text) --------------------
x_rings = RING_X[:len(_rc_net)]
x_lbl   = [f'R{i}' for i in range(len(_rc_net))]

fig, axes = plt.subplots(1, len(formal_mut),
                          figsize=(4.5 * len(formal_mut), 4.5),
                          sharey=False)
if len(formal_mut) == 1: axes = [axes]

row_map_net = {r['mutation']: r for r in shape_rows_net}
for ax, mc in zip(axes, formal_mut):
    if mc not in row_map_net:
        ax.set_visible(False); continue
    row   = row_map_net[mc]
    rc    = row['_cols']
    flag  = pat_mut[mc].fillna(0).astype(bool)
    wt_v  = pat_mut[~flag][rc].values
    mut_v = pat_mut[flag][rc].values
    p_pt  = row['_p_pt']

    for vals, lbl, col, ls in [
            (wt_v,  f'WT (n={len(wt_v)})',     _COLOR_WT,    '--'),
            (mut_v, f'{mc}+ (n={len(mut_v)})', mut_color(mc), '-')]:
        m = vals.mean(0); s = vals.std(0) / np.sqrt(len(vals))
        ax.plot(x_rings, m, ls, color=col, lw=2, marker='o', ms=6, label=lbl)
        ax.fill_between(x_rings, m - s, m + s, alpha=0.18, color=col)

    for ri, pp in enumerate(p_pt):
        if pp < 0.05:
            ypos = max(wt_v.mean(0)[ri], mut_v.mean(0)[ri]) + 0.04
            ax.text(x_rings[ri], ypos, '*', ha='center', fontsize=11,
                    color=mut_color(mc))

    adj_p = float(shape_df_net.loc[shape_df_net['mutation'] == mc, 'p_adj_BH'].iloc[0])
    sig   = shape_df_net.loc[shape_df_net['mutation'] == mc, 'sig'].iloc[0]
    ax.text(0.03, 0.97, f'{mc}  p={adj_p:.3f} {sig}',
            transform=ax.transAxes, va='top', fontsize=9,
            fontweight='bold', color=mut_color(mc))
    ax.set_xticks(x_rings); ax.set_xticklabels(x_lbl, fontsize=8)
    ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.4)
    ax.set_xlabel('Ring (boundary → centre)')
    #if ax is axes[0]:
    ax.set_ylabel('Mean net attribution (patient medians)')
    ax.legend(fontsize=8, loc='lower left')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'Figure_8A.pdf', bbox_inches='tight')
plt.show()

# -- Figure 8B: Volcano — net attribution, broken axis -----------------------
# Net attribution only. Formal (n>=9): BH-FDR corrected p_adj.
# Exploratory: nominal MWU p. Single y-axis with auto-detected break.
SIG_THRESH = 0.05

formal_8b = []
for r in shape_rows_net:
    mc    = r['mutation']
    p_adj = float(shape_df_net.loc[shape_df_net['mutation'] == mc, 'p_adj_BH'].iloc[0])
    formal_8b.append(dict(mutation=mc, formal=True,
                           cliff_d=r['cliff_d'], cliff_lo=r['cliff_lo'],
                           cliff_hi=r['cliff_hi'], p_plot=p_adj))

explor_8b = []
_rc_net_mut = [c for c in _rc_net if c in pat_mut.columns]
for mc in small_n_mut:
    flag = pat_mut[mc].fillna(0).astype(bool)
    xm   = pat_mut[flag][_rc_net_mut].mean(1).dropna().values
    xw   = pat_mut[~flag][_rc_net_mut].mean(1).dropna().values
    if len(xm) < 2 or len(xw) < 2: continue
    d, lo, hi = cliff_delta_ci(xm, xw)
    _, p_raw  = mannwhitneyu(xm, xw, alternative='two-sided')
    explor_8b.append(dict(mutation=mc, formal=False,
                           cliff_d=round(d,3), cliff_lo=round(lo,3),
                           cliff_hi=round(hi,3), p_plot=p_raw))

fig8b_df = pd.DataFrame(formal_8b + explor_8b)
fig8b_df.to_csv(OUTPUT_DIR / 'Figure_8B_cliff_forest.csv', index=False)

thresh_y = -np.log10(SIG_THRESH)

# Pre-collect all points; detect break from data
_pts_8b = []
for _, row in fig8b_df.iterrows():
    y = -np.log10(max(float(row['p_plot']), 1e-10))
    _pts_8b.append((float(row['cliff_d']), y, row['mutation'],
                    mut_color(row['mutation']), bool(row['formal']),
                    float(row['cliff_lo']), float(row['cliff_hi'])))

_y_sorted = sorted({p[1] for p in _pts_8b})
_gaps = [(_y_sorted[i+1] - _y_sorted[i], _y_sorted[i], _y_sorted[i+1])
         for i in range(len(_y_sorted)-1)]
_best_gap = max(_gaps, key=lambda g: g[0]) if _gaps else (0, 0, 0)
_use_break = _best_gap[0] > 0.7

def _place_labels_v(ax, pts, fontsize=7.5, x_tol=0.12, y_step=0.08):
    by_x = sorted(pts, key=lambda p: p[0])
    top = {}
    for x, y, label, color in by_x:
        xbin = round(x / x_tol) * x_tol
        ly = y + 0.03
        if xbin in top:
            ly = max(ly, top[xbin] + y_step)
        top[xbin] = ly
        ax.text(x, ly, label, ha='center', va='bottom',
                fontsize=fontsize, color=color, fontweight='normal')

if _use_break:
    _y_lo  = _best_gap[1] + 0.15   # top of bottom panel
    _y_hi  = _best_gap[2] - 0.15   # bottom of top panel
    _y_max = _y_sorted[-1] + 0.6
    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, sharex=True, figsize=(6.5, 5.2),
        gridspec_kw={'height_ratios': [1, 3], 'hspace': 0.06})
    ax_top.set_ylim(_y_hi, _y_max)
    ax_bot.set_ylim(-0.15, _y_lo)
    ax_top.spines.bottom.set_visible(False)
    ax_bot.spines.top.set_visible(False)
    ax_top.tick_params(bottom=False, labelbottom=False)
    for _ax in (ax_top, ax_bot):
        _ax.spines[['top', 'right']].set_visible(False)
    d = 0.5
    _bkw = dict(marker=[(-1, -d), (1, d)], markersize=12,
                linestyle='none', color='k', mew=1, clip_on=False)
    ax_top.plot([0, 1], [0, 0], transform=ax_top.transAxes, **_bkw)
    ax_bot.plot([0, 1], [1, 1], transform=ax_bot.transAxes, **_bkw)
    _all_axes = [ax_top, ax_bot]
else:
    _y_lo = _y_hi = None
    fig, ax_bot = plt.subplots(figsize=(6.5, 4.5))
    ax_bot.spines[['top', 'right']].set_visible(False)
    ax_top = None
    _all_axes = [ax_bot]

for _ax in _all_axes:
    _ax.axhline(thresh_y, color='#c0392b', lw=0.9, ls='--', alpha=0.7, zorder=1)
    _ax.axvline(0,    color='k',    lw=0.7, ls='--', alpha=0.35, zorder=1)
    _ax.axvline(-0.3, color='gray', lw=0.7, ls=':',  alpha=0.55, zorder=1)
    _ax.axvline( 0.3, color='gray', lw=0.7, ls=':',  alpha=0.55, zorder=1)
    for x, y, mc, col, formal, lo, hi in _pts_8b:
        _ax.plot([lo, hi], [y, y], '-', color=col, lw=1.3, alpha=0.55, zorder=3)
        _ax.scatter(x, y, color=col, s=70,
                    marker='D' if formal else 'o',
                    alpha=0.90, zorder=4, linewidths=0.8,
                    edgecolors='white' if formal else 'none')

if _use_break:
    _l_top = [(x, y, mc, col) for x, y, mc, col, *_ in _pts_8b if y >= _y_hi]
    _l_bot = [(x, y, mc, col) for x, y, mc, col, *_ in _pts_8b if y <= _y_lo]
    if _l_top:
        _place_labels_v(ax_top, _l_top)
else:
    _l_bot = [(x, y, mc, col) for x, y, mc, col, *_ in _pts_8b]
_place_labels_v(ax_bot, _l_bot)

#ax_bot.text(1.01, thresh_y, ' p=0.05',
#            transform=ax_bot.get_yaxis_transform(), va='center',
#            fontsize=6.5, color='#c0392b')
ax_bot.set_xlabel("Cliff's delta  (mut+ vs WT,  mean net attribution)", fontsize=9)
ax_bot.set_ylabel('-log10(p)', fontsize=9)

from matplotlib.lines import Line2D
ax_bot.legend(handles=[
    Line2D([0],[0], marker='D', color='#666', ms=7, lw=0,
           markerfacecolor='#666', markeredgecolor='white',
           markeredgewidth=0.8, label='BH-FDR corrected p-values'),
    Line2D([0],[0], marker='o', color=_COLOR_EXPL, ms=6, lw=0,
           markerfacecolor=_COLOR_EXPL, markeredgecolor='none',
           label='Uncorrected p-values'),
           
        Line2D([0],[0], color='#c0392b', lw=0.9, ls='--', alpha=0.7, label='p=0.05'),
    Line2D([0],[0], color='#888', lw=1.3, alpha=0.6, label='95% bootstrap CI'),
], fontsize=7.5, loc='center left', framealpha=0.7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'Figure_8B.pdf', bbox_inches='tight')
plt.show()

# -- Figure S8A: net / supporting / suppressing profiles (supplement) ----------
# Robustness check: does net attribution mask an asymmetry between
# AML-promoting and AML-opposing spatial signals?
n_spec = sum(1 for _, _, rows, _, _ in ATTR_SPECS[1:] if rows)
n_mc   = len(formal_mut)
fig, axes_s = plt.subplots(n_spec, n_mc,
                            figsize=(4.0 * n_mc, 3.5 * n_spec),
                            sharey='row', squeeze=False)

ri = 0
for atype, albl, rows_list, df_type, rc_cols in ATTR_SPECS[1:]:  # pos and neg only
    if not rows_list: continue
    row_map = {r['mutation']: r for r in rows_list}
    for ci, mc in enumerate(formal_mut):
        ax = axes_s[ri][ci]
        if mc not in row_map: ax.set_visible(False); continue
        row   = row_map[mc]
        rc    = row['_cols']
        x_r   = RING_X[:len(rc)]
        x_l   = [f'R{i}' for i in range(len(rc))]
        flag  = pat_mut[mc].fillna(0).astype(bool)
        wt_v  = pat_mut[~flag][rc].values
        mut_v = pat_mut[flag][rc].values
        p_pt  = row['_p_pt']
        for vals, lbl, col, ls in [
                (wt_v,  f'WT (n={len(wt_v)})',     _COLOR_WT,    '--'),
                (mut_v, f'{mc}+ (n={len(mut_v)})', mut_color(mc), '-')]:
            m = vals.mean(0); s = vals.std(0) / np.sqrt(len(vals))
            ax.plot(x_r, m, ls, color=col, lw=2, marker='o', ms=5, label=lbl)
            ax.fill_between(x_r, m - s, m + s, alpha=0.18, color=col)
        for rii, pp in enumerate(p_pt):
            if pp < 0.05:
                ypos = max(wt_v.mean(0)[rii], mut_v.mean(0)[rii]) + 0.04
                ax.text(x_r[rii], ypos, '*', ha='center', fontsize=10, color=mut_color(mc))
        adj_p = float(df_type.loc[df_type['mutation'] == mc, 'p_adj_BH'].iloc[0])
        sig   = df_type.loc[df_type['mutation'] == mc, 'sig'].iloc[0]
        ax.text(0.03, 0.97, f'{mc}  p={adj_p:.3f} {sig}',
                transform=ax.transAxes, va='top', fontsize=8.5,
                fontweight='bold', color=mut_color(mc))
        ax.set_xticks(x_r); ax.set_xticklabels(x_l, fontsize=7.5)
        ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.4)
        ax.set_ylabel(albl, fontsize=9)
        ax.legend(fontsize=6.5, loc='best')
        if ri == n_spec - 1:
            ax.set_xlabel('Ring (boundary → centre)', fontsize=8)
    ri += 1

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'Figure_S8A.pdf', bbox_inches='tight')
plt.show()
print('Saved Figure_8A.pdf  Figure_8B.pdf  Figure_S8A.pdf')


In [ ]:
# ── § 7b-D: LMM sensitivity (group x ring, cluster permutation) ──────────────
# Permutes mutation label at patient level — valid for clustered cell data.
# Use as sensitivity; concordance with Family-A shape test strengthens claims.
try:
    from statsmodels.formula.api import mixedlm

    lmm_rows = []
    for mc in formal_mut:
        flag_map = pos.groupby(_pat_col)[mc].first().fillna(0).astype(int)
        _df      = pos[[_pat_col] + _ring_cols].copy().dropna()
        _df['mut'] = _df[_pat_col].map(flag_map)

        # Long format: one row per (cell, ring)
        long_parts = []
        for ri, rc in enumerate(_ring_cols):
            t = _df[[_pat_col, 'mut', rc]].rename(columns={rc: 'y'})
            t['ring'] = ri; long_parts.append(t)
        ld = pd.concat(long_parts, ignore_index=True)
        ld['ring'] = ld['ring'].astype('category')

        def _fit_stat(df_in):
            try:
                res = mixedlm('y ~ C(ring) * mut', df_in,
                              groups=df_in[_pat_col]).fit(
                              reml=False, method='lbfgs', maxiter=300)
                inter = [k for k in res.params.index if 'mut' in k and 'ring' in k.lower()]
                return float(np.sum(res.params[inter]**2)) if inter else 0.0
            except Exception:
                return 0.0

        obs_stat = _fit_stat(ld)
        pats     = ld[_pat_col].unique()
        pat_labs = ld.groupby(_pat_col)['mut'].first()
        n_lmm    = 200
        null_s   = []
        for _ in range(n_lmm):
            pmap   = dict(zip(pats, _RNG.permutation(pat_labs.loc[pats].values)))
            ld2    = ld.copy(); ld2['mut'] = ld2[_pat_col].map(pmap)
            null_s.append(_fit_stat(ld2))
        p_perm = float(np.mean(np.array(null_s) >= obs_stat))
        lmm_rows.append({'mutation': mc, 'obs_stat': round(obs_stat,4),
                          'p_cluster_perm': round(p_perm,3), 'n_perm': n_lmm})
        print(f'{mc}: stat={obs_stat:.4f}  p_cluster_perm={p_perm:.3f}')

    lmm_df = pd.DataFrame(lmm_rows)
    lmm_df.to_csv(OUTPUT_DIR / '07b_lmm_sensitivity.csv', index=False)
    print("\nSaved 07b_lmm_sensitivity.csv")
    print(lmm_df.to_string(index=False))

except ImportError:
    print("statsmodels not found — install with: pip install statsmodels")

# ── Methods paragraph ─────────────────────────────────────────────────────────
print("""
══════════════════════════════════════════════════════════════════
SUPPLEMENTARY METHODS — Mutation-group attribution analysis
══════════════════════════════════════════════════════════════════
Only AML cells were used; mutation status is a patient-level covariate.
All tests used patient-level medians as the inferential unit to avoid
pseudoreplication from multiple cells per patient.

Formal-testing threshold: mutations with ≥9 patients in the positive arm
(NPM1, TET2, DNMT3A, NRAS, FLT3) were designated for confirmatory
analysis. Mutations with ≤4 positive patients are exploratory; only
effect sizes are reported.

Family A — Shape tests: A two-sample permutation test was applied to the
8-ring patient-median attribution profiles (N=2000 permutations). At each
ring, the group-mean difference (mut − WT) was computed; simultaneous
(max-T) correction across rings yields ring-level p-values equivalent to
the Interval-Wise Testing framework (Pini & Vantini 2017). BH-FDR was
applied across the 5 tests in Family A (q < 0.05).

Family B — Summary features: Mann-Whitney U tests on patient-level
feature medians; BH-FDR across all mutation × feature combinations.
Effect sizes: Cliff's delta and Hodges-Lehmann shift, each with 95%
bootstrap CI (N=2000).

Sensitivity — LMM: A linear mixed model (random patient intercept,
group × ring interaction) was fitted to cell-level data; mutation labels
were permuted at the patient (cluster) level (N=200) for valid inference.
Concordance with Family A supports robustness of any detected signal.

Exploratory small-n: Cliff's delta with 95% bootstrap CI reported in
supplementary forest plot; CIs are wide and imprecise at n≤4.
══════════════════════════════════════════════════════════════════
""")


---
## §7c · Mutation × DAPI (chromatin density) analyses

Four analyses extending the mutation attribution work in §7b:

- **A**: Per-ring DAPI z-score profiles by mutation (chromatin architecture)
- **B**: Model confidence (prediction score) by mutation
- **C**: Chromatin density gradient by mutation (centre/boundary ratio and slope)
- **D**: Mediation — does DAPI gradient mediate mutation → attribution?


In [ ]:
# ── § 7c-A: DAPI ring profiles by mutation (chromatin architecture) ────────
# Tests whether mutation status affects per-ring DAPI z-score independently of
# attribution. Same permutation shape test as §7b (max-T simultaneous correction),
# on patient-level medians. Requires variables from § 7b-A (cell 33).

_int_cols = [c for c in INT_RINGS if c in df.columns]   # DAPI ring columns

# Patient-level DAPI medians (AML only)
pat_dapi = pos.groupby(_pat_col)[_int_cols].median().reset_index()
pat_dapi = pat_dapi.merge(
    pat_mut[[_pat_col] + [c for c in _avail_mut if c in pat_mut.columns]],
    on=_pat_col, how='left'
)

dapi_shape_rows = []
for mc in formal_mut:
    if mc not in pat_dapi.columns:
        continue
    flag  = pat_dapi[mc].fillna(0).astype(bool)
    wt_d  = pat_dapi[~flag][_int_cols].values
    mut_d = pat_dapi[flag][_int_cols].values
    if len(mut_d) < 3 or len(wt_d) < 3:
        continue
    obs, p_pt, p_sim, null_d = perm_shape_test(wt_d, mut_d)
    best_r = int(np.argmin(p_sim))
    d, lo, hi = cliff_delta_ci(mut_d.mean(1), wt_d.mean(1))
    dapi_shape_rows.append({
        'mutation': mc, 'n_mut': int(len(mut_d)), 'n_wt': int(len(wt_d)),
        'global_p': float(p_sim[best_r]), 'best_ring': f'R{best_r}',
        'cliff_d': round(d, 3), 'cliff_lo': round(lo, 3), 'cliff_hi': round(hi, 3),
        '_obs': obs, '_p_pt': p_pt, '_p_sim': p_sim, '_null': null_d,
    })

dapi_shape_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                                for r in dapi_shape_rows])
if len(dapi_shape_df):
    _, dapi_shape_df['p_adj_BH'], _, _ = multipletests(
        dapi_shape_df['global_p'], method='fdr_bh')
    dapi_shape_df['p_adj_BH'] = dapi_shape_df['p_adj_BH'].round(4)
    dapi_shape_df['sig'] = dapi_shape_df['p_adj_BH'].apply(
        lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns')

    print('=== §7c-A: DAPI ring shape tests (BH-FDR, Family C) ===')
    print(dapi_shape_df[['mutation','n_mut','n_wt','global_p','p_adj_BH','sig',
                           'best_ring','cliff_d','cliff_lo','cliff_hi']].to_string(index=False))

    x_rings = RING_X[:len(_int_cols)]
    fig, axes = plt.subplots(1, len(dapi_shape_rows),
                              figsize=(4.5 * len(dapi_shape_rows), 4.5), sharey=False)
    if len(dapi_shape_rows) == 1:
        axes = [axes]

    for ax, row in zip(axes, dapi_shape_rows):
        mc    = row['mutation']
        flag  = pat_dapi[mc].fillna(0).astype(bool)
        wt_v  = pat_dapi[~flag][_int_cols].values
        mut_v = pat_dapi[flag][_int_cols].values
        env_lo, env_hi = np.percentile(row['_null'], [2.5, 97.5], axis=0)

        for vals, lbl, col, ls in [
            (wt_v,  f'WT (n={len(wt_v)})',    '#7f7f7f', '--'),
            (mut_v, f'{mc}+ (n={len(mut_v)})', mut_color(mc), '-'),
        ]:
            m = vals.mean(0); s = vals.std(0) / np.sqrt(len(vals))
            ax.plot(x_rings, m, ls, color=col, lw=2, marker='o', ms=6, label=lbl)
            ax.fill_between(x_rings, m - s, m + s, alpha=0.18, color=col)


        for ri, pp in enumerate(row['_p_pt']):
            if pp < 0.05:
                ypos = max(wt_v.mean(0)[ri], mut_v.mean(0)[ri]) + 0.05
                ax.text(x_rings[ri], ypos, '*', ha='center', fontsize=11, color=mut_color(mc))

        sig  = dapi_shape_df.loc[dapi_shape_df['mutation'] == mc, 'sig'].iloc[0]
        padj = dapi_shape_df.loc[dapi_shape_df['mutation'] == mc, 'p_adj_BH'].iloc[0]
        ax.set_title(f'{mc}  p_adj={padj:.3f} {sig}', fontweight='bold', fontsize=10)
        ax.set_xticks(x_rings)
        ax.set_xticklabels([f'R{i}' for i in range(len(_int_cols))], fontsize=8)
        ax.set_xlabel('Ring (boundary → centre)')
        ax.set_ylabel('Mean DAPI z-score (patient medians)')
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'Figure_S8B.pdf', bbox_inches='tight')
    plt.show()

    dapi_shape_df.to_csv(OUTPUT_DIR / 'Figure_S8B_dapi_shape_tests.csv', index=False)
    print('Saved → 07c_dapi_shape_tests.csv  |  07c_dapi_mutation_shape_profiles.pdf')


In [ ]:
# ── § 7c-B: Model confidence (prediction score) by mutation ────────
# Tests whether DenseNet-121 assigns higher AML-class probability to specific
# mutation genotypes. Uses prediction_score per cell, aggregated to patient median.
# Formal mutations (n>=9): BH-FDR corrected. Small-n mutations: Cliff's delta only.

score_col = 'prediction_score'
if score_col not in pos.columns:
    print(f"'{score_col}' not found in pos — skipping §7c-B")
else:
    pat_score = (pos.groupby(_pat_col)[[score_col]]
                   .median()
                   .rename(columns={score_col: 'med_score'})
                   .reset_index())
    pat_score = pat_score.merge(
        pat_mut[[_pat_col] + [c for c in _avail_mut if c in pat_mut.columns]],
        on=_pat_col, how='left'
    )

    conf_rows = []
    for mc in formal_mut + small_n_mut:
        if mc not in pat_score.columns:
            continue
        flag = pat_score[mc].fillna(0).astype(bool)
        xm   = pat_score.loc[flag,  'med_score'].dropna().values
        xw   = pat_score.loc[~flag, 'med_score'].dropna().values
        if len(xm) < 2 or len(xw) < 2:
            continue
        d, lo, hi = cliff_delta_ci(xm, xw)
        _, p_mwu  = mannwhitneyu(xm, xw, alternative='two-sided')
        conf_rows.append({
            'mutation': mc, 'n_mut': len(xm), 'n_wt': len(xw),
            'med_mut': round(float(np.median(xm)), 3),
            'med_wt':  round(float(np.median(xw)), 3),
            'MWU_p':   round(p_mwu, 4),
            'cliff_d': round(d, 3), 'cliff_lo': round(lo, 3), 'cliff_hi': round(hi, 3),
            'formal':  mc in formal_mut,
        })

    conf_df = pd.DataFrame(conf_rows)
    conf_df['p_adj_BH'] = np.nan
    fm = conf_df['formal'].values
    if fm.sum() > 0:
        _, adj_p, _, _ = multipletests(conf_df.loc[fm, 'MWU_p'], method='fdr_bh')
        conf_df.loc[fm, 'p_adj_BH'] = np.round(adj_p, 4)
    conf_df['sig'] = conf_df.apply(
        lambda r: ('***' if r['p_adj_BH'] < 0.001 else '**' if r['p_adj_BH'] < 0.01
                   else '*' if r['p_adj_BH'] < 0.05 else 'ns') if r['formal']
        else 'expl.', axis=1)

    print('=== §7c-B: Model confidence by mutation ===')
    print(conf_df[['mutation','n_mut','n_wt','med_mut','med_wt',
                    'MWU_p','p_adj_BH','sig','cliff_d','cliff_lo','cliff_hi']].to_string(index=False))

    # -- Figure S8B: Volcano — model confidence, broken axis --------------------
    # Formal (n>=9): BH-FDR corrected p_adj. Exploratory: nominal MWU p.
    # Single y-axis with auto-detected break.
    SIG_THRESH = 0.05

    plot_conf = conf_df.copy()
    plot_conf['p_plot'] = np.where(
        plot_conf['formal'], plot_conf['p_adj_BH'], plot_conf['MWU_p'])

    thresh_y = -np.log10(SIG_THRESH)

    # Pre-collect all points; detect break from data
    _pts_sc = []
    for _, row in plot_conf.iterrows():
        y = -np.log10(max(float(row['p_plot']), 1e-10))
        _pts_sc.append((float(row['cliff_d']), y, row['mutation'],
                        mut_color(row['mutation']), bool(row['formal']),
                        float(row['cliff_lo']), float(row['cliff_hi'])))

    _y_sorted_sc = sorted({p[1] for p in _pts_sc})
    _gaps_sc = [(_y_sorted_sc[i+1] - _y_sorted_sc[i],
                 _y_sorted_sc[i], _y_sorted_sc[i+1])
                for i in range(len(_y_sorted_sc)-1)]
    _best_gap_sc = max(_gaps_sc, key=lambda g: g[0]) if _gaps_sc else (0, 0, 0)
    _use_break_sc = _best_gap_sc[0] > 1.5
    def _place_labels_sc(ax, pts, fontsize=8, x_tol=0.12, y_step=0.08):
        by_x = sorted(pts, key=lambda p: p[0])
        top = {}
        for x, y, label, color in by_x:
            xbin = round(x / x_tol) * x_tol
            ly = y + 0.01
            if xbin in top:
                ly = max(ly, top[xbin] + y_step)
            top[xbin] = ly
            ax.text(x, ly, label, ha='center', va='bottom',
                    fontsize=fontsize, color=color, fontweight='normal')

    if _use_break_sc:
        _y_lo_sc  = _best_gap_sc[1] + 0.15
        _y_hi_sc  = _best_gap_sc[2] - 1.2
        _y_max_sc = _y_sorted_sc[-1] + 0.6
        fig, (ax_top_sc, ax_bot_sc) = plt.subplots(
            2, 1, sharex=True, figsize=(6.5, 5.2),
            gridspec_kw={'height_ratios': [1, 3], 'hspace': 0.06})
        ax_top_sc.set_ylim(_y_hi_sc, _y_max_sc)
        ax_bot_sc.set_ylim(-0.15, _y_lo_sc)
        ax_top_sc.spines.bottom.set_visible(False)
        ax_bot_sc.spines.top.set_visible(False)
        ax_top_sc.tick_params(bottom=False, labelbottom=False)
        for _ax in (ax_top_sc, ax_bot_sc):
            _ax.spines[['top', 'right']].set_visible(False)
        d = 0.5
        _bkw = dict(marker=[(-1, -d), (1, d)], markersize=12,
                    linestyle='none', color='k', mew=1, clip_on=False)
        ax_top_sc.plot([0, 1], [0, 0], transform=ax_top_sc.transAxes, **_bkw)
        ax_bot_sc.plot([0, 1], [1, 1], transform=ax_bot_sc.transAxes, **_bkw)
        _all_axes_sc = [ax_top_sc, ax_bot_sc]
    else:
        _y_lo_sc = _y_hi_sc = None
        fig, ax_bot_sc = plt.subplots(figsize=(6.5, 4.5))
        ax_bot_sc.spines[['top', 'right']].set_visible(False)
        ax_top_sc = None
        _all_axes_sc = [ax_bot_sc]

    for _ax in _all_axes_sc:
        _ax.axhline(thresh_y, color='#c0392b', lw=0.9, ls='--', alpha=0.7, zorder=1)
        _ax.axvline(0,    color='k',    lw=0.7, ls='--', alpha=0.35, zorder=1)
        _ax.axvline(-0.3, color='gray', lw=0.7, ls=':',  alpha=0.55, zorder=1)
        _ax.axvline( 0.3, color='gray', lw=0.7, ls=':',  alpha=0.55, zorder=1)
        for x, y, mc, col, formal, lo, hi in _pts_sc:
            _ax.plot([lo, hi], [y, y], '-', color=col, lw=1.3, alpha=0.55, zorder=3)
            _ax.scatter(x, y, color=col, s=70,
                        marker='D' if formal else 'o',
                        alpha=0.90, zorder=4, linewidths=0.8,
                        edgecolors='white' if formal else 'none')

    if _use_break_sc:
        _l_top_sc = [(x, y, mc, col) for x, y, mc, col, *_ in _pts_sc if y >= _y_hi_sc]
        _l_bot_sc = [(x, y, mc, col) for x, y, mc, col, *_ in _pts_sc if y <= _y_lo_sc]
        if _l_top_sc:
            _place_labels_sc(ax_top_sc, _l_top_sc)
    else:
        _l_bot_sc = [(x, y, mc, col) for x, y, mc, col, *_ in _pts_sc]
    _place_labels_sc(ax_bot_sc, _l_bot_sc)

    ax_bot_sc.set_xlabel("Cliff's delta  (mut+ vs WT,  prediction score)", fontsize=9)
    ax_bot_sc.set_ylabel('-log10(p)', fontsize=9)

    from matplotlib.lines import Line2D
    ax_bot_sc.legend(handles=[
        Line2D([0],[0], marker='D', color='#555', ms=7, lw=0,
               markerfacecolor='#555', markeredgecolor='white',
               markeredgewidth=0.8, label='BH-FDR corrected p-values'),
        Line2D([0],[0], marker='o', color=_COLOR_EXPL, ms=6, lw=0,
               markerfacecolor=_COLOR_EXPL, markeredgecolor='none',
               label='Uncorrected p-values'),
        Line2D([0],[0], color='#c0392b', lw=0.9, ls='--', alpha=0.7, label='p=0.05'),
        Line2D([0],[0], color='#666', lw=1.3, alpha=0.6, label='95% bootstrap CI'),
    ], fontsize=8, loc='center left', framealpha=0.7, bbox_to_anchor=(1.01, 0.5))

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'Figure_S8C.pdf', bbox_inches='tight')
    plt.show()

    conf_df.to_csv(OUTPUT_DIR / 'Figure_S8C_confidence_mutation.csv', index=False)
    print('Saved -> Figure_S8C_confidence_mutation.csv  |  Figure_S8C.pdf')



In [ ]:
# ── EXCLUDED — DAPI gradient scalars superseded by Figure S8C ────────────────
if False:
    # Summarises the 8-ring DAPI profile into two scalars per patient:
    #   1. Centre/boundary ratio = DAPI(R7) / DAPI(R0)  (>1 = denser at centre)
    #   2. Linear slope (boundary -> centre)             (positive = denser inward)
    # Compares scalars between mutation+ and WT using MWU + Cliff's delta.
    # Requires pat_dapi from §7c-A.

    if 'pat_dapi' not in dir():
        print('Run §7c-A first to build pat_dapi.')
    else:
        r0_col = _int_cols[0]
        r7_col = _int_cols[-1]
        X_ring = np.arange(len(_int_cols), dtype=float)
        X_c    = X_ring - X_ring.mean()

        pat_dapi['dapi_centre_boundary'] = np.where(
            pat_dapi[r0_col].abs() > 1e-6,
            pat_dapi[r7_col] / pat_dapi[r0_col], np.nan)

        slopes = []
        for _, rp in pat_dapi.iterrows():
            y = rp[_int_cols].values.astype(float)
            ok = ~np.isnan(y)
            slopes.append(float(np.polyfit(X_c[ok], y[ok], 1)[0]) if ok.sum() > 2 else np.nan)
        pat_dapi['dapi_slope'] = slopes

        grad_specs = [
            ('dapi_centre_boundary', 'Centre/boundary ratio (R7/R0)', 1.0),
            ('dapi_slope',           'Linear DAPI slope (boundary→centre)', 0.0),
        ]

        grad_rows = []
        for mc in formal_mut + small_n_mut:
            if mc not in pat_dapi.columns:
                continue
            flag = pat_dapi[mc].fillna(0).astype(bool)
            for feat, feat_lbl, ref in grad_specs:
                xm = pat_dapi.loc[flag,  feat].dropna().values
                xw = pat_dapi.loc[~flag, feat].dropna().values
                if len(xm) < 2 or len(xw) < 2:
                    continue
                d, lo, hi = cliff_delta_ci(xm, xw)
                _, p_mwu  = mannwhitneyu(xm, xw, alternative='two-sided')
                grad_rows.append({
                    'mutation': mc, 'feature': feat, 'feature_label': feat_lbl,
                    'n_mut': len(xm), 'n_wt': len(xw),
                    'med_mut': round(float(np.median(xm)), 3),
                    'med_wt':  round(float(np.median(xw)), 3),
                    'MWU_p':   round(p_mwu, 4),
                    'cliff_d': round(d, 3), 'cliff_lo': round(lo, 3), 'cliff_hi': round(hi, 3),
                    'formal':  mc in formal_mut,
                })

        grad_df = pd.DataFrame(grad_rows)
        grad_df['p_adj_BH'] = np.nan
        fm = grad_df['formal'].values
        if fm.sum() > 0:
            _, adj_p, _, _ = multipletests(grad_df.loc[fm, 'MWU_p'], method='fdr_bh')
            grad_df.loc[fm, 'p_adj_BH'] = np.round(adj_p, 4)
        grad_df['sig'] = grad_df.apply(
            lambda r: ('***' if r['p_adj_BH'] < 0.001 else '**' if r['p_adj_BH'] < 0.01
                       else '*' if r['p_adj_BH'] < 0.05 else 'ns') if r['formal']
            else 'expl.', axis=1)

        print('=== §7c-C: DAPI gradient by mutation ===')
        print(grad_df[['mutation','feature','n_mut','n_wt','med_mut','med_wt',
                        'MWU_p','p_adj_BH','sig','cliff_d']].to_string(index=False))

        rng_jit = np.random.default_rng(42)
        fig, axes = plt.subplots(len(grad_specs), len(formal_mut),
                                  figsize=(3.5 * len(formal_mut), 3.8 * len(grad_specs)),
                                  squeeze=False)

        for fi, (feat, feat_lbl, ref) in enumerate(grad_specs):
            for ci, mc in enumerate(formal_mut):
                ax = axes[fi][ci]
                if mc not in pat_dapi.columns:
                    ax.set_visible(False); continue
                flag  = pat_dapi[mc].fillna(0).astype(bool)
                grps  = [pat_dapi.loc[~flag, feat].dropna().values,
                          pat_dapi.loc[flag,  feat].dropna().values]
                lbls  = [f'WT\n(n={len(grps[0])})', f'{mc}+\n(n={len(grps[1])})']
                cols  = [_COLOR_WT, mut_color(mc)]

                bp = ax.boxplot(grps, patch_artist=True, widths=0.42,
                                 medianprops=dict(color='black', lw=2),
                                 whiskerprops=dict(lw=1), capprops=dict(lw=1))
                for patch, col in zip(bp['boxes'], cols):
                    patch.set_facecolor(col); patch.set_alpha(0.50)
                for gi, (grp, col) in enumerate(zip(grps, cols)):
                    jit = rng_jit.uniform(-0.13, 0.13, len(grp))
                    ax.scatter(gi + 1 + jit, grp, color=col, alpha=0.70, s=22, zorder=3)

                sub = grad_df[(grad_df['mutation'] == mc) & (grad_df['feature'] == feat)]
                if not sub.empty:
                    r = sub.iloc[0]
                    sig_txt = r['sig'] if r['sig'] != 'expl.' else f"p={r['MWU_p']:.2f}"
                    ax.set_title(f'{mc}\nδ={r["cliff_d"]:.2f}  {sig_txt}',
                                  fontsize=8.5, fontweight='bold')

                ax.set_xticklabels(lbls, fontsize=8)
                ax.axhline(ref, color='grey', lw=0.7, ls='--', alpha=0.5)
                if ci == 0:
                    ax.set_ylabel(feat_lbl, fontsize=8.5)

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'Figure_S8C.pdf', bbox_inches='tight')
        plt.show()

        grad_df.to_csv(OUTPUT_DIR / 'Figure_S8C_dapi_gradient_mutation.csv', index=False)
        print('Saved → 07c_dapi_gradient_mutation.csv  |  07c_dapi_gradient_mutation.pdf')


In [ ]:
# ── EXCLUDED — mediation analysis dropped ────────────────────────────────────
if False:
    # Bootstrap mediation (2000 iterations) testing whether the chromatin density
    # gradient (centre/boundary DAPI ratio) mediates mutation -> mean attribution.
    #
    #   a-path   : mutation -> DAPI gradient         (OLS coefficient)
    #   b-path   : DAPI gradient -> attribution | mutation  (partial OLS)
    #   c_total  : total effect mutation -> attribution
    #   c_prime  : direct effect mutation -> attribution controlling for gradient
    #   indirect : a x b  (95% CI by bootstrap; sig if CI excludes zero)
    #
    # Requires dapi_centre_boundary from §7c-C and pat_mut from §7b-A.

    if 'dapi_centre_boundary' not in pat_dapi.columns:
        print('Run §7c-C first to compute dapi_centre_boundary.')
    else:
        med_feat = 'dapi_centre_boundary'
        pat_med = pat_mut.copy()
        pat_med['_attr_mean'] = pat_med[_ring_cols].mean(axis=1)
        pat_med = pat_med.merge(
            pat_dapi[[_pat_col, med_feat]], on=_pat_col, how='left')

        def _ols_coef(Xmat, yvec, col_idx=0):
            # OLS: returns coefficient at col_idx of Xmat (intercept added internally).
            Xd = np.column_stack([np.ones(len(yvec)), Xmat])
            coef, _, _, _ = np.linalg.lstsq(Xd, yvec, rcond=None)
            return float(coef[col_idx + 1])   # +1 offset for intercept column

        med_rows = []
        for mc in formal_mut:
            if mc not in pat_med.columns:
                continue
            sub = pat_med[[_pat_col, mc, med_feat, '_attr_mean']].dropna()
            if len(sub) < 6:
                continue
            X  = sub[mc].astype(float).values        # mutation indicator (predictor)
            M  = sub[med_feat].values                # DAPI gradient (mediator)
            Y  = sub['_attr_mean'].values            # mean attribution (outcome)
            XM = np.column_stack([X, M])

            a     = _ols_coef(X.reshape(-1, 1), M, col_idx=0)   # X -> M
            b     = _ols_coef(XM, Y, col_idx=1)                  # M -> Y | X
            c_tot = _ols_coef(X.reshape(-1, 1), Y, col_idx=0)    # total X -> Y
            c_p   = _ols_coef(XM, Y, col_idx=0)                  # direct X -> Y | M
            indirect_obs = a * b

            n = len(sub)
            boot_ind = []
            for _ in range(N_BOOT):
                idx = _RNG.choice(n, n, replace=True)
                Xb, Mb, Yb = X[idx], M[idx], Y[idx]
                XMb = np.column_stack([Xb, Mb])
                a_b = _ols_coef(Xb.reshape(-1, 1), Mb, col_idx=0)
                b_b = _ols_coef(XMb, Yb, col_idx=1)
                boot_ind.append(a_b * b_b)
            ci_lo, ci_hi = np.percentile(boot_ind, [2.5, 97.5])

            med_rows.append({
                'mutation': mc, 'mediator': med_feat, 'n': n,
                'a_mut_to_DAPI':       round(a, 4),
                'b_DAPI_to_attr_mut':  round(b, 4),
                'c_total':             round(c_tot, 4),
                'c_prime_direct':      round(c_p, 4),
                'indirect_a_x_b':      round(indirect_obs, 4),
                'CI_lo': round(ci_lo, 4), 'CI_hi': round(ci_hi, 4),
                'sig_mediation': 'yes' if (ci_lo > 0 or ci_hi < 0) else 'no',
            })

        if med_rows:
            med_df = pd.DataFrame(med_rows)
            print('=== §7c-D: Bootstrap mediation results (2000 iterations) ===')
            print(med_df.to_string(index=False))
            med_df.to_csv(OUTPUT_DIR / 'Figure_S8D_mediation_dapi_attribution.csv', index=False)
            print('\nSaved → 07c_mediation_dapi_attribution.csv')
            print('\n'
                  'INTERPRETATION:\n'
                  '  sig_mediation=yes : chromatin gradient carries part of the mutation effect\n'
                  '                      on attribution. Genotype may act partly via altering\n'
                  '                      nuclear chromatin density architecture.\n'
                  '  sig_mediation=no  : mutation affects attribution independently of gradient;\n'
                  '                      the attribution difference is not explained by global\n'
                  '                      chromatin density shifts in the DAPI ring profile.\n'
                  '\n'
                  '  c_prime << c_total : gradient explains most of the effect (large mediation)\n'
                  '  c_prime ~ c_total  : direct effect dominates (gradient not the pathway)\n')
        else:
            print('Insufficient data for mediation in all formal mutations.')


---
## 8 · DAPI / intensity spatial correlation with attribution

`attr_dapi_pearson` and `attr_dapi_spearman` measure the **pixel-level spatial agreement**  
between the occlusion attribution map and the DAPI (nucleus) intensity.  
Positive correlation → the model attends to bright (nuclear) pixels.

In [ ]:
dapi_cols = [c for c in ['attr_dapi_pearson', 'attr_dapi_spearman'] if c in df.columns]

if dapi_cols:
    fig, axes = plt.subplots(1, len(dapi_cols) * 2, figsize=(FW, 4 * FW/(5 * len(dapi_cols) * 2)))
    col_ax = iter(axes)

    for cc in dapi_cols:
        ax_hist = next(col_ax)
        ax_box  = next(col_ax)

        for cls in CLASSES:
            vals = df[df['_class'] == cls][cc].dropna()
            ax_hist.hist(vals, bins=35, alpha=0.65, color=COLORS[cls],
                         label=f'{cls}  μ={vals.mean():.3f}', edgecolor='white')
        ax_hist.axvline(0, color='k', lw=1, ls='--')
        ax_hist.set_title(f'{cc}\ndistribution', fontweight='bold')
        ax_hist.set_xlabel(cc)
        ax_hist.legend(fontsize=8)

        data = [df[df['_class'] == cls][cc].dropna() for cls in CLASSES]
        bp = ax_box.boxplot(data, patch_artist=True, labels=CLASSES,
                            medianprops=dict(color='black', lw=2))
        for patch, cls in zip(bp['boxes'], CLASSES):
            patch.set_facecolor(COLORS[cls]); patch.set_alpha(0.6)
        _pat_col_d = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'
        # AML vs Control specifically — CLASSES[0]/[1] would be Control vs
        # Preleukemia given CLASSES = [Control, Preleukemia, AML].
        _aml_pat = df[df['_class']==positive_name].groupby(_pat_col_d)[cc].median()
        _ctrl_pat = df[df['_class']==negative_name].groupby(_pat_col_d)[cc].median()
        star, p = mwu_stars(_aml_pat, _ctrl_pat)
        ax_box.set_title(f'{cc}\n{star}  p={p:.2e}', fontweight='bold')
        ax_box.axhline(0, color='k', lw=0.7, ls='--')

    plt.suptitle('Attribution–intensity spatial correlation', 
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    _rows_08 = []
    for _cc in dapi_cols:
        for _cls in CLASSES:
            _vals = df[df['_class'] == _cls][_cc].dropna()
            _rows_08.append({'feature': _cc, 'class': _cls, 'mean': _vals.mean(),
                              'median': _vals.median(), 'n': len(_vals)})
    pd.DataFrame(_rows_08).to_csv(OUTPUT_DIR / '08_dapi_correlation_data.csv', index=False)

    plt.savefig(OUTPUT_DIR / "08_dapi_correlation.pdf", bbox_inches="tight")
plt.show()

In [ ]:
%matplotlib inline
# ── Fig 6E: whole-cell AML-class attribution masses — Control, Preleukemia, AML, MRD ──
from scipy.stats import false_discovery_control, kruskal
from itertools import combinations

pat_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

plot_df = df[['_class', pat_col, 'attr_total_abs', 'attr_pos_sum', 'attr_neg_sum']].copy()

# All four classes (was AML-vs-Control only). Kept as an explicit list rather
# than CLASSES so this stays correct regardless of what CLASSES holds, and so
# MRD (which CLASSES deliberately excludes) is still included here.
CMP_CLASSES_6E = ['Control', 'Preleukemia', 'AML', 'MRD']
CMP_CLASSES_6E = [c for c in CMP_CLASSES_6E if c in plot_df['_class'].unique()]

PANELS_6E = [
    ('attr_pos_sum',  'AML-supporting\nmass',
     r'$\Sigma\,\mathrm{supporting\;attribution}$'),
    ('attr_neg_sum',  'AML-suppressing\nmass',
     r'$\Sigma\,|\mathrm{suppressing\;attribution}|$'),
]

# Patient-median aggregation — avoids pseudo-replication (n = patients, not cells)
_test_cols = ['attr_pos_sum', 'attr_neg_sum']
pat_med_6e = (plot_df.dropna(subset=_test_cols + [pat_col])
              .groupby([pat_col, '_class'])[_test_cols]
              .median()
              .reset_index())

# ── Statistics ────────────────────────────────────────────────────────────────
# Correct order for >2 independent groups: Kruskal-Wallis omnibus test first
# (does ANY group differ?), then pairwise Mann-Whitney U as post-hoc, BH-FDR
# corrected across all 6 pairs per feature. (Dunn's test would be the more
# textbook-proper post-hoc for Kruskal-Wallis — it shares rank information
# across all groups rather than testing each pair in isolation — but that
# lives in scikit-posthocs, not available here; pairwise MWU+BH-FDR is the
# standard, defensible fallback.)
_kw_results_6e = {}
for col in _test_cols:
    _kw_groups = [pat_med_6e.loc[pat_med_6e['_class'] == cls, col].dropna().values
                  for cls in CMP_CLASSES_6E]
    _kw_stat, _kw_p = kruskal(*_kw_groups)
    _kw_results_6e[col] = {'H': _kw_stat, 'p': _kw_p}

_pairwise_rows_6e = []
for col in _test_cols:
    _pairs = list(combinations(CMP_CLASSES_6E, 2))
    _raw_p = []
    for g1, g2 in _pairs:
        v1 = pat_med_6e.loc[pat_med_6e['_class'] == g1, col]
        v2 = pat_med_6e.loc[pat_med_6e['_class'] == g2, col]
        _, p = mannwhitneyu(v1, v2, alternative='two-sided')
        _raw_p.append(p)
    _adj_p = false_discovery_control(_raw_p) if _raw_p else []
    for (g1, g2), p_raw, p_adj in zip(_pairs, _raw_p, _adj_p):
        _pairwise_rows_6e.append({'feature': col, 'group1': g1, 'group2': g2,
                                   'p_raw': p_raw, 'p_adj_BH': p_adj,
                                   'kruskal_wallis_p': _kw_results_6e[col]['p']})
pd.DataFrame(_pairwise_rows_6e).to_csv(OUTPUT_DIR / 'Fig6E_all_pairwise_stats.csv', index=False)

def _stars6e(p):
    return 'n.s.' if p >= .05 else ('***' if p < .001 else ('**' if p < .01 else '*'))

sns.set_theme(style='whitegrid')
mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 8,
    'axes.titlesize': 8, 'axes.labelsize': 8,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'pdf.fonttype': 42, 'figure.dpi': 120,
    'axes.linewidth': 0.8,
    'grid.linestyle': '--', 'grid.linewidth': 0.5,
})

fig, axes = plt.subplots(1, 2, figsize=(FW * 2 / 3, FW * 0.42), sharey=True)

_x_of = {cls: i + 1 for i, cls in enumerate(CMP_CLASSES_6E)}  # class -> x position

_panel_ranges_6e = []  # (y_lo, y_top) per panel — applied once, after the loop
for ax, (col, title, ylabel) in zip(axes, PANELS_6E):
    # clip(lower=0.1), not 0 — a handful of cells have attr_pos_sum/neg_sum
    # exactly 0 (e.g. Preleukemia), and on a log-scale axis a whisker that
    # tries to render at y=0 has nowhere valid to go. Flooring at 0.1 keeps
    # the box/whisker geometry (and the range measured from it below) fully
    # on-scale, consistent with the same floor already used for the
    # patient-median scatter dots.
    grp = [plot_df.loc[plot_df['_class'] == cls, col].dropna().clip(lower=0.1).values
           for cls in CMP_CLASSES_6E]

    # Log-scale boxplot — handles right-skewed sum data without blade artefact
    bp = ax.boxplot(grp, positions=list(_x_of.values()), widths=0.45, showfliers=False,
                    patch_artist=True,
                    medianprops=dict(color='k', linewidth=2),
                    whiskerprops=dict(linewidth=0.8),
                    capprops=dict(linewidth=0.8),
                    boxprops=dict(linewidth=0.8))
    for patch, cls in zip(bp['boxes'], CMP_CLASSES_6E):
        patch.set_facecolor(COLORS[cls]);# patch.set_alpha(0.8)

    ax.set_yscale('log')

    # Patient medians overlaid — these are the units the tests use
    _pm_vals_6e = []
    for cls in CMP_CLASSES_6E:
        pm = pat_med_6e.loc[pat_med_6e['_class'] == cls, col].clip(lower=0.1)
        ax.scatter(np.full(len(pm), float(_x_of[cls])), pm.values,
                   marker='o', s=18, color='0.15', alpha=0.65, zorder=4)
        _pm_vals_6e.append(pm.values)

    # y-limits from what's ACTUALLY drawn (whisker extents + patient-median
    # dots), not a fixed percentile-of-pooled-raw-data guess — the latter
    # clipped whiskers once this went from 2 groups to 4 with different
    # spreads.
    _whisker_ys_6e = np.concatenate([w.get_ydata() for w in bp['whiskers']])
    _drawn_6e = np.concatenate([_whisker_ys_6e] + _pm_vals_6e)
    _drawn_6e = _drawn_6e[_drawn_6e > 0]
    y_lo = _drawn_6e.min()
    y_hi = _drawn_6e.max()

    # Bracket ladder — one bracket per SIGNIFICANT pairwise comparison
    # (p_adj_BH < 0.05), shortest span first so brackets nest cleanly without
    # crossing. Only drawn when Kruskal-Wallis itself is significant, so an
    # underpowered omnibus test doesn't get papered over by cherry-picked
    # pairwise stars.
    _kw_p = _kw_results_6e[col]['p']
    _sig_pairs = [r for r in _pairwise_rows_6e
                  if r['feature'] == col and r['p_adj_BH'] < 0.05]
    _sig_pairs.sort(key=lambda r: abs(_x_of[r['group1']] - _x_of[r['group2']]))

    _step = 1.55  # multiplicative y-step between stacked brackets (log space)
    _y_level = y_hi * 1.35
    if _kw_p < 0.05:
        for r in _sig_pairs:
            x1, x2 = sorted([_x_of[r['group1']], _x_of[r['group2']]])
            ax.plot([x1, x2], [_y_level, _y_level], color='0.25', lw=1.0)
            ax.text((x1 + x2) / 2, _y_level * 0.73, _stars6e(r['p_adj_BH']),
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
            _y_level *= _step
    elif _sig_pairs:
        # KW n.s. but some pairwise tests nominally significant — report but
        # don't visually assert them; annotate instead of drawing brackets.
        ax.text(0.5, 0.99, 'Kruskal-Wallis n.s. — pairwise stars not shown',
                transform=ax.transAxes, ha='center', va='top',
                fontsize=6, color='0.4', style='italic')

    y_top = _y_level if (_kw_p < 0.05 and _sig_pairs) else y_hi * 1.6
    _panel_ranges_6e.append((y_lo * 0.5, y_top * 1.8))

    ax.set_xticks(list(_x_of.values())); ax.set_xticklabels(CMP_CLASSES_6E)
    ax.set_ylabel('log scale ' + ylabel, fontsize=8, fontweight='bold')
    #ax.set_title(f'{title}\nKruskal-Wallis p={_kw_p:.2e}', fontweight='bold', fontsize=8)
    sns.despine(ax=ax)

# axes share a y-axis (sharey=True), so only ONE set_ylim call actually
# takes effect for both panels — calling it per-panel inside the loop above
# meant the second panel silently overwrote the first panel's range and
# clipped its whiskers. Take the union across both panels' needed ranges
# and apply it once, after both panels are fully drawn.
_shared_y_lo = min(r[0] for r in _panel_ranges_6e)
_shared_y_top = max(r[1] for r in _panel_ranges_6e)
axes[0].set_ylim(_shared_y_lo, _shared_y_top)

plt.tight_layout()
pat_med_6e.to_csv(OUTPUT_DIR / 'Fig6E_attribution_masses_data.csv', index=False)

plt.savefig(OUTPUT_DIR / 'Fig6E_attribution_masses.pdf', bbox_inches='tight')
plt.show()

for col in _test_cols:
    print(f'{col}: Kruskal-Wallis H={_kw_results_6e[col]["H"]:.2f}  '
          f'p={_kw_results_6e[col]["p"]:.3e}  {_stars6e(_kw_results_6e[col]["p"])}')
    for cls in CMP_CLASSES_6E:
        n = (pat_med_6e['_class'] == cls).sum()
        med = pat_med_6e.loc[pat_med_6e['_class'] == cls, col].median()
        print(f'    {cls:12s} n={n:3d}  median={med:.1f}')

print(f'\nAll six pairwise comparisons (BH-FDR within each feature) across '
      f'{CMP_CLASSES_6E} saved to Fig6E_all_pairwise_stats.csv')
print(pd.DataFrame(_pairwise_rows_6e).to_string(index=False))

---
## 9 · BM vs PB source (AML cells)

AML blasts appear in both bone marrow (BM) and peripheral blood (PB), but may look morphologically different.  
Do ring attribution profiles differ by sample source?

In [ ]:
# ── § 9: BM vs PB — patient-level (AML only) ─────────────────────────────────
# Source (BM/PB) is a sample-level attribute. Multiple cells from the same
# patient × source share the same origin, so cell-level MWU inflates n.
# Fix: median attribution per (patient, source), then compare groups.
# Note: patients who contributed both BM and PB appear in both groups —
# this is a paired design and is actually more informative than between-patient.

patient_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'
src_col     = 'source_x'    if 'source_x'    in df.columns else 'source'
feat_box    = [f for f in ['attr_pos_neg_ratio', 'attr_pos_gini',
                             'attr_pos_centroid_offset'] if f in df.columns]

if src_col in df.columns and patient_col in df.columns and avail_rings:
    # Aggregate to patient × source level (AML only)
    aml = pos.copy()
    agg_cols = avail_rings + feat_box
    pat_src = (aml.groupby([patient_col, src_col])[agg_cols]
                  .median()
                  .reset_index())

    bm = pat_src[pat_src[src_col] == 'BM']
    pb = pat_src[pat_src[src_col] == 'PB']

    # Patients in both groups (paired)
    paired_pats = set(bm[patient_col]) & set(pb[patient_col])
    print(f"BM patients: {len(bm)}   PB patients: {len(pb)}   "
          f"paired (both): {len(paired_pats)}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Ring profiles ─────────────────────────────────────────────────────────
    ax = axes[0]
    for grp, lbl, col in [(bm, f'BM (n={len(bm)} patients)', '#1b4f72'),
                           (pb, f'PB (n={len(pb)} patients)', '#5dade2')]:
        if len(grp) < 3:
            continue
        m = grp[avail_rings].mean().values
        s = grp[avail_rings].sem().values
        ax.plot(RING_X[:len(avail_rings)], m, 'o-', label=lbl, lw=2, color=col)
        ax.fill_between(RING_X[:len(avail_rings)], m-s, m+s, alpha=0.2, color=col)

    for i, col_r in enumerate(avail_rings):
        if len(bm) > 2 and len(pb) > 2:
            star, _ = mwu_stars(bm[col_r], pb[col_r])
            if star != 'ns':
                ax.text(i, ax.get_ylim()[1] * 0.97, star, ha='center', fontsize=9)

    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.set_xticks(RING_X[:len(avail_rings)])
    ax.set_xticklabels(RING_LBL[:len(avail_rings)], fontsize=8)
    ax.set_title('Ring attribution — BM vs PB\n(patient-level medians, AML only)',
                 fontweight='bold')
    ax.set_ylabel('Mean attribution (across patients)')
    ax.legend(fontsize=9)

    # ── Attribution feature boxplots ──────────────────────────────────────────
    ax2 = axes[1]
    x = np.arange(len(feat_box)); w = 0.35
    for xi, fc in enumerate(feat_box):
        for offset, grp, fc_col in [(-w/2, bm, '#1b4f72'), (w/2, pb, '#5dade2')]:
            vals = grp[fc].dropna().values
            if len(vals) < 3:
                continue
            ax2.boxplot(vals, positions=[xi + offset], widths=w*0.9,
                        patch_artist=True,
                        boxprops=dict(facecolor=fc_col, alpha=0.75),
                        medianprops=dict(color='white', lw=2),
                        flierprops=dict(marker='o', ms=3, alpha=0.4))
        if len(bm[fc].dropna()) > 2 and len(pb[fc].dropna()) > 2:
            star, _ = mwu_stars(bm[fc], pb[fc])
            ymax = max(bm[fc].quantile(0.9), pb[fc].quantile(0.9))
            ax2.text(xi, ymax * 1.05, star, ha='center', fontsize=10)

    ax2.set_xticks(x)
    ax2.set_xticklabels([f.replace('attr_', '') for f in feat_box], fontsize=8)
    ax2.set_title('Attribution features — BM vs PB\n(patient-level medians)',
                  fontweight='bold')
    from matplotlib.patches import Patch
    ax2.legend(handles=[Patch(color='#1b4f72', alpha=0.75, label='BM'),
                         Patch(color='#5dade2', alpha=0.75, label='PB')],
               fontsize=9)

    plt.suptitle('Sample source: BM vs PB (AML · patient-level medians)',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "09_bm_vs_pb.pdf", bbox_inches="tight")
    plt.show()

    # Statistical summary table
    rows_bm_pb = []
    for fc in avail_rings + feat_box:
        star, p = mwu_stars(bm[fc], pb[fc])
        rows_bm_pb.append({
            'feature':    fc,
            'BM_mean':    round(bm[fc].mean(), 4),
            'BM_median':  round(bm[fc].median(), 4),
            'PB_mean':    round(pb[fc].mean(), 4),
            'PB_median':  round(pb[fc].median(), 4),
            'n_BM':       int(bm[fc].notna().sum()),
            'n_PB':       int(pb[fc].notna().sum()),
            'p_mwu':      round(p, 5),
            'sig':        star,
        })
    bm_pb_df = pd.DataFrame(rows_bm_pb)
    bm_pb_df.to_csv(OUTPUT_DIR / "09_bm_vs_pb_stats.csv", index=False)
    print(bm_pb_df[bm_pb_df['sig'] != 'ns'].to_string(index=False))
else:
    print("source or patientID column not found — skipping.")


---
## 10 · High-confidence prediction analysis

Cells predicted with **very high model confidence** (top quartile of prediction score)  
may share distinctive morphological and attribution patterns that reveal what the model has learned.

We split cells into **high-confidence** (score ≥ 0.9) and **low-confidence** (score < 0.9) within each class.

In [ ]:
score_col = 'prediction_score'

if score_col not in df.columns:
    # Reconstruct from attr_mean sign — not ideal but usable as proxy
    print(f"'{score_col}' not in CSV. Using attr_total_abs as proxy for confidence.")
    HIGH_THRESH = df['attr_total_abs'].quantile(0.75)
    df['_high_conf'] = df['attr_total_abs'] >= HIGH_THRESH
    score_label = 'attr_total_abs (proxy)'
else:
    HIGH_THRESH = 0.9
    df['_high_conf'] = df[score_col] >= HIGH_THRESH
    score_label = f'{score_col} ≥ {HIGH_THRESH}'

for cls in CLASSES:
    sub = df[df['_class'] == cls]
    n_hi = sub['_high_conf'].sum()
    n_lo = (~sub['_high_conf']).sum()
    print(f"{cls:12s} — high-conf: {n_hi:4d}  low-conf: {n_lo:4d}")

print(f"\nThreshold: {score_label}")

In [ ]:
# Ring attribution: high vs low confidence, separately per class
fig, axes = plt.subplots(1, len(CLASSES), figsize=(FW * len(CLASSES) / 2, FW*5/14))

palette_by_class = {
    'AML':         {'high': '#e31a1c', 'low': '#fb9a99'},
    'Control':     {'high': '#1f78b4', 'low': '#a6cee3'},
    'Preleukemia': {'high': '#33a02c', 'low': '#b2df8a'},
}
_fallback_pal = {'high': '#6a3d9a', 'low': '#cab2d6'}

for ax, cls in zip(axes, CLASSES):
    pal = palette_by_class.get(cls, _fallback_pal)
    sub = df[df['_class'] == cls]
    hi  = sub[sub['_high_conf']]
    lo  = sub[~sub['_high_conf']]

    for grp, lbl, col in [(hi, f'High-conf (n={len(hi)})', pal['high']),
                           (lo, f'Low-conf  (n={len(lo)})', pal['low'])]:
        if len(grp) < 3:
            continue
        m = grp[avail_rings].mean().values
        s = grp[avail_rings].sem().values
        ax.plot(RING_X[:len(avail_rings)], m, 'o-', label=lbl, lw=2.5, color=col)
        ax.fill_between(RING_X[:len(avail_rings)], m-s, m+s, alpha=0.18, color=col)

    for i, col in enumerate(avail_rings):
        star, _ = mwu_stars(hi[col], lo[col])
        if star != 'ns':
            ax.text(i, ax.get_ylim()[1] * 0.97, star, ha='center', fontsize=9)

    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.set_xticks(RING_X[:len(avail_rings)])
    ax.set_xticklabels(RING_LBL[:len(avail_rings)], fontsize=8)
    ax.set_title(f'{cls.upper()} — high vs low confidence', fontweight='bold')
    ax.set_ylabel('Mean attribution')
    ax.legend(fontsize=8)

plt.suptitle('Ring attribution: high-confidence vs low-confidence predictions', 
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
_rows_10a = []
for _cls in CLASSES:
    _sub = df[df['_class'] == _cls]
    _hi, _lo = _sub[_sub['_high_conf']], _sub[~_sub['_high_conf']]
    for _ri, _col in enumerate(avail_rings):
        _rows_10a.append({'class': _cls, 'ring': _ri,
                           'high_conf_mean': _hi[_col].mean(), 'high_conf_sem': _hi[_col].sem(),
                           'low_conf_mean': _lo[_col].mean(), 'low_conf_sem': _lo[_col].sem(),
                           'n_high': len(_hi), 'n_low': len(_lo)})
pd.DataFrame(_rows_10a).to_csv(OUTPUT_DIR / '10a_high_conf_rings_data.csv', index=False)

# Per-cell ring attribution underlying this figure (raw, not aggregated) —
# lets any downstream/external analysis rerun its own test on the real data.
_pat_col_10a = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'
df[['image_name', _pat_col_10a, '_class', '_high_conf'] + avail_rings] \
    .dropna(subset=avail_rings) \
    .to_csv(OUTPUT_DIR / '10a_high_conf_rings_percell.csv', index=False)

# Ring-profile shape tests across confidence-split subgroups — same
# permutation-based functional test as §7b (perm_shape_test, cell 7b-A),
# patient-level medians to avoid pseudo-replication (matches that convention).
#
# Primary question: Preleukemia (low-conf) vs AML (low-conf) — do the "hard"
# cells of each class share a ring-profile shape?
# Positive controls, run alongside so a null result above is interpretable
# (a null could mean genuine overlap OR just low test power at these n's):
#   - Preleukemia (low-conf) vs Control (low-conf) — should differ if the
#     test has power to separate Preleukemia from anything at this n.
#   - Preleukemia (high-conf) vs AML (high-conf) — should differ sharply,
#     since the class-mean curves for confident predictions are far apart.
# Plus the requested cross-confidence comparison:
#   - Preleukemia (high-conf) vs AML (low-conf)

def _conf_subset_patients(cls, high_conf):
    """Patient-level median ring profile for one (class, confidence) subgroup."""
    sub = df[(df['_class'] == cls) & (df['_high_conf'] == high_conf)]
    return sub.groupby(_pat_col_10a)[avail_rings].median().dropna()

def _run_conf_shape_test(label_a, pat_a, label_b, pat_b):
    """Shape test for label_b vs label_a (reference). Returns a results dict or None."""
    if len(pat_a) < 2 or len(pat_b) < 2:
        print(f'Not enough patients for {label_b} vs {label_a} shape test '
              f'(n_{label_a}={len(pat_a)}, n_{label_b}={len(pat_b)}).')
        return None
    # perm_shape_test(p_wt, p_mut) returns p_mut - p_wt, so `a` is the
    # reference (p_wt) and `b` the comparison group (p_mut).
    obs, p_pt, p_sim, _null = perm_shape_test(pat_a.values, pat_b.values)
    best_ring = int(np.argmin(p_sim))
    print(f'{label_b} vs {label_a} ring-profile shape test '
          f'(n_{label_a}={len(pat_a)} pat, n_{label_b}={len(pat_b)} pat):')
    print(f'  global p (max-T, simultaneous) = {p_sim[best_ring]:.4f}  '
          f'(strongest ring: R{best_ring})')
    print(f'  per-ring pointwise p: {np.round(p_pt, 4).tolist()}')
    return {'comparison': f'{label_b}_vs_{label_a}', 'reference': label_a, 'target': label_b,
            'n_reference_pat': len(pat_a), 'n_target_pat': len(pat_b),
            'global_p_maxT': p_sim[best_ring], 'best_ring': best_ring,
            'obs_diff': obs, 'p_pointwise': p_pt, 'p_maxT_simultaneous': p_sim}

_pat_prelk_lo = _conf_subset_patients('Preleukemia', False)
_pat_aml_lo   = _conf_subset_patients(positive_name, False)
_pat_aml_hi   = _conf_subset_patients(positive_name, True)
_pat_prelk_hi = _conf_subset_patients('Preleukemia', True)
_pat_ctrl_lo  = _conf_subset_patients(negative_name, False)

_conf_shape_results = [
    _run_conf_shape_test('AML_low',     _pat_aml_lo,  'Prelk_low',  _pat_prelk_lo),   # primary question
    _run_conf_shape_test('Control_low', _pat_ctrl_lo, 'Prelk_low',  _pat_prelk_lo),   # positive control 1
    _run_conf_shape_test('AML_high',    _pat_aml_hi,  'Prelk_high', _pat_prelk_hi),   # positive control 2
    _run_conf_shape_test('AML_low',     _pat_aml_lo,  'Prelk_high', _pat_prelk_hi),   # requested cross-confidence
]
_conf_shape_results = [r for r in _conf_shape_results if r is not None]

if _conf_shape_results:
    pd.DataFrame([
        {'comparison': r['comparison'], 'reference': r['reference'], 'target': r['target'],
         'n_reference_pat': r['n_reference_pat'], 'n_target_pat': r['n_target_pat'],
         'global_p_maxT': r['global_p_maxT'], 'best_ring': r['best_ring']}
        for r in _conf_shape_results
    ]).to_csv(OUTPUT_DIR / '10a_confidence_shape_tests_summary.csv', index=False)

    pd.concat([
        pd.DataFrame({
            'comparison': r['comparison'], 'ring': range(len(avail_rings)),
            'obs_diff': r['obs_diff'], 'p_pointwise': r['p_pointwise'],
            'p_maxT_simultaneous': r['p_maxT_simultaneous'],
        })
        for r in _conf_shape_results
    ], ignore_index=True).to_csv(OUTPUT_DIR / '10a_confidence_shape_tests_per_ring.csv', index=False)

plt.savefig(OUTPUT_DIR / "10a_high_conf_rings.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Morphological fingerprint of the three LOW-confidence groups ─────────────
# Mechanistic follow-up to the ring-profile shape test above: if Control_low,
# Prelk_low, and AML_low share a "hard-to-segment / atypical morphology"
# phenotype (irregular shape, rough boundary, heterogeneous intensity), that
# would explain shape-test convergence without invoking chromatin biology —
# i.e. all three are hard for the SAME structural reason, not because
# Preleukemia chromatin resembles AML chromatin.
# Same features as §10b (avail_morph_local), same patient-median MWU
# convention as the rest of the notebook (avoids pseudo-replication).

_LOWCONF_GROUPS = {
    'Control_low': df[(df['_class'] == negative_name) & (~df['_high_conf'])],
    'Prelk_low':   df[(df['_class'] == 'Preleukemia') & (~df['_high_conf'])],
    'AML_low':     df[(df['_class'] == positive_name) & (~df['_high_conf'])],
}
_LOWCONF_COLORS = {'Control_low': '#a6cee3', 'Prelk_low': '#b2df8a', 'AML_low': '#fb9a99'}
_pat_col_lc = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'

avail_morph_local = [c for c in MORPH_COLS if c in df.columns]
n_m = len(avail_morph_local)
_n_ok = all(len(g) > 3 for g in _LOWCONF_GROUPS.values())

if n_m and _n_ok:
    n_cols = 4
    n_rows = int(np.ceil(n_m / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(FW, n_rows * FW/n_cols * 3.2/3.4))
    axes_flat = axes.flatten()

    _rows_lowconf_morph = []
    for i, mc in enumerate(avail_morph_local):
        ax = axes_flat[i]
        data = [_LOWCONF_GROUPS[g][mc].dropna() for g in _LOWCONF_GROUPS]
        bp = ax.boxplot(data, patch_artist=True, labels=list(_LOWCONF_GROUPS.keys()),
                        medianprops=dict(color='black', lw=2))
        for box, g in zip(bp['boxes'], _LOWCONF_GROUPS):
            box.set_facecolor(_LOWCONF_COLORS[g]); box.set_alpha(0.7)

        # Patient-median MWU, all three pairwise comparisons
        _pat_med = {g: grp.groupby(_pat_col_lc)[mc].median().dropna()
                    for g, grp in _LOWCONF_GROUPS.items()}
        star_pa, p_pa = mwu_stars(_pat_med['Prelk_low'], _pat_med['AML_low'])
        star_pc, p_pc = mwu_stars(_pat_med['Prelk_low'], _pat_med['Control_low'])
        star_ac, p_ac = mwu_stars(_pat_med['AML_low'],   _pat_med['Control_low'])

        label = mc.replace('_x', '').replace('_', ' ')
        ax.set_title(f'{label}\nPrelk-AML:{star_pa}  Prelk-Ctrl:{star_pc}  AML-Ctrl:{star_ac}',
                     fontsize=6)
        ax.tick_params(labelsize=7)

        # Which reference is Prelk_low's mean closer to, in z-score units
        # (z-scored across all three low-conf groups pooled for this feature)
        _pooled = pd.concat([_LOWCONF_GROUPS[g][mc] for g in _LOWCONF_GROUPS]).dropna()
        _mu, _sig = _pooled.mean(), _pooled.std()
        _z = {g: (_LOWCONF_GROUPS[g][mc].mean() - _mu) / _sig if _sig > 0 else np.nan
              for g in _LOWCONF_GROUPS}
        _dist_to_aml  = abs(_z['Prelk_low'] - _z['AML_low'])
        _dist_to_ctrl = abs(_z['Prelk_low'] - _z['Control_low'])
        _closer_to = 'AML' if _dist_to_aml < _dist_to_ctrl else 'Control'

        _rows_lowconf_morph.append({
            'feature': mc.replace('_x', ''),
            'Control_low_mean': _LOWCONF_GROUPS['Control_low'][mc].mean(),
            'Prelk_low_mean':   _LOWCONF_GROUPS['Prelk_low'][mc].mean(),
            'AML_low_mean':     _LOWCONF_GROUPS['AML_low'][mc].mean(),
            'z_dist_Prelk_to_AML':     round(_dist_to_aml, 3),
            'z_dist_Prelk_to_Control': round(_dist_to_ctrl, 3),
            'Prelk_closer_to':  _closer_to,
            'p_Prelk_vs_AML':     p_pa, 'sig_Prelk_vs_AML':     star_pa,
            'p_Prelk_vs_Control': p_pc, 'sig_Prelk_vs_Control': star_pc,
            'p_AML_vs_Control':   p_ac, 'sig_AML_vs_Control':   star_ac,
        })

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle('Morphological fingerprint of low-confidence cells — Control vs Preleukemia vs AML\n'
                 '(stars: Prelk-AML / Prelk-Ctrl / AML-Ctrl, patient-median MWU)',
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()

    _df_lowconf_morph = pd.DataFrame(_rows_lowconf_morph)
    _df_lowconf_morph.to_csv(OUTPUT_DIR / '10d_lowconf_morph_fingerprint.csv', index=False)

    _n_closer_aml = (_df_lowconf_morph['Prelk_closer_to'] == 'AML').sum()
    _n_closer_ctrl = (_df_lowconf_morph['Prelk_closer_to'] == 'Control').sum()
    print(f'Of {n_m} morphology features, Preleukemia (low-conf) mean is closer to '
          f'AML (low-conf) in {_n_closer_aml} and closer to Control (low-conf) in {_n_closer_ctrl}.')
    print(_df_lowconf_morph[['feature', 'z_dist_Prelk_to_AML', 'z_dist_Prelk_to_Control',
                              'Prelk_closer_to', 'sig_Prelk_vs_AML', 'sig_Prelk_vs_Control']]
          .to_string(index=False))

    plt.savefig(OUTPUT_DIR / "10d_lowconf_morph_fingerprint.pdf", bbox_inches="tight")
else:
    print(f'Not enough cells in one of the low-confidence groups to compare morphology '
          f'(n = {[len(g) for g in _LOWCONF_GROUPS.values()]}).')
plt.show()

In [ ]:
# High-confidence AML: morphological fingerprint
cls = 'AML'
sub = df[df['_class'] == cls].copy()
hi  = sub[sub['_high_conf']]
lo  = sub[~sub['_high_conf']]

avail_morph_local = [c for c in MORPH_COLS if c in df.columns]
n_m = len(avail_morph_local)

if n_m and len(hi) > 3 and len(lo) > 3:
    n_cols = 4
    n_rows = int(np.ceil(n_m / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(FW, n_rows * FW/n_cols * 3.2/3.4))
    axes_flat = axes.flatten()

    for i, mc in enumerate(avail_morph_local):
        ax = axes_flat[i]
        data = [hi[mc].dropna(), lo[mc].dropna()]
        bp = ax.boxplot(data, patch_artist=True, labels=['High', 'Low'],
                        medianprops=dict(color='black', lw=2))
        bp['boxes'][0].set_facecolor('#e31a1c'); bp['boxes'][0].set_alpha(0.6)
        bp['boxes'][1].set_facecolor('#fb9a99'); bp['boxes'][1].set_alpha(0.6)
        star, p = mwu_stars(data[0], data[1])
        label = mc.replace('_x', '').replace('_', ' ')
        ax.set_title(f'{label}\n{star}  p={p:.1e}', fontsize=7.5)
        ax.tick_params(labelsize=8)

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle('Morphological features — high vs low confidence AML predictions', 
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    _rows_10b = []
    for _mc in avail_morph_local:
        _star, _p = mwu_stars(hi[_mc], lo[_mc])
        _rows_10b.append({'feature': _mc.replace('_x',''),
                           'high_conf_mean': hi[_mc].mean(), 'low_conf_mean': lo[_mc].mean(),
                           'n_high': hi[_mc].notna().sum(), 'n_low': lo[_mc].notna().sum(),
                           'mwu_p': _p, 'sig': _star})
    pd.DataFrame(_rows_10b).to_csv(OUTPUT_DIR / '10b_morph_high_conf_aml_data.csv', index=False)

    plt.savefig(OUTPUT_DIR / "10b_morph_high_conf_aml.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Attribution features: high vs low confidence (all classes)
avail_sum_local = [c for c in ATTR_SUMMARY if c in df.columns]

_PAL_HILO = {
    positive_name: ('#e31a1c', '#fb9a99'),
    negative_name: ('#1f78b4', '#a6cee3'),
    'Preleukemia':  ('#33a02c', '#b2df8a'),
}
_fallback_pal_hilo = ('#6a3d9a', '#cab2d6')

fig, axes = plt.subplots(len(CLASSES), len(avail_sum_local),
                          figsize=(FW*2, 8 * len(CLASSES) * FW/(len(avail_sum_local) * 2.8)))
for row, cls in enumerate(CLASSES):
    pal = _PAL_HILO.get(cls, _fallback_pal_hilo)
    sub = df[df['_class'] == cls]
    hi  = sub[sub['_high_conf']]
    lo  = sub[~sub['_high_conf']]

    for j, ac in enumerate(avail_sum_local):
        ax = axes[row, j]
        data = [hi[ac].dropna(), lo[ac].dropna()]
        if all(len(d) > 2 for d in data):
            bp = ax.boxplot(data, patch_artist=True, labels=['Hi', 'Lo'],
                            medianprops=dict(color='black', lw=1.5),
                            widths=0.5)
            bp['boxes'][0].set_facecolor(pal[0]); bp['boxes'][0].set_alpha(0.65)
            bp['boxes'][1].set_facecolor(pal[1]); bp['boxes'][1].set_alpha(0.65)
            star, p = mwu_stars(data[0], data[1])
        else:
            star, p = 'nd', 1.0
        lbl = ac.replace('attr_', '').replace('_', '\n')
        ax.set_title(f'{lbl}\n{star}', fontsize=6.5)
        ax.tick_params(labelsize=6.5)
        if j == 0:
            ax.set_ylabel(f'{cls}\n(n hi={len(hi)}, lo={len(lo)})', fontsize=7)

plt.suptitle('Attribution features — high (Hi) vs low (Lo) confidence, per class', 
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
_rows_10c = []
for _cls in CLASSES:
    _sub = df[df['_class'] == _cls]
    _hi, _lo = _sub[_sub['_high_conf']], _sub[~_sub['_high_conf']]
    for _ac in avail_sum_local:
        _rows_10c.append({'class': _cls, 'feature': _ac,
                           'high_conf_mean': _hi[_ac].mean(), 'low_conf_mean': _lo[_ac].mean(),
                           'n_high': _hi[_ac].notna().sum(), 'n_low': _lo[_ac].notna().sum()})
pd.DataFrame(_rows_10c).to_csv(OUTPUT_DIR / '10c_attr_high_conf_data.csv', index=False)

# Per-cell attribution features underlying this figure (raw, not aggregated).
_pat_col_10c = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'
df[['image_name', _pat_col_10c, '_class', '_high_conf'] + avail_sum_local] \
    .dropna(subset=avail_sum_local) \
    .to_csv(OUTPUT_DIR / '10c_attr_high_conf_percell.csv', index=False)

# Preleukemia (low-conf) vs AML (low-conf) per feature — patient-median MWU,
# same convention as the rest of the notebook (avoids pseudo-replication).
_rows_10c_plo = []
_low_aml_10c   = df[(df['_class'] == positive_name) & (~df['_high_conf'])]
_low_prelk_10c = df[(df['_class'] == 'Preleukemia') & (~df['_high_conf'])]
for _ac in avail_sum_local:
    _a_pat = _low_aml_10c.groupby(_pat_col_10c)[_ac].median().dropna()
    _p_pat = _low_prelk_10c.groupby(_pat_col_10c)[_ac].median().dropna()
    if len(_a_pat) >= 2 and len(_p_pat) >= 2:
        _star_plo, _p_plo = mwu_stars(_p_pat, _a_pat)
    else:
        _star_plo, _p_plo = 'nd', np.nan
    _rows_10c_plo.append({'feature': _ac, 'n_Preleukemia_pat': len(_p_pat), 'n_AML_pat': len(_a_pat),
                           'Preleukemia_pat_median': _p_pat.median(), 'AML_pat_median': _a_pat.median(),
                           'mwu_p': _p_plo, 'sig': _star_plo})
pd.DataFrame(_rows_10c_plo).to_csv(OUTPUT_DIR / '10c_prelk_vs_aml_lowconf_mwu.csv', index=False)
print('Preleukemia (low-conf) vs AML (low-conf) — patient-median MWU per feature:')
print(pd.DataFrame(_rows_10c_plo).to_string(index=False))

plt.savefig(OUTPUT_DIR / "10c_attr_high_conf.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Ranked feature table for high-confidence AML
rows = []
sub = df[df['_class'] == positive_name].copy()
hi  = sub[sub['_high_conf']]
lo  = sub[~sub['_high_conf']]

all_feat = avail_morph_local + avail_sum_local + avail_rings
for fc in all_feat:
    a, b = hi[fc].dropna(), lo[fc].dropna()
    if len(a) < 3 or len(b) < 3:
        continue
    star, p = mwu_stars(a, b)
    rows.append({
        'feature': fc.replace('_x', ''),
        'high_conf_mean': a.mean(),
        'low_conf_mean': b.mean(),
        'delta': a.mean() - b.mean(),
        'p_value': p,
        'sig': star,
    })

feat_rank = (pd.DataFrame(rows)
               .assign(abs_delta=lambda x: x['delta'].abs())
               .sort_values('p_value')
               .drop(columns='abs_delta')
               .reset_index(drop=True))

print(f"Top features distinguishing HIGH vs LOW confidence AML predictions (n_hi={len(hi)}, n_lo={len(lo)}):")
feat_rank.head(20).style \
    .format({'high_conf_mean': '{:.4f}', 'low_conf_mean': '{:.4f}',
             'delta': '{:.4f}', 'p_value': '{:.2e}'}) \
    .background_gradient(subset=['delta'], cmap='RdBu_r', vmin=-1, vmax=1)

---
## 11 · Visual gallery — individual high-confidence cells

Spider (radar) chart showing the **attribution profile** for each ring,  
comparing the top-N high-confidence AML cells a`gainst the class mean.

In [ ]:
def radar_chart(ax, values_list, labels, colors, ring_names, title=''):
    """Draw overlaid radar charts."""
    N = len(ring_names)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(ring_names, size=7)
    ax.set_title(title, fontsize=8, fontweight='bold', pad=20)

    for vals, col, lbl in zip(values_list, colors, labels):
        v = list(vals) + [vals[0]]
        ax.plot(angles, v, 'o-', color=col, lw=1.5, label=lbl)
        ax.fill(angles, v, color=col, alpha=0.08)

    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=6)


# Top-10 high-confidence AML cells by (proxy) confidence
# change color of mean lines to be more distinct from cell line
sub = df[df['_class'] == positive_name].copy()

if score_col in df.columns:
    top_cells = sub.nlargest(10, score_col)
else:
    top_cells = sub.nlargest(10, 'attr_total_abs')

class_mean = sub[avail_rings].mean().values
all_mean   = df[avail_rings].mean().values

ring_names_radar = [f'R{i}' for i in range(len(avail_rings))]

n_cells = len(top_cells)
n_cols  = 5
n_rows  = int(np.ceil(n_cells / n_cols))

fig = plt.figure(figsize=(n_cols * 3.5, n_rows * 3.5))

for idx, (_, row_cell) in enumerate(top_cells.iterrows()):
    ax = fig.add_subplot(n_rows, n_cols, idx + 1, polar=True)
    cell_vals = row_cell[avail_rings].values.astype(float)
    img_name  = row_cell.get('image_file', row_cell.get('File', f'cell_{idx}'))
    score_val = row_cell.get(score_col, row_cell.get('attr_total_abs', np.nan))

    radar_chart(
        ax,
        values_list=[cell_vals, class_mean, all_mean],
        colors=['#e31a1c', "#d38f28", '#aaaaaa'],
        labels=[f'cell (score={score_val:.3f})', 'AML mean', 'all mean'],
        ring_names=ring_names_radar,
        title=str(img_name)[-25:],
    )

plt.suptitle('Radar charts — top-10 high-confidence AML cells vs class mean',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
top_cells[['image_name'] + avail_rings].to_csv(OUTPUT_DIR / '11a_radar_aml_data.csv', index=False)

plt.savefig(OUTPUT_DIR / "11a_radar_aml.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Same for high-confidence Control cells
sub = df[df['_class'] == negative_name].copy()

if score_col in df.columns:
    top_ctrl = sub.nlargest(10, score_col)
else:
    top_ctrl = sub.nlargest(10, 'attr_total_abs')

class_mean_neg = sub[avail_rings].mean().values

fig = plt.figure(figsize=(FW, n_rows * FW/n_cols))

for idx, (_, row_cell) in enumerate(top_ctrl.iterrows()):
    ax = fig.add_subplot(n_rows, n_cols, idx + 1, polar=True)
    cell_vals = row_cell[avail_rings].values.astype(float)
    img_name  = row_cell.get('image_file', row_cell.get('File', f'cell_{idx}'))
    score_val = row_cell.get(score_col, row_cell.get('attr_total_abs', np.nan))

    radar_chart(
        ax,
        values_list=[cell_vals, class_mean_neg, all_mean],
        colors=['#1f78b4', "#5d943e", '#aaaaaa'],
        labels=[f'cell (score={score_val:.3f})', 'Ctrl mean', 'all mean'],
        ring_names=ring_names_radar,
        title=str(img_name)[-25:],
    )

plt.suptitle('Radar charts — top-10 high-confidence Control cells vs class mean',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
top_ctrl[['image_name'] + avail_rings].to_csv(OUTPUT_DIR / '11b_radar_control_data.csv', index=False)

plt.savefig(OUTPUT_DIR / "11b_radar_control.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Same for high-confidence Preleukemia cells
sub = df[df['_class'] == 'Preleukemia'].copy()

if score_col in df.columns:
    top_prelk = sub.nlargest(10, score_col)
else:
    top_prelk = sub.nlargest(10, 'attr_total_abs')

class_mean_prelk = sub[avail_rings].mean().values

fig = plt.figure(figsize=(FW, n_rows * FW/n_cols))

for idx, (_, row_cell) in enumerate(top_prelk.iterrows()):
    ax = fig.add_subplot(n_rows, n_cols, idx + 1, polar=True)
    cell_vals = row_cell[avail_rings].values.astype(float)
    img_name  = row_cell.get('image_file', row_cell.get('File', f'cell_{idx}'))
    score_val = row_cell.get(score_col, row_cell.get('attr_total_abs', np.nan))

    radar_chart(
        ax,
        values_list=[cell_vals, class_mean_prelk, all_mean],
        colors=['#33a02c', '#ff7f00', '#aaaaaa'],
        labels=[f'cell (score={score_val:.3f})', 'Prelk mean', 'all mean'],
        ring_names=ring_names_radar,
        title=str(img_name)[-25:],
    )

plt.suptitle('Radar charts — top-10 high-confidence Preleukemia cells vs class mean',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
top_prelk[['image_name'] + avail_rings].to_csv(OUTPUT_DIR / '11c_radar_preleukemia_data.csv', index=False)

plt.savefig(OUTPUT_DIR / "11c_radar_preleukemia.pdf", bbox_inches="tight")
plt.show()

In [ ]:
FEATURE_GROUPS = {
    'Size': [
        'area_x', 'convex_hull_area_x', 'radius_equivalent_x',
        'radius_circumscribed_x', 'perimeter_x',
        'major_axis_length_x', 'minor_axis_length_x',
    ],
    'Roundness / Circularity': [
        'circularity_x', 'solidity_x', 'convex_fill_ratio_x',
        'extent_x', 'radius_inscribed_ratio_x',
    ],
    'Elongation': [
        'eccentricity_x', 'aspect_ratio_x', 'radius_gyration_normalised_x',
    ],
    'Boundary Roughness': [
        'boundary_roughness_x', 'boundary_smoothness_x', 'radius_boundary_std_x',
        'texture_contrast_x', 'texture_dissimilarity_x', 'texture_homogeneity_x',
    ],
    'Mass Distribution': [
        'radial_inner_outer_ratio_x', 'radial_monotonicity_x',
        'radial_cv_x', 'radial_intensity_std_x',
        'intensity_com_offset_x', 'radial_peak_ring_x',
    ],
}

# Filter to available columns
FEATURE_GROUPS = {
    grp: [c for c in cols if c in df.columns]
    for grp, cols in FEATURE_GROUPS.items()
}

GROUP_COLORS = {
    'Size':                   '#2196F3',
    'Roundness / Circularity':'#4CAF50',
    'Elongation':             '#FF9800',
    'Boundary Roughness':     '#9C27B0',
    'Mass Distribution':      '#F44336',
}

for grp, cols in FEATURE_GROUPS.items():
    print(f"{grp:28s}: {cols}")


### 12.1 · AML vs Control — feature distributions by group

Each panel shows all features in a group.  
Significance: *** p<0.001 · ** p<0.01 · * p<0.05 · ns.

In [ ]:
df['_class'].unique()

In [ ]:
from scipy.stats import zscore

pat_col = 'patientID_x' if 'patientID_x' in df.columns else 'patient_ID'
fig = plt.figure(figsize=(FW, FW*22/18))
outer = gridspec.GridSpec(len(FEATURE_GROUPS), 1, figure=fig, hspace=0.55)

for gi, (grp_name, feat_cols) in enumerate(FEATURE_GROUPS.items()):
    if not feat_cols:
        continue
    n_f = len(feat_cols)
    inner = gridspec.GridSpecFromSubplotSpec(1, n_f, subplot_spec=outer[gi],
                                              wspace=0.45)
    grp_col = GROUP_COLORS[grp_name]

    for fi, fc in enumerate(feat_cols):
        ax = fig.add_subplot(inner[fi])

        # Violin + strip — Control, Preleukemia, AML
        _violin_classes = [negative_name, 'Preleukemia', positive_name]
        _violin_classes = [c for c in _violin_classes if c in CLASSES]
        _violin_data = [df[df['_class'] == c][fc].dropna() for c in _violin_classes]

        vp = ax.violinplot(_violin_data, positions=range(len(_violin_classes)),
                           showmedians=True, showextrema=False)
        for body, cls in zip(vp['bodies'], _violin_classes):
            body.set_facecolor(COLORS[cls])
            body.set_alpha(0.45)
        vp['cmedians'].set_color('black')
        vp['cmedians'].set_linewidth(2)

        # Raw points
        _markers = {negative_name: 'o', 'Preleukemia': 's', positive_name: '^'}
        for _pos_i, (cls, data_cls) in enumerate(zip(_violin_classes, _violin_data)):
            jitter = np.random.default_rng(42 + _pos_i).uniform(-0.07, 0.07, len(data_cls))
            ax.scatter(_pos_i + jitter, data_cls, s=4, alpha=0.35, color=COLORS[cls],
                       rasterized=True, marker=_markers.get(cls, 'o'))

        # AML vs Control significance (primary comparison; Preleukemia shown for context)
        aml_pat  = df[df['_class']==positive_name].groupby(pat_col)[fc].median()
        ctrl_pat = df[df['_class']==negative_name].groupby(pat_col)[fc].median()
        star, p = mwu_stars(aml_pat, ctrl_pat)
        label = fc.replace('_x', '').replace('_', '\n')
        ax.set_title(f'{label}\n{star}', fontsize=7.5, color=grp_col, fontweight='bold')
        ax.set_xticks(range(len(_violin_classes)))
        ax.set_xticklabels(['Ctrl' if c == negative_name else 'Prelk' if c == 'Preleukemia' else 'AML'
                             for c in _violin_classes], fontsize=8)
        ax.tick_params(labelsize=7)

    # Group label on the left
    fig.text(0.01, outer[gi].get_position(fig).y0 + outer[gi].get_position(fig).height / 2,
             grp_name, va='center', ha='left', fontsize=11, fontweight='bold',
             color=grp_col, rotation=90)

fig.suptitle('Cell morphological features — Control vs Preleukemia vs AML  (violin + individual cells)',
             fontsize=13, fontweight='bold', y=1.005)
_rows_12a = []
for _grp_name, _feat_cols in FEATURE_GROUPS.items():
    for _fc in _feat_cols:
        _row_12a = {'group': _grp_name, 'feature': _fc.replace('_x','')}
        for _cls in CLASSES:
            _vals = df[df['_class']==_cls][_fc].dropna()
            _row_12a[f'{_cls}_mean'] = _vals.mean()
            _row_12a[f'n_{_cls}'] = len(_vals)
        _rows_12a.append(_row_12a)
pd.DataFrame(_rows_12a).to_csv(OUTPUT_DIR / '12a_grouped_aml_ctrl_data.csv', index=False)

plt.savefig(OUTPUT_DIR / "12a_grouped_aml_ctrl.pdf", bbox_inches="tight")
plt.show()


### 12.2 · Feature groups × attribution correlation

Spearman r between each morphological feature and key attribution measures  
(`attr_pos_neg_ratio`, `attr_pos_gini`, `attr_ring07_mean` = centre ring, `attr_ring00_mean` = boundary ring).

Run separately per class to reveal class-specific associations.

In [ ]:
ATTR_KEY = [c for c in [
    'attr_pos_neg_ratio', 'attr_pos_gini',
    'attr_ring07_mean', 'attr_ring00_mean',
    'attr_pos_centroid_offset', 'attr_dapi_pearson',
] if c in df.columns]

fig, axes_grid = plt.subplots(
    len(FEATURE_GROUPS), len(ATTR_KEY),
    figsize=(FW, len(FEATURE_GROUPS) * FW/len(ATTR_KEY) * 2.8/2.6),
)

for gi, (grp_name, feat_cols) in enumerate(FEATURE_GROUPS.items()):
    if not feat_cols:
        continue
    grp_col = GROUP_COLORS[grp_name]

    for aj, ac in enumerate(ATTR_KEY):
        ax = axes_grid[gi, aj]

        for cls, marker, alpha in [(positive_name, '^', 0.55),
                                    (negative_name, 'o', 0.3),
                                    ('Preleukemia', 's', 0.3)]:
            sub = df[df['_class'] == cls]
            mc_best = feat_cols[0]          # use first feature as representative
            # Find the feature in this group with highest |r| for this attr col
            best_r, best_mc = 0, feat_cols[0]
            for mc in feat_cols:
                valid = sub[[mc, ac]].dropna()
                if len(valid) > 8:
                    r, _ = spearmanr(valid[mc], valid[ac])
                    if abs(r) > abs(best_r):
                        best_r, best_mc = r, mc

            valid = sub[[best_mc, ac]].dropna()
            if len(valid) < 5:
                ax.text(0.5, 0.5, 'n<5', ha='center', va='center',
                        transform=ax.transAxes, fontsize=8, color='gray')
                continue

            ax.scatter(valid[best_mc], valid[ac], s=6, alpha=alpha,
                       color=COLORS[cls], marker=marker, rasterized=True,
                       label=f'{cls}')
            r, p = spearmanr(valid[best_mc], valid[ac])
            m, b_int = np.polyfit(valid[best_mc], valid[ac], 1)
            xr = np.linspace(valid[best_mc].min(), valid[best_mc].max(), 80)
            ax.plot(xr, m * xr + b_int, color=COLORS[cls], lw=1.2, alpha=0.85)

        if gi == 0:
            ax.set_title(ac.replace('attr_', '').replace('_', '\n'),
                         fontsize=7.5, fontweight='bold')
        if aj == 0:
            ax.set_ylabel(grp_name, fontsize=8, color=grp_col, fontweight='bold')
        ax.tick_params(labelsize=6)
        ax.set_xlabel(best_mc.replace('_x',''), fontsize=6)

plt.suptitle('Best-feature × key-attribution correlations (per group per class)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "12b_group_attr_corr.pdf", bbox_inches="tight")
plt.show()


### 12.3 · Attribution correlation heatmaps per group

For each morphological group: Spearman r matrix between **all features in the group**  
and **all attribution features** (rings + summary stats).  
Dots mark p < 0.05.

In [ ]:
attr_all = [c for c in avail_rings + avail_summary if c in df.columns]
attr_labels = [c.replace('attr_','').replace('_mean','').replace('_x','')
               for c in attr_all]

n_groups = len(FEATURE_GROUPS)
fig, axes_grid = plt.subplots(1, n_groups,
                               figsize=(FW, 6 * FW/(n_groups * 4.5)))

for gi, (grp_name, feat_cols) in enumerate(FEATURE_GROUPS.items()):
    ax = axes_grid[gi]
    if not feat_cols:
        ax.axis('off')
        continue

    grp_col = GROUP_COLORS[grp_name]
    R = np.zeros((len(feat_cols), len(attr_all)))
    P = np.ones_like(R)

    for fi, fc in enumerate(feat_cols):
        for ai, ac in enumerate(attr_all):
            valid = df[[fc, ac]].dropna()
            if len(valid) > 10:
                r, p = spearmanr(valid[fc], valid[ac])
                R[fi, ai] = r
                P[fi, ai] = p

    vmax = max(0.3, np.abs(R).max())
    im = ax.imshow(R, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(attr_all)))
    ax.set_xticklabels(attr_labels, rotation=90, fontsize=5.5)
    ax.set_yticks(range(len(feat_cols)))
    ax.set_yticklabels([c.replace('_x','') for c in feat_cols], fontsize=7)
    ax.set_title(grp_name, fontsize=9, fontweight='bold', color=grp_col, pad=8)

    for fi in range(len(feat_cols)):
        for ai in range(len(attr_all)):
            if P[fi, ai] < 0.05:
                ax.text(ai, fi, '●', ha='center', va='center',
                        fontsize=4.5, color='white' if abs(R[fi,ai]) > 0.15 else 'black')

    plt.colorbar(im, ax=ax, fraction=0.06, label='Spearman r')

plt.suptitle('Per-group: morphological features × attribution features  (● p<0.05)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
_rows_12c = []
for _grp_name, _feat_cols in FEATURE_GROUPS.items():
    if not _feat_cols:
        continue
    for _fc in _feat_cols:
        for _ac in attr_all:
            _valid = df[[_fc, _ac]].dropna()
            if len(_valid) > 10:
                _r, _p = spearmanr(_valid[_fc], _valid[_ac])
                _rows_12c.append({'group': _grp_name, 'morph_feature': _fc.replace('_x',''),
                                   'attr_feature': _ac, 'spearman_r': _r, 'p_value': _p})
pd.DataFrame(_rows_12c).to_csv(OUTPUT_DIR / '12c_group_heatmaps_data.csv', index=False)

plt.savefig(OUTPUT_DIR / "12c_group_heatmaps.pdf", bbox_inches="tight")
plt.show()


### 12.4 · High-confidence AML cells — morphological signature per group

For AML cells, compare **high-confidence** (score ≥ threshold) vs **low-confidence** predictions.  
Each group is shown as a normalised z-score bar chart (group mean ± SEM) so all features are comparable.

In [ ]:
aml_sub = df[df['_class'] == positive_name].copy()
hi_aml   = aml_sub[aml_sub['_high_conf']]
lo_aml   = aml_sub[~aml_sub['_high_conf']]

print(f"AML high-conf: {len(hi_aml)}   low-conf: {len(lo_aml)}")

fig, axes_row = plt.subplots(1, len(FEATURE_GROUPS),
                              figsize=(FW, 10 * FW/(len(FEATURE_GROUPS) * 3.8)))

for ax, (grp_name, feat_cols) in zip(axes_row, FEATURE_GROUPS.items()):
    if not feat_cols:
        ax.axis('off')
        continue

    grp_col = GROUP_COLORS[grp_name]
    feats_ok = [fc for fc in feat_cols
                if hi_aml[fc].dropna().shape[0] >= 3
                and lo_aml[fc].dropna().shape[0] >= 3]
    if not feats_ok:
        ax.text(0.5, 0.5, 'insufficient data', ha='center', va='center',
                transform=ax.transAxes, fontsize=8, color='gray')
        continue

    # Z-score within AML so all features are comparable
    combined = aml_sub[feats_ok].copy()
    z_combined = combined.apply(zscore, nan_policy='omit')

    hi_z = z_combined.loc[hi_aml.index]
    lo_z = z_combined.loc[lo_aml.index]

    x     = np.arange(len(feats_ok))
    width = 0.35
    hi_m  = hi_z.mean()
    hi_s  = hi_z.sem()
    lo_m  = lo_z.mean()
    lo_s  = lo_z.sem()

    b1 = ax.bar(x - width/2, hi_m, width, color='#e31a1c', alpha=0.75,
                label=f'High-conf (n={len(hi_aml)})', yerr=hi_s, capsize=3,
                error_kw=dict(elinewidth=1))
    b2 = ax.bar(x + width/2, lo_m, width, color='#fb9a99', alpha=0.75,
                label=f'Low-conf (n={len(lo_aml)})', yerr=lo_s, capsize=3,
                error_kw=dict(elinewidth=1))

    # Significance star above each pair
    for xi, fc in enumerate(feats_ok):
        star, _ = mwu_stars(hi_aml[fc], lo_aml[fc])
        if star != 'ns':
            ymax = max(hi_m[fc] + hi_s[fc], lo_m[fc] + lo_s[fc])
            ax.text(xi, ymax + 0.08, star, ha='center', fontsize=10,
                    color=grp_col, fontweight='bold')

    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.set_xticks(x)
    xlabels_short = [fc.replace('_x','').replace('radial_','').replace('texture_','tex_')
                     for fc in feats_ok]
    ax.set_xticklabels(xlabels_short, rotation=40, ha='right', fontsize=7)
    ax.set_title(grp_name, fontsize=9, fontweight='bold', color=grp_col)
    ax.set_ylabel('Z-score (within AML)' if ax == axes_row[0] else '')
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Morphological profile: high-confidence vs low-confidence AML  (z-scored within class)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
_rows_12d = []
for _grp_name, _feat_cols in FEATURE_GROUPS.items():
    _feats_ok = [_fc for _fc in _feat_cols
                 if hi_aml[_fc].dropna().shape[0] >= 3 and lo_aml[_fc].dropna().shape[0] >= 3]
    if not _feats_ok:
        continue
    _z = aml_sub[_feats_ok].apply(zscore, nan_policy='omit')
    for _fc in _feats_ok:
        _rows_12d.append({'group': _grp_name, 'feature': _fc.replace('_x',''),
                           'high_conf_z_mean': _z.loc[hi_aml.index, _fc].mean(),
                           'low_conf_z_mean': _z.loc[lo_aml.index, _fc].mean(),
                           'n_high': len(hi_aml), 'n_low': len(lo_aml)})
pd.DataFrame(_rows_12d).to_csv(OUTPUT_DIR / '12d_high_conf_group_data.csv', index=False)

plt.savefig(OUTPUT_DIR / "12d_high_conf_group.pdf", bbox_inches="tight")
plt.show()


### 12.5 · Top discriminating feature per group (AML vs Control)

Effect size (rank-biserial correlation, i.e. normalised Mann-Whitney U) for the single most  
significant feature from each group.  A horizontal bar chart gives an at-a-glance comparison  
of which morphological category is most informative for class separation.

In [ ]:
from scipy.stats import mannwhitneyu as mwu_raw

def rank_biserial(a, b):
    # Effect size r = 1 - 2U / (n_a * n_b)
    a, b = a.dropna(), b.dropna()
    U, _ = mwu_raw(a, b, alternative='two-sided')
    r = 1 - 2 * U / (len(a) * len(b))
    _, p = mwu_raw(a, b, alternative='two-sided')
    return r, p

summary_rows = []
for grp_name, feat_cols in FEATURE_GROUPS.items():
    best_r, best_p, best_fc = 0, 1, None
    for fc in feat_cols:
        r, p = rank_biserial(pos[fc], neg[fc])
        if abs(r) > abs(best_r):
            best_r, best_p, best_fc = r, p, fc
    if best_fc:
        summary_rows.append({
            'Group': grp_name,
            'Best feature': best_fc.replace('_x',''),
            'Effect size (r)': best_r,
            'p-value': best_p,
        })

summ_df = pd.DataFrame(summary_rows).sort_values('Effect size (r)', key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Bar chart of effect sizes
colors_bar = [GROUP_COLORS[g] for g in summ_df['Group']]
bars = axes[0].barh(summ_df['Group'], summ_df['Effect size (r)'],
                    color=colors_bar, alpha=0.8, edgecolor='white')
axes[0].axvline(0, color='k', lw=0.8)
for bar, val in zip(bars, summ_df['Effect size (r)']):
    x_text = val + 0.01 if val >= 0 else val - 0.01
    ha = 'left' if val >= 0 else 'right'
    axes[0].text(x_text, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', ha=ha, fontsize=9, fontweight='bold')
axes[0].set_xlabel('Rank-biserial correlation (r)')
axes[0].set_title('Effect size — best feature per group\n(AML vs Control)', fontweight='bold')
axes[0].set_xlim(-1, 1)

# Scatter: best feature per group coloured by group
for _, row in summ_df.iterrows():
    fc_full = row['Best feature'] + '_x'
    if fc_full not in df.columns:
        fc_full = [c for c in df.columns if c.startswith(row['Best feature'])][0]
    for cls in CLASSES:
        sub = df[df['_class'] == cls]
        axes[1].scatter(
            sub[fc_full].dropna(),
            sub['attr_ring07_mean'].reindex(sub[fc_full].dropna().index),
            s=6, alpha=0.35, color=COLORS[cls], rasterized=True
        )
    r, p = spearmanr(df[fc_full].dropna(),
                     df['attr_ring07_mean'].reindex(df[fc_full].dropna().index))
    axes[1].text(0.02, 0.98, f'{row["Group"]}\n{row["Best feature"]}\nr={r:.2f}',
                 transform=axes[1].transAxes, va='top', fontsize=7.5,
                 color=GROUP_COLORS[row['Group']])
    break  # just show the top one

axes[1].set_xlabel('Feature value (top group)')
axes[1].set_ylabel('attr_ring07_mean (centre ring attribution)')
axes[1].set_title('Centre-ring attribution vs best group feature', fontweight='bold')

plt.suptitle('Summary: discriminability and attribution link per morphological group',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
summ_df.to_csv(OUTPUT_DIR / '12e_group_summary_data.csv', index=False)

plt.savefig(OUTPUT_DIR / "12e_group_summary.pdf", bbox_inches="tight")
plt.show()

print()
summ_df.style \
    .format({'Effect size (r)': '{:.3f}', 'p-value': '{:.2e}'}) \
    .background_gradient(subset=['Effect size (r)'], cmap='RdBu_r', vmin=-0.5, vmax=0.5)


---
## 13 · DAPI Signal at Positive vs Negative Occlusion Pixels

Rather than ring aggregates, we load the raw **128×128 occlusion heatmaps** for every cell  
and split pixels by the sign of their attribution:

- **Positive pixels** (`occ > 0`) — occluding that patch *raised* the AML prediction  
- **Negative pixels** (`occ < 0`) — occluding that patch *lowered* the AML prediction

We then measure the **mean DAPI intensity** under each pixel group.  
DAPI is z-scored within patient before pooling to remove staining variability.

In [ ]:
px_df.to_csv(OUTPUT_DIR / '13_dapi_pixel_stats.csv', index=False)

In [ ]:
# ── Figure 6SE: attribution magnitude AND area fraction vs DAPI contrast ───────
# Panel row A: magnitude panels (occ_pos_mean / occ_neg_mean vs dapi_diff)
# Panel row B: area-fraction panels (pos_area_frac / neg_area_frac vs dapi_diff)
# Both classes on same axes; patient-clustered Spearman r with 95% CI bootstrap.
# Saved as Figure_6SE.pdf (cells 66+67 merged; see cell 67 — now disabled).
from matplotlib.lines import Line2D

fig, axes = plt.subplots(2, 2, figsize=(FW, FW * 10 / 13))

# ── Row A: attribution magnitude ────────────────────────────────────────────
y_all_mag = px_df['dapi_diff'].dropna()
y_lim_mag = (np.nanpercentile(y_all_mag, 1) - 0.1, np.nanpercentile(y_all_mag, 99) + 0.1)
for ai, (xcol, side) in enumerate([
    ('occ_pos_mean', 'AML-supporting'),
    ('occ_neg_mean', 'AML-suppressing'),
]):
    ax = axes[0, ai]
    for i, cls in enumerate(CLASSES):
        sub = px_df[px_df['cls'] == cls].dropna(subset=[xcol, 'dapi_diff', 'patient'])
        r, ci_lo, ci_hi = cluster_bootstrap_r(
            sub[xcol].values, sub['dapi_diff'].values, sub['patient'].values)
        ax.scatter(sub[xcol], sub['dapi_diff'],
                   s=8, alpha=0.25, color=COLORS[cls], rasterized=True, label=cls,
                   marker=marker_class[cls])
        y_txt = 0.97 - i * 0.10
        ax.text(0.04, y_txt,
                f'{cls}: r={r:.2f} [{ci_lo:.2f},{ci_hi:.2f}]',
                transform=ax.transAxes, fontsize=8, color=COLORS[cls], va='top')
    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.axvline(0, color='k', lw=0.7, ls='--')
    ax.set_ylim(y_lim_mag)
    ax.set_xlabel(f'Mean {side.lower()} attribution magnitude')
    ax.set_ylabel('Chromatin density contrast\n(support − suppress regions; DAPI z-score)' if ai == 0 else '')
    ax.set_title(f'{side} magnitude\nvs chromatin density contrast', fontweight='bold')
    ax.legend(fontsize=8, markerscale=2, loc='lower right')

# ── Row B: area fraction ─────────────────────────────────────────────────────
px_merged['pos_area_frac'] = px_merged['n_pos_px'] / total_px
px_merged['neg_area_frac'] = px_merged['n_neg_px'] / total_px

y_all_frac = px_merged['dapi_diff'].dropna()
y_lim_frac = (np.nanpercentile(y_all_frac, 1) - 0.1, np.nanpercentile(y_all_frac, 99) + 0.1)
for ai, (xcol, side) in enumerate([
    ('pos_area_frac', 'Positive'),
    ('neg_area_frac', 'Negative'),
]):
    ax = axes[1, ai]
    for i, cls in enumerate(CLASSES):
        sub = px_merged[px_merged['cls'] == cls].dropna(
            subset=[xcol, 'dapi_diff', 'patient'])
        r, ci_lo, ci_hi = cluster_bootstrap_r(
            sub[xcol].values, sub['dapi_diff'].values, sub['patient'].values)
        ax.scatter(sub[xcol], sub['dapi_diff'],
                   s=8, alpha=0.25, color=COLORS[cls], rasterized=True, label=cls,
                   marker=marker_class[cls])
        y_txt = 0.97 - i * 0.10
        ax.text(0.04, y_txt,
                f'{cls}: r={r:.2f} [{ci_lo:.2f},{ci_hi:.2f}]',
                transform=ax.transAxes, fontsize=8, color=COLORS[cls], va='top')
    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.set_ylim(y_lim_frac)
    ax.set_xlabel(f'{side} area fraction  (n_{side.lower()[:3]}_px / total attributed px)')
    ax.set_ylabel('Chromatin density contrast \n(support − suppress regions; DAPI z-score)' if ai == 0 else '')
    ax.set_title(f'{side} area fraction\nvs chromatin density contrast', fontweight='bold')
    ax.legend(fontsize=8, markerscale=2, loc='lower right')

#plt.suptitle('Figure 6SE — Attribution magnitude (row A) and area fraction (row B) vs chromatin density contrast\n'
#             '95% CI patient-clustered Spearman r  |  magnitude and fraction test the same underlying'
#             ' circularity,\nneither is confound-elimination; both are consistent with the chromatin-density interpretation',
#s             fontsize=11, fontweight='bold', y=1.03)
plt.tight_layout()
_rows_6se = []
for _row_lbl, _xcol, _frame in [('magnitude_pos', 'occ_pos_mean', px_df),
                                 ('magnitude_neg', 'occ_neg_mean', px_df),
                                 ('area_frac_pos', 'pos_area_frac', px_merged),
                                 ('area_frac_neg', 'neg_area_frac', px_merged)]:
    for _cls in CLASSES:
        _sub = _frame[_frame['cls'] == _cls].dropna(subset=[_xcol, 'dapi_diff', 'patient'])
        _r, _lo, _hi = cluster_bootstrap_r(_sub[_xcol].values, _sub['dapi_diff'].values, _sub['patient'].values)
        _rows_6se.append({'panel': _row_lbl, 'class': _cls, 'r': _r, 'ci_lo': _lo, 'ci_hi': _hi, 'n': len(_sub)})
pd.DataFrame(_rows_6se).to_csv(OUTPUT_DIR / 'Figure_6SE_data.csv', index=False)

plt.savefig(OUTPUT_DIR / 'Figure_6SE.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── DAPI intensity vs DAPI contrast (pos−neg), patient-level clustering ───────
# X: overall cell DAPI intensity (weighted mean of pos/neg pixel groups, z-scored)
# Y: DAPI contrast = dapi_pos − dapi_neg (how much brighter the model-attended
#    region is vs the suppressed region, within the same cell)
# Small dots  = individual cells
# Large markers = patient-level means (▲ AML, ● Control)
# Cluster-bootstrapped Spearman r annotated per class

# Compute overall cell DAPI intensity (weighted by pixel count)
px_df['dapi_mean'] = (
    (px_df['dapi_pos'] * px_df['n_pos_px'] + px_df['dapi_neg'] * px_df['n_neg_px'])
    / (px_df['n_pos_px'] + px_df['n_neg_px'])
)

fig, axes = plt.subplots(1, len(CLASSES), figsize=(FW * len(CLASSES) / 2, FW*5/13))

# Shared axis limits
all_x = px_df['dapi_mean'].dropna()
all_y = px_df['dapi_diff'].dropna()
x_lim = (np.nanpercentile(all_x, 1) - 0.1, np.nanpercentile(all_x, 99) + 0.1)
y_lim = (np.nanpercentile(all_y, 1) - 0.1, np.nanpercentile(all_y, 99) + 0.1)

PAT_MARKERS = {'AML': '^', 'Control': 'o', 'Preleukemia': 's'}

for ai, cls in enumerate(CLASSES):
    ax = axes[ai]
    sub = px_df[px_df['cls'] == cls].dropna(subset=['dapi_mean', 'dapi_diff', 'patient'])

    # Individual cells
    ax.scatter(sub['dapi_mean'], sub['dapi_diff'],
               s=8, alpha=0.25, color=COLORS[cls], rasterized=True, label='Cell',
               marker=marker_class[cls])

    # Patient-level means
    pat_means = sub.groupby('patient')[['dapi_mean', 'dapi_diff']].mean().reset_index()
    ax.scatter(pat_means['dapi_mean'], pat_means['dapi_diff'],
               s=60, alpha=0.85, color=COLORS[cls],
               marker=PAT_MARKERS[cls], edgecolors='k', linewidths=0.6,
               zorder=5, label='Patient mean')

    # Cluster-bootstrapped Spearman r
    r, ci_lo, ci_hi = cluster_bootstrap_r(
        sub['dapi_mean'].values, sub['dapi_diff'].values, sub['patient'].values)
    ax.text(0.04, 0.97,
            f'r={r:.2f} [{ci_lo:.2f},{ci_hi:.2f}]\n'
            f'95% CI, patient-clustered\n'
            f'n={len(sub)} cells, {sub["patient"].nunique()} patients',
            transform=ax.transAxes, fontsize=8, va='top',
            linespacing=1.4)

    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.axvline(0, color='k', lw=0.7, ls='--')
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_xlabel('Overall relative chromatin density (DAPI z-score, within patient)')
    ax.set_ylabel('Local chromatin density contrast\n(support − suppress; DAPI z-score)' if ai == 0 else '')
    ax.set_title(f'{cls} cells', fontweight='bold', color=COLORS[cls])
    ax.legend(fontsize=8, loc='lower right', markerscale=1.2)

#plt.suptitle(
#    'Figure 6SG — Overall chromatin density vs local chromatin density contrast\n'
#    '(overall: mean DAPI z-score across full nucleus; local contrast: support − suppress regions)\n'
#    'Large markers = patient means  |  Small dots = individual cells',
#    fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
_rows_6sg = []
for _cls in CLASSES:
    _sub = px_df[px_df['cls'] == _cls].dropna(subset=['dapi_mean', 'dapi_diff', 'patient'])
    _r, _lo, _hi = cluster_bootstrap_r(_sub['dapi_mean'].values, _sub['dapi_diff'].values, _sub['patient'].values)
    _rows_6sg.append({'class': _cls, 'r': _r, 'ci_lo': _lo, 'ci_hi': _hi,
                       'n_cells': len(_sub), 'n_patients': _sub['patient'].nunique()})
pd.DataFrame(_rows_6sg).to_csv(OUTPUT_DIR / 'Figure_6SG_data.csv', index=False)

plt.savefig(OUTPUT_DIR / 'Figure_6SG.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure S6C: three-panel scatter ─────────────────────────────────────────
# B: AML-supporting attribution magnitude vs chromatin contrast
# C: AML-suppressing attribution magnitude vs chromatin contrast
# D: overall DAPI intensity vs chromatin contrast (both classes in one panel)
# Stats: r_obs = Spearman on all cells; CI = patient-clustered bootstrap.
# Patient means overlaid as large markers so the patient-level trend is visible.

PAT_MARKERS = {'AML': '^', 'Control': 'o', 'Preleukemia': 's'}

fig, axes = plt.subplots(1, 3, figsize=(FW*0.9, FW * 0.35), sharey=False)

y_lim = (np.nanpercentile(px_df['dapi_diff'].dropna(), 1) - 0.1,
          np.nanpercentile(px_df['dapi_diff'].dropna(), 99) + 0.1)
YLABEL = 'Chromatin density contrast\n(support − suppress; DAPI z-score)'

def _scatter_pat(ax, sub, xcol, cls, with_label=False):
    marker = PAT_MARKERS[cls]
    ax.scatter(sub[xcol], sub["dapi_diff"],
               s=7, alpha=0.18, color=COLORS[cls], rasterized=True,
               marker=marker, label=cls if with_label else None)
    pat_m = sub.groupby("patient")[[xcol, "dapi_diff"]].mean().reset_index()
    ax.scatter(pat_m[xcol], pat_m["dapi_diff"],
               s=45, alpha=0.85, color=COLORS[cls], marker=PAT_MARKERS[cls],
               edgecolors="k", linewidths=0.5, zorder=5,
               label="Patient mean" if with_label else None)

# ── Panel B: supporting magnitude ────────────────────────────────────────────
ax = axes[0]
xcol = 'occ_pos_mean'
for i, cls in enumerate(CLASSES):
    sub = px_df[px_df['cls'] == cls].dropna(subset=[xcol, 'dapi_diff', 'patient'])
    r, lo, hi = cluster_bootstrap_r(sub[xcol].values, sub["dapi_diff"].values,
                                     sub["patient"].values)
    _scatter_pat(ax, sub, xcol, cls, with_label=(i == 0))
    y_txt = 0.97 - i * 0.10
    ax.text(0.04, y_txt,
            f'{cls}: r={r:.2f} [{lo:.2f},{hi:.2f}] ({sub["patient"].nunique()} pts)',
            transform=ax.transAxes, fontsize=6.5, va="top")
ax.axhline(0, color="k", lw=0.6, ls="--")
ax.axvline(0, color="k", lw=0.6, ls="--")
ax.set_ylim(y_lim)
ax.set_xlabel('AML-supporting mean attribution', fontsize=7.5)
ax.set_ylabel(YLABEL, fontsize=7.5)
ax.set_title('Supporting magnitude', fontweight='bold', fontsize=8, loc='center')
ax.legend(fontsize=6.5, markerscale=1.4, loc='lower right')

# ── Panel C: suppressing magnitude ───────────────────────────────────────────
ax = axes[1]
xcol = 'occ_neg_mean'
for i, cls in enumerate(CLASSES):
    sub = px_df[px_df['cls'] == cls].dropna(subset=[xcol, 'dapi_diff', 'patient'])
    r, lo, hi = cluster_bootstrap_r(sub[xcol].values, sub["dapi_diff"].values,
                                     sub["patient"].values)
    _scatter_pat(ax, sub, xcol, cls)
    y_txt = 0.97 - i * 0.10
    ax.text(0.04, y_txt,
            f'{cls}: r={r:.2f} [{lo:.2f},{hi:.2f}] ({sub["patient"].nunique()} pts)',
            transform=ax.transAxes, fontsize=6.5, va="top")
ax.axhline(0, color="k", lw=0.6, ls="--")
ax.axvline(0, color="k", lw=0.6, ls="--")
ax.set_ylim(y_lim)
ax.set_xlabel('AML-suppressing mean attribution', fontsize=7.5)
ax.set_title('Suppressing magnitude', fontweight='bold', fontsize=8, loc='center')

# ── Panel D: overall DAPI intensity vs local contrast ────────────────────────
ax = axes[2]
xcol = 'dapi_mean'
for i, cls in enumerate(CLASSES):
    sub = px_df[px_df['cls'] == cls].dropna(subset=[xcol, 'dapi_diff', 'patient'])
    r, lo, hi = cluster_bootstrap_r(sub[xcol].values, sub["dapi_diff"].values,
                                     sub["patient"].values)
    _scatter_pat(ax, sub, xcol, cls)
    y_txt = 0.97 - i * 0.10
    ax.text(0.04, y_txt,
            f'{cls}: r={r:.2f} [{lo:.2f},{hi:.2f}] ({sub["patient"].nunique()} pts)',
            transform=ax.transAxes, fontsize=6.5, va="top")
ax.axhline(0, color="k", lw=0.6, ls="--")
ax.axvline(0, color="k", lw=0.6, ls="--")
ax.set_ylim(y_lim)
ax.set_xlabel('Overall DAPI (z-score, within patient)', fontsize=7.5)
ax.set_title('Overall DAPI vs contrast', fontweight='bold', fontsize=8, loc='center')

#plt.suptitle('Figure S6C — Small dots = cells · large markers = patient means · '
#             'r = cell-level Spearman · CI = patient-clustered bootstrap (2000 resamples)',
#             fontsize=8, fontweight="bold", y=1.03)
plt.tight_layout()
_rows_6sc = []
for _panel_lbl, _xcol in [('supporting_magnitude', 'occ_pos_mean'),
                          ('suppressing_magnitude', 'occ_neg_mean'),
                          ('overall_dapi', 'dapi_mean')]:
    for _cls in CLASSES:
        _sub = px_df[px_df['cls'] == _cls].dropna(subset=[_xcol, 'dapi_diff', 'patient'])
        _r, _lo, _hi = cluster_bootstrap_r(_sub[_xcol].values, _sub['dapi_diff'].values, _sub['patient'].values)
        _rows_6sc.append({'panel': _panel_lbl, 'class': _cls, 'r': _r, 'ci_lo': _lo, 'ci_hi': _hi,
                           'n_patients': _sub['patient'].nunique()})
pd.DataFrame(_rows_6sc).to_csv(OUTPUT_DIR / 'Figure_6SC_data.csv', index=False)

plt.savefig(OUTPUT_DIR / 'Figure_6SC.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig B: attribution magnitude vs area fraction — KDE contours ─────────────
# X: attribution area as a FRACTION OF NUCLEUS AREA (n_pos_px / (n_pos_px+n_neg_px)
#    and its complement), not the raw pixel count used before -- raw counts
#    confound with nucleus size, which we haven't verified is matched across
#    all four groups. Computed fresh here on px_merged: pos_area_frac was
#    previously only added to px_df AFTER px_merged was already built via
#    .merge(), so it never actually reached px_merged -- recomputing locally
#    from n_pos_px/n_neg_px (both present pre-merge) avoids that stale-column bug.
# Y: mean magnitude (occ_pos_mean / |occ_neg_mean|)
# Contours per class reveal hotspot (small area, high mag) vs diffuse
# (large area, low mag) regimes. All four classes (Control, Preleukemia, MRD,
# AML) via ALL_CLASSES, matching cell 20's pattern -- MRD rows are already in
# px_merged (df was concatenated with mrd at cell 13, well before px_df/px_merged
# were built at cell 25).
# Bold outlined markers = per-patient means (one per patient), overlaid on the
# KDE -- addresses the "cells aren't independent, patients are" concern
# directly on the plot rather than only in the stats. No per-cell scatter
# (too dense/messy to read); the KDE contour carries the per-cell density.
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

ALL_CLASSES = [c for c in COLORS if c in px_merged['cls'].unique()]
print(f"Classes plotted: {ALL_CLASSES}")

px_merged = px_merged.copy()
_total_px = px_merged['n_pos_px'] + px_merged['n_neg_px']
px_merged['pos_area_frac'] = px_merged['n_pos_px'] / _total_px
px_merged['neg_area_frac'] = px_merged['n_neg_px'] / _total_px

fig, axes = plt.subplots(1, 2, figsize=(FW, FW*5/13))

for ai, (area_col, mag_col, sign, panel_title, xlabel, ylabel) in enumerate([
    ('pos_area_frac', 'occ_pos_mean', 'AML-supporting',
     'AML-supporting regions',
     'Spatial extent of supporting region\n(fraction of nucleus area)',
     'Mean attribution per supporting pixel'),
    ('neg_area_frac', 'occ_neg_mean', 'AML-suppressing',
     'AML-suppressing regions',
     'Spatial extent of suppressing region\n(fraction of nucleus area)',
     'Mean attribution magnitude per suppressing pixel'),
]):
    ax = axes[ai]
    all_x = px_merged[area_col].dropna()
    all_y = (px_merged[mag_col].abs() if sign == 'AML-suppressing' else px_merged[mag_col]).dropna()
    x_lim = (np.nanpercentile(all_x, 1), np.nanpercentile(all_x, 99))
    y_lim = (np.nanpercentile(all_y, 1), np.nanpercentile(all_y, 99))

    for cls in ALL_CLASSES:
        sub = px_merged[px_merged['cls'] == cls].dropna(subset=[area_col, mag_col]).copy()
        sub['_mag'] = sub[mag_col].abs() if sign == 'AML-suppressing' else sub[mag_col]

        # KDE contour (cell-level density), evaluated over this class's own
        # data range (padded) rather than the shared/pooled grid -- avoids
        # drawing contours where this class has no data. Levels are fractions
        # of this class's own peak density, not a bare count, so the
        # threshold adapts regardless of how skewed the distribution is.
        x_vals, y_vals = sub[area_col].values, sub['_mag'].values
        xy = np.vstack([x_vals, y_vals])
        if xy.shape[1] > 30:
            kde = gaussian_kde(xy, bw_method=0.3)
            x_pad = 0.1 * (x_vals.max() - x_vals.min())
            y_pad = 0.1 * (y_vals.max() - y_vals.min())
            xg = np.linspace(x_vals.min() - x_pad, x_vals.max() + x_pad, 60)
            yg = np.linspace(y_vals.min() - y_pad, y_vals.max() + y_pad, 60)
            Xg, Yg = np.meshgrid(xg, yg)
            Zg = kde(np.vstack([Xg.ravel(), Yg.ravel()])).reshape(Xg.shape)
            levels = [f * Zg.max() for f in (0.15, 0.35, 0.60, 0.85)]
            ax.contour(Xg, Yg, Zg, levels=levels, colors=[COLORS[cls]],
                       linewidths=1.2, alpha=0.7)

        # Per-patient means (bold, outlined, on top)
        pat_means = sub.groupby('patient')[[area_col, '_mag']].mean()
        ax.scatter(pat_means[area_col], pat_means['_mag'],
                   s=45, color=COLORS[cls], edgecolor='black', linewidth=0.7,
                   marker=marker_class.get(cls, 'o'), zorder=5)

    ax.set_xlim(x_lim); ax.set_ylim(y_lim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(panel_title, fontweight='bold')

    # Legend: one colour-coded line per class (KDE contour), plus a marker
    # key explaining the bold outlined dots (patient means).
    class_handles = [Line2D([0], [0], color=COLORS[cls], lw=1.8, label=cls)
                      for cls in ALL_CLASSES]
    patient_handle = Line2D([0], [0], marker='o', linestyle='none',
                             markerfacecolor='none', markeredgecolor='black',
                             markersize=6, label='Patient mean')
    ax.legend(handles=class_handles + [patient_handle], fontsize=8, loc='best')

plt.suptitle('Spatial extent versus attribution strength of AML-supporting'
             ' and -suppressing regions',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
pd.concat([
    px_merged[px_merged['cls'] == _cls][['image_name', 'cls', 'patient', 'pos_area_frac',
                                          'occ_pos_mean', 'neg_area_frac', 'occ_neg_mean']]
    for _cls in ALL_CLASSES
], ignore_index=True).to_csv(OUTPUT_DIR / 'Figure_6SA_data.csv', index=False)

plt.savefig(OUTPUT_DIR / 'Figure_6SA.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig B (shared-axis variant): same as above, but both panels share one
# y-axis (fraction of nucleus area -- pos_area_frac and neg_area_frac are
# complements of the same 0-1 scale, so a shared axis is directly
# comparable) and one shared legend instead of a per-panel legend.
# Axis labels use "mean attribution magnitude" consistently on both panels
# (sign/direction is already conveyed by the panel title), since
# occ_neg_mean is negative by construction and only meaningful after abs().
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

ALL_CLASSES = [c for c in COLORS if c in px_merged['cls'].unique()]

px_merged = px_merged.copy()
_total_px = px_merged['n_pos_px'] + px_merged['n_neg_px']
px_merged['pos_area_frac'] = px_merged['n_pos_px'] / _total_px
px_merged['neg_area_frac'] = px_merged['n_neg_px'] / _total_px

panels = [
    ('pos_area_frac', 'occ_pos_mean', 'AML-supporting',
     'AML-supporting regions',
     'Mean attribution magnitude (per supporting pixel)',
     'Fraction of nucleus area'),
    ('neg_area_frac', 'occ_neg_mean', 'AML-suppressing',
     'AML-suppressing regions',
     'Mean attribution magnitude (per suppressing pixel)',
     'Fraction of nucleus area'),
]

# Shared y-limits across both panels (pos_area_frac and neg_area_frac pooled)
_all_area = pd.concat([px_merged['pos_area_frac'], px_merged['neg_area_frac']]).dropna()
y_lim = (np.nanpercentile(_all_area, 0), np.nanpercentile(_all_area, 99))

fig, axes = plt.subplots(1, 2, figsize=(FW, FW*5/13), sharey=True)

for ai, (area_col, mag_col, sign, panel_title, xlabel, ylabel) in enumerate(panels):
    ax = axes[ai]
    all_x = (px_merged[mag_col].abs() if sign == 'AML-suppressing' else px_merged[mag_col]).dropna()
    x_lim = (-0.2, np.nanpercentile(all_x, 99))

    for cls in ALL_CLASSES:
        sub = px_merged[px_merged['cls'] == cls].dropna(subset=[area_col, mag_col]).copy()
        sub['_mag'] = sub[mag_col].abs() if sign == 'AML-suppressing' else sub[mag_col]

        # KDE contour (cell-level density) -- x = attribution magnitude, y = area
        # fraction. Evaluated over this class's own data range (padded)
        # rather than the shared/pooled grid -- avoids drawing contours
        # where this class has no data. Levels are fractions of this
        # class's own peak density, not a bare count, so the threshold
        # adapts regardless of how skewed the distribution is.
        x_vals, y_vals = sub['_mag'].values, sub[area_col].values
        xy = np.vstack([x_vals, y_vals])
        if xy.shape[1] > 30:
            kde = gaussian_kde(xy, bw_method=0.3)
            x_pad = 0.1 * (x_vals.max() - x_vals.min())
            y_pad = 0.1 * (y_vals.max() - y_vals.min())
            xg = np.linspace(x_vals.min() - x_pad, x_vals.max() + x_pad, 60)
            yg = np.linspace(y_vals.min() - y_pad, y_vals.max() + y_pad, 60)
            Xg, Yg = np.meshgrid(xg, yg)
            Zg = kde(np.vstack([Xg.ravel(), Yg.ravel()])).reshape(Xg.shape)
            levels = [f * Zg.max() for f in (0.15, 0.35, 0.60, 0.85)]
            ax.contour(Xg, Yg, Zg, levels=levels, colors=[COLORS[cls]],
                       linewidths=1.2, alpha=0.7)

        # Per-patient means (bold, outlined, on top)
        pat_means = sub.groupby('patient')[['_mag', area_col]].mean()
        ax.scatter(pat_means['_mag'], pat_means[area_col],
                   s=45, color=COLORS[cls], edgecolor='black', linewidth=0.7,
                   marker=marker_class.get(cls, 'o'), zorder=5)

    ax.set_xlim(x_lim); ax.set_ylim(y_lim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(panel_title, fontweight='bold')

# One shared legend for the whole figure (class colours + patient-mean marker)
class_handles = [Line2D([0], [0], color=COLORS[cls], lw=1.8, label=cls)
                  for cls in ALL_CLASSES]
patient_handle = Line2D([0], [0], marker='o', linestyle='none',
                         markerfacecolor='none', markeredgecolor='black',
                         markersize=6, label='Patient mean')
fig.legend(handles=class_handles + [patient_handle], fontsize=8,
           loc='upper center', bbox_to_anchor=(0.5, 1.08),
           ncol=len(class_handles) + 1, frameon=False)

plt.suptitle('Spatial extent versus attribution strength of AML-supporting'
             ' and -suppressing regions\n(shared y-axis and legend)',
             fontsize=11, fontweight='bold', y=1.18)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'Figure_6SA_sharedaxis.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Sanity check: Control subset used in the left panel (pos_area_frac vs occ_pos_mean) ──
_sub_control = px_merged[px_merged['cls'] == 'Control'].dropna(subset=['pos_area_frac', 'occ_pos_mean'])
print(f"n = {len(_sub_control)}")
_sub_control['occ_pos_mean'].describe()

In [ ]:
# plot the occlsuion attribution for AML-class map (target=1, always AML) for AML vs Control in different panels using the same approach as above


# plot the third in separate figure for clarity separating AML vs Control in different panels
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors

# Shared axis limits across both classes
all_x, all_y = [], []
for cls in CLASSES:
    sub_pos = px_df[px_df['cls'] == cls].dropna(subset=['occ_aml_pos_median', 'dapi_raw_pos'])
    sub_neg = px_df[px_df['cls'] == cls].dropna(subset=['occ_aml_neg_median', 'dapi_raw_neg'])
    all_x.extend(sub_pos['occ_aml_pos_median'].tolist() + sub_neg['occ_aml_neg_median'].tolist())
    all_y.extend(sub_pos['dapi_raw_pos'].tolist() + sub_neg['dapi_raw_neg'].tolist())

x_lo, x_hi = np.nanpercentile(all_x, 1), np.nanpercentile(all_x, 99)
y_lo, y_hi = np.nanpercentile(all_y, 1), np.nanpercentile(all_y, 99)
x_pad, y_pad = (x_hi - x_lo) * 0.05, (y_hi - y_lo) * 0.05
shared_xlim = (x_lo - x_pad, x_hi + x_pad)
shared_ylim = (y_lo - y_pad, y_hi + y_pad)

marker_legend = [
    Line2D([0], [0], marker='o', color='gray', ls='None', ms=6, label='AML-supporting pixels'),
    Line2D([0], [0], marker='^', color='gray', ls='None', ms=6, label='AML-suppressing pixels'),
]

CMAPS = {'AML': 'Reds', 'Control': 'Blues', 'Preleukemia': 'Greens'}

fig, axes = plt.subplots(1, len(CLASSES), figsize=(7 * len(CLASSES), 5))
for ai, cls in enumerate(CLASSES):
    ax = axes[ai]
    cmap = plt.colormaps[CMAPS[cls]]

    # Use cells that have both pos and neg data
    sub = px_df[px_df['cls'] == cls].dropna(
        subset=['occ_aml_pos_median', 'occ_aml_neg_median', 'dapi_raw_pos', 'dapi_raw_neg', 'dapi_diff'])

    # Normalise dapi_diff to [0.25, 0.95] of the colormap to avoid near-white shades
    norm = mcolors.Normalize(
        vmin=sub['dapi_diff'].quantile(0.02),
        vmax=sub['dapi_diff'].quantile(0.98))
    cell_colors = cmap(0.25 + 0.70 * norm(sub['dapi_diff'].values))

    ax.scatter(sub['occ_aml_pos_median'], sub['dapi_raw_pos'],
               c=cell_colors, marker='o', s=10, alpha=0.6, rasterized=True)
    ax.scatter(sub['occ_aml_neg_median'], sub['dapi_raw_neg'],
               c=cell_colors, marker='^', s=10, alpha=0.6, rasterized=True)

    r_p, ci_p_lo, ci_p_hi = cluster_bootstrap_r(
        sub['occ_aml_pos_median'].values, sub['dapi_raw_pos'].values, sub['patient'].values)
    r_n, ci_n_lo, ci_n_hi = cluster_bootstrap_r(
        sub['occ_aml_neg_median'].values, sub['dapi_raw_neg'].values, sub['patient'].values)
    annot = (f'r_pos={r_p:.2f} [{ci_p_lo:.2f},{ci_p_hi:.2f}]\n'
             f'r_neg={r_n:.2f} [{ci_n_lo:.2f},{ci_n_hi:.2f}]\n'
             f'95% CI, patient-clustered')
    ax.text(0.04, 0.97, annot,
            transform=ax.transAxes, fontsize=8, va='top',
            linespacing=1.4)

    ax.axvline(0, color='k', lw=0.7, ls='--')
    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)
    ax.set_xlabel('Mean occlusion attribution')
    #ax.set_ylabel(DAPI_LBL if ai == 0 else '')
    ax.set_title(f'{cls} cells', fontweight='bold', color=COLORS[cls])
    ax.legend(handles=marker_legend, fontsize=7.5, loc='lower right')

    # Per-class colorbar (colour = DAPI contrast per cell)
    sm = plt.cm.ScalarMappable(
        cmap=cmap,
        norm=mcolors.Normalize(
            vmin=sub['dapi_diff'].quantile(0.02),
            vmax=sub['dapi_diff'].quantile(0.98)))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02,
                 label='DAPI contrast per cell\n(pos - neg pixels, z-score)')

plt.suptitle('Relative chromatin density vs attribution magnitude\n'
             '(colour = per-cell DAPI contrast, shared across both panels)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
_rows_13b = []
for _cls in CLASSES:
    _sub = px_df[px_df['cls'] == _cls].dropna(
        subset=['occ_aml_pos_median', 'occ_aml_neg_median', 'dapi_raw_pos', 'dapi_raw_neg', 'dapi_diff'])
    _r_p, _lo_p, _hi_p = cluster_bootstrap_r(_sub['occ_aml_pos_median'].values, _sub['dapi_raw_pos'].values, _sub['patient'].values)
    _r_n, _lo_n, _hi_n = cluster_bootstrap_r(_sub['occ_aml_neg_median'].values, _sub['dapi_raw_neg'].values, _sub['patient'].values)
    _rows_13b.append({'class': _cls, 'r_pos': _r_p, 'ci_lo_pos': _lo_p, 'ci_hi_pos': _hi_p,
                       'r_neg': _r_n, 'ci_lo_neg': _lo_n, 'ci_hi_neg': _hi_n, 'n': len(_sub)})
pd.DataFrame(_rows_13b).to_csv(OUTPUT_DIR / '13b_dapi_pixel_pos_neg_separate_AML_data.csv', index=False)

plt.savefig(OUTPUT_DIR / '13b_dapi_pixel_pos_neg_separate_AML.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── EXCLUDED — not included in manuscript ──────────────────────────────────
# Results (span ns p=0.07, net *** p=4.76e-137) stated directly in text; net-attr panel duplicates Figure 6SE
# Keeping cell for reference; savefig disabled.
# ── Figure 13c: Attribution contrast (pos - neg) per cell, coloured by class ──
# attr_contrast = occ_pos_mean - occ_neg_mean
# occ_neg_mean is already negative, so this equals occ_pos_mean + |occ_neg_mean|:
# a large positive value means strong attribution in BOTH directions (high certainty).
# Signed version: occ_pos_mean + occ_neg_mean (net attribution; positive = pos dominates).

sub = px_df.dropna(subset=['occ_pos_mean', 'occ_neg_mean', 'dapi_diff'])
sub = sub.copy()
sub['attr_span']   = sub['occ_pos_mean'] - sub['occ_neg_mean']   # always > 0
sub['attr_net']    = sub['occ_pos_mean'] + sub['occ_neg_mean']   # signed net

fig, axes = plt.subplots(1, 3, figsize=(FW, FW*5/10))

# ── (0) Distribution of attribution span ─────────────────────────────────────
ax = axes[0]
for cls in CLASSES:
    vals = sub[sub['cls'] == cls]['attr_span']
    ax.hist(vals, bins=40, alpha=0.55, color=COLORS[cls], edgecolor='white',
            density=True, label=f'{cls}  med={vals.median():.3f}')
    # KDE overlay
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(vals.dropna())
    xr = np.linspace(vals.min(), vals.max(), 200)
    ax.plot(xr, kde(xr), color=COLORS[cls], lw=2)
_pat_col_76 = 'patient' if 'patient' in px_df.columns else 'patientID_x'
_aml_sp  = sub[sub['cls']=='AML'    ].groupby(_pat_col_76)['attr_span'].median()
_ctrl_sp = sub[sub['cls']=='Control'].groupby(_pat_col_76)['attr_span'].median()
star_s, p_s = mwu_stars(_aml_sp, _ctrl_sp)
ax.set_xlabel('AML attribution span  (supporting − suppressing per cell)')
ax.set_ylabel('Density')
ax.set_title(f'AML attribution span distribution\n{star_s}  p={p_s:.2e}', fontweight='bold', pad=50)
ax.legend(fontsize=9, bbox_to_anchor=(0., 1.02, 1., .102), loc=3,
       ncol=1, mode="expand", borderaxespad=0.)

# ── (1) Distribution of net attribution ──────────────────────────────────────
ax2 = axes[1]
for cls in CLASSES:
    vals = sub[sub['cls'] == cls]['attr_net']
    ax2.hist(vals, bins=40, alpha=0.55, color=COLORS[cls], edgecolor='white',
             density=True, label=f'{cls}  med={vals.median():.3f}')
    kde = gaussian_kde(vals.dropna())
    xr = np.linspace(vals.min(), vals.max(), 200)
    ax2.plot(xr, kde(xr), color=COLORS[cls], lw=2)
ax2.axvline(0, color='k', lw=1.2, ls='--')
_aml_nt  = sub[sub['cls']=='AML'    ].groupby(_pat_col_76)['attr_net'].median()
_ctrl_nt = sub[sub['cls']=='Control'].groupby(_pat_col_76)['attr_net'].median()
star_n, p_n = mwu_stars(_aml_nt, _ctrl_nt)
ax2.set_xlabel('Net AML attribution  (supporting + suppressing per cell)')
ax2.set_ylabel('Density')
ax2.set_title(f'Net AML attribution distribution\n{star_n}  p={p_n:.2e}', fontweight='bold', pad=50)
ax2.legend(fontsize=9, bbox_to_anchor=(0., 1.02, 1., .102), loc=3,
       ncol=1, mode="expand", borderaxespad=0.)

# ── (2) Scatter: net attribution vs DAPI contrast, coloured by class ─────────
ax3 = axes[2]
for i, cls in enumerate(CLASSES):
    s = sub[sub['cls'] == cls]
    ax3.scatter(s['attr_net'], s['dapi_diff'],
                s=10, alpha=0.45, color=COLORS[cls], rasterized=True,
                label=f'{cls}  (n={len(s)})', marker=marker_class[cls])
    r, p = spearmanr(s['attr_net'], s['dapi_diff'])
    # Trend line
    m, b = np.polyfit(s['attr_net'], s['dapi_diff'], 1)
    xr = np.linspace(s['attr_net'].quantile(0.01), s['attr_net'].quantile(0.99), 100)
    ax3.plot(xr, m*xr + b, color=COLORS[cls], lw=2, alpha=0.9)
    y_txt = 0.97 - i * 0.10
    ax3.text(0.04, y_txt, f'{cls}: r={r:.2f}  p={p:.2e}',
             transform=ax3.transAxes, fontsize=8, va='top')
ax3.axvline(0, color='k', lw=0.7, ls='--')
ax3.axhline(0, color='k', lw=0.7, ls='--')
ax3.set_xlabel('Net AML attribution per cell  (supporting + suppressing)')
ax3.set_ylabel('DAPI contrast per cell  (pos pixels - neg pixels, z-score)')
ax3.set_title('Net AML attribution vs DAPI contrast', fontweight='bold', pad=50)
ax3.legend(fontsize=9, bbox_to_anchor=(0., 1.02, 1., .102), loc=3,
       ncol=1, mode="expand", borderaxespad=0.)

plt.suptitle('AML attribution contrast (supporting − suppressing) per cell, coloured by class',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
_rows_s6k = []
for _cls in CLASSES:
    _s = sub[sub['cls'] == _cls]
    _r, _p = spearmanr(_s['attr_net'], _s['dapi_diff'])
    _rows_s6k.append({'class': _cls, 'attr_span_median': _s['attr_span'].median(),
                       'attr_net_median': _s['attr_net'].median(),
                       'r_net_vs_dapi_contrast': _r, 'p_value': _p, 'n': len(_s)})
pd.DataFrame(_rows_s6k).to_csv(OUTPUT_DIR / 'Figure_S6K_data.csv', index=False)

plt.savefig(OUTPUT_DIR / 'Figure_S6K.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXCLUDED — this figure (13d_dapi_class_relative) is displaced by
# the per-ring Spearman r figure (cell 12) which is now Figure_6SC.
# Retain code for reference; savefig is disabled.
# ══════════════════════════════════════════════════════════════════
# ── Figure 13d: DAPI at AML-supporting vs AML-suppressing pixels ───────────────
# occ > 0 = AML-supporting (promotes AML) for ALL cells — no sign flip for Control
# (Both classes use the AML-class occlusion map: target=1)

DAPI_LBL = 'DAPI intensity (z-score, within patient)'

fig, axes = plt.subplots(1, 2, figsize=(FW, FW*5/17))

# ── (0) Violin: DAPI at AML-supporting vs AML-suppressing pixels ───────────────
ax = axes[0]
groups, xlabels, xcolors = [], [], []
SUP_COL = '#2ca02c'
OPP_COL = '#ff7f0e'
for cls in CLASSES:
    sub = px_df[px_df['cls'] == cls]
    groups.append(sub['dapi_support'].dropna().values)
    xlabels.append(f'{cls}\nAML-support')
    xcolors.append(SUP_COL)
    groups.append(sub['dapi_oppose'].dropna().values)
    xlabels.append(f'{cls}\nAML-suppress')
    xcolors.append(OPP_COL)

parts = ax.violinplot(groups, showmedians=True, showextrema=False)
for body, col, lbl in zip(parts['bodies'], xcolors, xlabels):
    body.set_facecolor(col)
    body.set_alpha(0.55 if 'AML' in lbl else 0.35)
parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
ax.set_xticks(range(1, len(groups)+1))
ax.set_xticklabels(xlabels, fontsize=8)
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.set_ylabel(DAPI_LBL)
ax.set_title('DAPI at AML-supporting vs AML-suppressing pixels', fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=SUP_COL, alpha=0.7, label='AML-supporting (occ>0)'),
                    Patch(color=OPP_COL, alpha=0.7, label='AML-suppressing (occ<0)')],
          fontsize=8)
from scipy.stats import wilcoxon
for ci, cls in enumerate(CLASSES):
    sub = px_df[px_df['cls'] == cls].dropna(subset=['dapi_support', 'dapi_oppose'])
    # Paired test: each cell contributes one support and one oppose value
    diff = sub['dapi_support'] - sub['dapi_oppose']
    _, p_w = wilcoxon(diff)
    star = '***' if p_w < 0.001 else '**' if p_w < 0.01 else '*' if p_w < 0.05 else 'ns'
    x1, x2 = ci*2+1, ci*2+2
    ymax = max(np.nanpercentile(sub['dapi_support'], 95),
               np.nanpercentile(sub['dapi_oppose'], 95)) * 1.03
    ax.plot([x1, x2], [ymax, ymax], color='k', lw=1.2)
    ax.text((x1+x2)/2, ymax*1.005, star, ha='center', fontsize=11, va='bottom')

# ── (1) Per-cell class DAPI contrast distribution ────────────────────────────
ax2 = axes[1]
from scipy.stats import gaussian_kde
for cls in CLASSES:
    vals = px_df[px_df['cls'] == cls]['dapi_class_diff'].dropna()
    ax2.hist(vals, bins=40, alpha=0.5, color=COLORS[cls], edgecolor='white',
             density=True, label=f'{cls}  med={vals.median():+.3f}')
    kde = gaussian_kde(vals)
    xr = np.linspace(vals.quantile(0.01), vals.quantile(0.99), 200)
    ax2.plot(xr, kde(xr), color=COLORS[cls], lw=2)
ax2.axvline(0, color='k', lw=1.5, ls='--')
# Patient-level medians — cells from the same patient are not independent
_pat_col_sc = 'patient' if 'patient' in px_df.columns else 'patientID_x'
_aml_pat  = px_df[px_df['cls']=='AML'    ].groupby(_pat_col_sc)['dapi_class_diff'].median()
_ctrl_pat = px_df[px_df['cls']=='Control'].groupby(_pat_col_sc)['dapi_class_diff'].median()
star_cd, p_cd = mwu_stars(_aml_pat, _ctrl_pat)
_n_aml_pat, _n_ctrl_pat = len(_aml_pat), len(_ctrl_pat)
ax2.set_xlabel('DAPI z-score  (AML-supporting − AML-suppressing pixels, per cell)')
ax2.set_ylabel('Density')
ax2.set_title(f'AML-class DAPI contrast (patient medians)\n{star_cd}  p={p_cd:.2e}  [n={_n_aml_pat} AML / {_n_ctrl_pat} Ctrl patients]', fontweight='bold')
ax2.legend(fontsize=9)

# ── (2) Scatter: class-relative DAPI contrast vs raw AML DAPI contrast ───────
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]
for cls in CLASSES:
    s = px_df[px_df['cls'] == cls].dropna(subset=['dapi_diff', 'dapi_class_diff'])
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]
# [PANEL 3 REMOVED — tautology: same map vs itself, r=1.00]

_rows_13d = []
for _cls in CLASSES:
    _sub = px_df[px_df['cls'] == _cls]
    _rows_13d.append({'class': _cls,
                       'dapi_support_mean': _sub['dapi_support'].mean(),
                       'dapi_oppose_mean': _sub['dapi_oppose'].mean(),
                       'dapi_class_diff_mean': _sub['dapi_class_diff'].mean(),
                       'dapi_class_diff_median': _sub['dapi_class_diff'].median(),
                       'n': len(_sub)})
pd.DataFrame(_rows_13d).to_csv(OUTPUT_DIR / '13d_dapi_class_relative_data.csv', index=False)

plt.suptitle('DAPI signal at AML-supporting vs AML-suppressing occlusion regions\n'
             '(AML-supporting: occ > 0 promotes AML; AML-suppressing: occ < 0)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
# EXCLUDED — displaced by cell 12 (02d_ring_dapi_attr_spearman → Figure_6SC.pdf)
# plt.savefig(OUTPUT_DIR / 'Figure_6SC.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# save the per-cell DAPI contrast data to CSV for further analysis
px_df.to_csv(OUTPUT_DIR / '13b_per_cell_dapi_contrast.csv', index=False)
print(f"Saved per-cell DAPI contrast data to {OUTPUT_DIR / '13b_per_cell_dapi_contrast.csv'}")

In [ ]:
# ── EXCLUDED — not included in manuscript ──────────────────────────────────
# QC/transparency item → supplementary table (Table S7) if retained; not a figure supporting a specific claim
# Keeping cell for reference; savefig disabled.
# plot per-patient summary of DAPI contrast
pat_summary = px_df.groupby('patient').agg(
    n_cells=('image_name', 'count'),
    mean_dapi_diff=('dapi_diff', 'mean'),
    sem_dapi_diff=('dapi_diff', lambda x: x.std() / np.sqrt(len(x))),
).reset_index()
# filter to patients with at least 5 cells
pat_summary = pat_summary[pat_summary['n_cells'] >= 5].sort_values('mean_dapi_diff', ascending=False)
# plot AML and Control patients in different colors 
pat_summary['color'] = pat_summary['patient'].apply(lambda x: '#d62728' if x.startswith('AML') else '#1f77b4')
fig, ax = plt.subplots(figsize=(10, 6))
for i, (_, row) in enumerate(pat_summary.iterrows()):
    ax.errorbar(row['mean_dapi_diff'], i,
                xerr=row['sem_dapi_diff'], fmt='o', capsize=5, capthick=2,
                color=row['color'], alpha=0.7, markersize=8)
ax.set_ylabel('Patient', fontsize=10)
ax.set_xlabel('Mean DAPI contrast (z-score, pos - neg)', fontsize=10)
ax.set_yticks(range(len(pat_summary)))
ax.set_yticklabels(pat_summary['patient'], fontsize=8)
ax.axvline(0, color='k', lw=0.7, ls='--')
ax.set_title('Per-Patient DAPI Contrast Summary (≥5 cells)', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'Figure_S6L.pdf', bbox_inches='tight')
plt.show()



In [ ]:
# save per patient DAPI contrast summary to CSV
pat_summary.to_csv(OUTPUT_DIR / 'Table_S7.csv', index=False)
print(f"Saved per-patient DAPI contrast summary to {OUTPUT_DIR / 'Table_S7.csv'}")

In [ ]:
# ── Figure 13j: 2-D occlusion heatmap + concentric ring overlay (compact) ──────
from scipy import ndimage
from skimage import filters, morphology
from skimage.measure import label, regionprops

PAD = 5   # dark border width (pixels) added before thresholding

mpl.rcParams['font.family'] = 'Arial'
plt.rc('font', size=8)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['axes.linewidth'] = 0.8

_POS_COLOR = COLORS['AML']      # AML-supporting
_NEG_COLOR = COLORS['Control']  # AML-suppressing


def _extract_cell_mask_bg(gray: np.ndarray, sigma: float = 1.5, pad: int = PAD) -> np.ndarray:
    """Exact replica of extract_cell_mask(..., mask_type='bg') from ainu_image_selection."""
    padded    = np.pad(gray, pad, mode='constant', constant_values=0)
    smoothed  = filters.gaussian(padded, sigma=sigma)
    thresh_bg = smoothed.max() * 0.2
    binary    = smoothed > thresh_bg
    binary    = ndimage.binary_fill_holes(binary)
    binary    = morphology.remove_small_objects(binary, min_size=int(0.01 * binary.size))
    labeled   = label(binary)
    if labeled.max() == 0:
        return binary[pad:-pad, pad:-pad].astype(bool)
    largest   = max(regionprops(labeled), key=lambda r: r.area).label
    mask      = ndimage.binary_fill_holes(labeled == largest)
    return mask[pad:-pad, pad:-pad].astype(bool)


def _ring_map(mask: np.ndarray, n_rings: int = 8, pad: int = PAD) -> np.ndarray:
    """
    Normalized-EDT ring labels: -1 = background, 0 = boundary, n-1 = centre.
    Matches occlusion_features() in ainu_image_selection: linspace(0,1,n+1) bins.
    """
    padded_mask = np.pad(mask, pad, mode='constant', constant_values=False)
    dist        = ndimage.distance_transform_edt(padded_mask)[pad:-pad, pad:-pad]
    max_d       = float(dist[mask].max()) if mask.any() else 1.0
    dist_n      = dist / max_d
    bin_edges   = np.linspace(0, 1, n_rings + 1)
    rmap        = np.full(mask.shape, -1, dtype=np.int8)
    for i in range(n_rings):
        shell = mask & (dist_n >= bin_edges[i]) & (dist_n <= bin_edges[i + 1])
        rmap[shell] = i
    return rmap


def plot_occ_rings(name, cls, axs, n_rings=8, vabs=None, pad=PAD):
    """Three panels: DAPI+rings | Occ+rings | per-ring bar chart."""
    occ = np.load(resolve_occ_path(name))[0]
    H, W = occ.shape
    gray = np.array(
        Image.open(IMG_DIR / f'{name}.png').convert('L').resize((W, H), Image.BILINEAR),
        dtype=np.float32,
    )
    if vabs is None:
        vabs = np.percentile(np.abs(occ), 99)

    mask   = _extract_cell_mask_bg(gray / 255.0, pad=pad)
    rmap   = _ring_map(mask, n_rings=n_rings)
    levels = np.arange(-1.5, n_rings + 0.5, 1)
    _norm  = TwoSlopeNorm(vcenter=0, vmin=-vabs, vmax=vabs)

    # Panel 1: DAPI + coloured ring boundaries
    ring_colors = [plt.cm.tab10(i / max(n_rings - 1, 1)) for i in range(n_rings + 1)]
    axs[0].imshow(gray, cmap='gray', vmin=0, vmax=255, origin='upper')
    axs[0].contour(rmap, levels=levels, colors=ring_colors, linewidths=0.9, alpha=0.9)
    axs[0].set_title(f'{cls}  |  DAPI + rings', fontsize=8,
                     fontweight='bold', color=COLORS[cls])
    axs[0].axis('off')

    # Panel 2: Occlusion heatmap + black ring boundaries
    axs[1].imshow(occ, cmap=CMAP_OCC, norm=_norm, origin='upper')
    axs[1].contour(rmap, levels=levels, colors='k', linewidths=0.75, alpha=0.6)
    axs[1].set_title('AML attribution + rings', fontsize=8, fontweight='bold')
    axs[1].axis('off')

    # Panel 3: per-ring positive and negative attribution grouped bars
    pos_means = [
        float(np.maximum(occ[rmap == r], 0).mean()) if (rmap == r).any() else 0.0
        for r in range(n_rings)
    ]
    neg_means = [
        float(abs(np.minimum(occ[rmap == r], 0).mean())) if (rmap == r).any() else 0.0
        for r in range(n_rings)
    ]
    x     = np.arange(n_rings)
    width = 0.38
    axs[2].bar(x - width / 2, pos_means, width, color=_POS_COLOR, edgecolor='k',
               linewidth=0.5, label='AML-supporting')
    axs[2].bar(x + width / 2, neg_means, width, color=_NEG_COLOR, edgecolor='k',
               linewidth=0.5, label='AML-suppressing')
    axs[2].axhline(0, color='k', lw=0.8, ls='--')
    axs[2].set_xticks(x)
    xlabels = (['R0\n(boundary)']
               + [f'R{i}' for i in range(1, n_rings - 1)]
               + [f'R{n_rings-1}\n(centre)'])
    axs[2].set_xticklabels(xlabels, fontsize=7)
    axs[2].set_ylabel('Mean attribution magnitude per ring', fontsize=7)
    axs[2].set_xlabel('Ring', fontsize=7)
    axs[2].set_title('AML attribution per ring', fontsize=8, fontweight='bold')
    axs[2].tick_params(axis='y', labelsize=7)
    axs[2].legend(fontsize=7, loc='upper left')

# Pick the cell with the richest mix of positive AND negative attribution.
ex_2d = {}
for cls in CLASSES:
    pool = px_df[px_df['cls'] == cls].dropna(subset=['occ_pos', 'occ_neg']).copy()
    pool['_bal_score'] = np.sqrt(pool['occ_pos'] * pool['occ_neg'])
    ex_2d[cls] = pool.nlargest(1, '_bal_score')['image_name'].iloc[0]

ex_2d['AML'] = 'AML29_0229'
ex_2d['Control'] = 'CTRL02_0109'
shared_vabs_2d = np.percentile(
    [np.percentile(np.abs(np.load(resolve_occ_path(n))[0]), 99)
     for n in ex_2d.values()],
    50,
)

pd.DataFrame([{'class': c, 'example_image_name': n} for c, n in ex_2d.items()]) \
    .to_csv(OUTPUT_DIR / 'Figure_6AB_example_cells.csv', index=False)

fig, axes = plt.subplots(
    len(CLASSES), 3,
    figsize=(7, len(CLASSES) * 2),
    gridspec_kw={'wspace': 0.12, 'hspace': 0.40},
)

# Control row (ri=0) is the more illustrative example for paper panels A/B:
# under the AML-class map the Control profile has the more dramatic curve
# (strong AML-suppressing attribution), while the AML trough is shallow.
for ri, cls in enumerate(CLASSES):
    plot_occ_rings(ex_2d[cls], cls, axes[ri],
                   n_rings=N_RINGS, vabs=shared_vabs_2d)

sm = plt.cm.ScalarMappable(
    cmap=CMAP_OCC,
    norm=TwoSlopeNorm(vcenter=0, vmin=-shared_vabs_2d, vmax=shared_vabs_2d),
)
sm.set_array([])
fig.colorbar(sm, ax=axes[:, 2], fraction=0.04, pad=0.04,
             label='AML-class attribution')
plt.tight_layout()
plt.suptitle(
    '2-D occlusion heatmap with concentric ring overlay\n'
    f'R0 = outermost boundary  and  R{N_RINGS-1} = nucleus centre',
    fontsize=11, fontweight='bold',
)
plt.savefig(OUTPUT_DIR / 'Figure_6AB.pdf', bbox_inches='tight')
plt.show()
    

In [ ]:
# ── Figure 13j: 2-D occlusion heatmap + concentric ring overlay (large) ───────
from scipy import ndimage
from skimage import filters, morphology
from skimage.measure import label, regionprops

PAD = 5   # dark border width (pixels) added before thresholding

mpl.rcParams['font.family'] = 'Arial'
plt.rc('font', size=8)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['axes.linewidth'] = 0.8

_POS_COLOR = COLORS['AML']      # AML-supporting
_NEG_COLOR = COLORS['Control']  # AML-suppressing


def _extract_cell_mask_bg(gray: np.ndarray, sigma: float = 1.5, pad: int = PAD) -> np.ndarray:
    """Exact replica of extract_cell_mask(..., mask_type='bg') from ainu_image_selection."""
    padded    = np.pad(gray, pad, mode='constant', constant_values=0)
    smoothed  = filters.gaussian(padded, sigma=sigma)
    thresh_bg = smoothed.max() * 0.2
    binary    = smoothed > thresh_bg
    binary    = ndimage.binary_fill_holes(binary)
    binary    = morphology.remove_small_objects(binary, min_size=int(0.01 * binary.size))
    labeled   = label(binary)
    if labeled.max() == 0:
        return binary[pad:-pad, pad:-pad].astype(bool)
    largest   = max(regionprops(labeled), key=lambda r: r.area).label
    mask      = ndimage.binary_fill_holes(labeled == largest)
    return mask[pad:-pad, pad:-pad].astype(bool)


def _ring_map(mask: np.ndarray, n_rings: int = 8, pad: int = PAD) -> np.ndarray:
    """
    Normalized-EDT ring labels: -1 = background, 0 = boundary, n-1 = centre.
    Matches occlusion_features() in ainu_image_selection: linspace(0,1,n+1) bins.
    """
    padded_mask = np.pad(mask, pad, mode='constant', constant_values=False)
    dist        = ndimage.distance_transform_edt(padded_mask)[pad:-pad, pad:-pad]
    max_d       = float(dist[mask].max()) if mask.any() else 1.0
    dist_n      = dist / max_d
    bin_edges   = np.linspace(0, 1, n_rings + 1)
    rmap        = np.full(mask.shape, -1, dtype=np.int8)
    for i in range(n_rings):
        shell = mask & (dist_n >= bin_edges[i]) & (dist_n <= bin_edges[i + 1])
        rmap[shell] = i
    return rmap


def plot_occ_rings(name, cls, axs, n_rings=8, vabs=None, pad=PAD):
    """Three panels: DAPI+rings | Occ+rings | per-ring bar chart."""
    occ = np.load(resolve_occ_path(name))[0]
    H, W = occ.shape
    gray = np.array(
        Image.open(IMG_DIR / f'{name}.png').convert('L').resize((W, H), Image.BILINEAR),
        dtype=np.float32,
    )
    if vabs is None:
        vabs = np.percentile(np.abs(occ), 99)

    mask   = _extract_cell_mask_bg(gray / 255.0, pad=pad)
    rmap   = _ring_map(mask, n_rings=n_rings)
    levels = np.arange(-1.5, n_rings + 0.5, 1)
    _norm  = TwoSlopeNorm(vcenter=0, vmin=-vabs, vmax=vabs)

    # Panel 1: DAPI + coloured ring boundaries
    ring_colors = [plt.cm.tab10(i / max(n_rings - 1, 1)) for i in range(n_rings + 1)]
    axs[0].imshow(gray, cmap='gray', vmin=0, vmax=255, origin='upper')
    axs[0].contour(rmap, levels=levels, colors=ring_colors, linewidths=0.9, alpha=0.9)
    axs[0].set_title(f'{cls}  |  DAPI + rings', fontsize=8,
                     fontweight='bold', color=COLORS[cls])
    axs[0].axis('off')

    # Panel 2: Occlusion heatmap + black ring boundaries
    axs[1].imshow(occ, cmap=CMAP_OCC, norm=_norm, origin='upper')
    axs[1].contour(rmap, levels=levels, colors='k', linewidths=0.75, alpha=0.6)
    axs[1].set_title('AML attribution + rings', fontsize=8, fontweight='bold')
    axs[1].axis('off')

    # Panel 3: per-ring positive and negative attribution grouped bars
    pos_means = [
        float(np.maximum(occ[rmap == r], 0).mean()) if (rmap == r).any() else 0.0
        for r in range(n_rings)
    ]
    neg_means = [
        float(abs(np.minimum(occ[rmap == r], 0).mean())) if (rmap == r).any() else 0.0
        for r in range(n_rings)
    ]
    x     = np.arange(n_rings)
    width = 0.38
    axs[2].bar(x - width / 2, pos_means, width, color=_POS_COLOR, edgecolor='k',
               linewidth=0.5, label='AML-supporting')
    axs[2].bar(x + width / 2, neg_means, width, color=_NEG_COLOR, edgecolor='k',
               linewidth=0.5, label='AML-suppressing')
    axs[2].axhline(0, color='k', lw=0.8, ls='--')
    axs[2].set_xticks(x)
    xlabels = (['R0\n(boundary)']
               + [f'R{i}' for i in range(1, n_rings - 1)]
               + [f'R{n_rings-1}\n(centre)'])
    axs[2].set_xticklabels(xlabels, fontsize=7)
    axs[2].set_ylabel('Mean attribution magnitude per ring', fontsize=7)
    axs[2].set_xlabel('Ring', fontsize=7)
    axs[2].set_title('AML attribution per ring', fontsize=8, fontweight='bold')
    axs[2].tick_params(axis='y', labelsize=7)
    axs[2].legend(fontsize=7, loc='upper left')

# Pick the cell with the richest mix of positive AND negative attribution.
ex_2d = {}
for cls in CLASSES:
    pool = px_df[px_df['cls'] == cls].dropna(subset=['occ_pos', 'occ_neg']).copy()
    pool['_bal_score'] = np.sqrt(pool['occ_pos'] * pool['occ_neg'])
    ex_2d[cls] = pool.nlargest(1, '_bal_score')['image_name'].iloc[0]

ex_2d['AML'] = 'AML29_0229'
ex_2d['Control'] = 'CTRL02_0109'
shared_vabs_2d = np.percentile(
    [np.percentile(np.abs(np.load(resolve_occ_path(n))[0]), 99)
     for n in ex_2d.values()],
    50,
)

pd.DataFrame([{'class': c, 'example_image_name': n} for c, n in ex_2d.items()]) \
    .to_csv(OUTPUT_DIR / 'Figure_6AB_example_cells.csv', index=False)

fig, axes = plt.subplots(
    len(CLASSES), 3,
    figsize=(15, len(CLASSES) * 3.5),
    gridspec_kw={'wspace': 0.12, 'hspace': 0.40},
)

# Control row (ri=0) is the more illustrative example for paper panels A/B:
# under the AML-class map the Control profile has the more dramatic curve
# (strong AML-suppressing attribution), while the AML trough is shallow.
for ri, cls in enumerate(CLASSES):
    plot_occ_rings(ex_2d[cls], cls, axes[ri],
                   n_rings=N_RINGS, vabs=shared_vabs_2d)

sm = plt.cm.ScalarMappable(
    cmap=CMAP_OCC,
    norm=TwoSlopeNorm(vcenter=0, vmin=-shared_vabs_2d, vmax=shared_vabs_2d),
)
sm.set_array([])
fig.colorbar(sm, ax=axes[:, 2], fraction=0.04, pad=0.04,
             label='AML-class attribution')
plt.tight_layout()
plt.suptitle(
    '2-D occlusion heatmap with concentric ring overlay\n'
    f'R0 = outermost boundary  and  R{N_RINGS-1} = nucleus centre',
    fontsize=11, fontweight='bold',
)
plt.savefig(OUTPUT_DIR / 'Figure_6AB_large.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── Statistical summary ───────────────────────────────────────────────────────
rows = []
for cls in CLASSES:
    sub = px_df[px_df['cls'] == cls].dropna(subset=['dapi_pos','dapi_neg'])
    star, p = mwu_stars(sub['dapi_pos'], sub['dapi_neg'])
    rows.append({
        'Class':                    cls,
        'N cells':                  len(sub),
        'DAPI z-score (AML-supporting pixels)': sub['dapi_pos'].mean(),
        'DAPI z-score (AML-suppressing pixels)': sub['dapi_neg'].mean(),
        'Δ z-score (AML-support − suppress)': sub['dapi_diff'].mean(),
        'Median Δ':                 sub['dapi_diff'].median(),
        '% cells Δ>0':              (sub['dapi_diff'] > 0).mean() * 100,
        'MWU p-value':              p,
        'sig':                      star,
    })

summary_px = pd.DataFrame(rows).set_index('Class')
summary_px.style \
    .format({'DAPI z-score (AML-supporting pixels)': '{:.3f}',
             'DAPI z-score (AML-suppressing pixels)': '{:.3f}',
             'Δ z-score (pos − neg)':    '{:.3f}',
             'Median Δ':                 '{:.3f}',
             '% cells Δ>0':             '{:.1f}',
             'MWU p-value':              '{:.2e}'}) \
    .background_gradient(subset=['Δ z-score (AML-support − suppress)'], cmap='RdBu_r', vmin=-0.5, vmax=0.5)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA EXPORT — save underlying figure data as CSVs for external analysis
# All files written to OUTPUT_DIR
# ══════════════════════════════════════════════════════════════════════════════
DATA_DIR = OUTPUT_DIR / 'exported_data'
DATA_DIR.mkdir(exist_ok=True)

# ── 1. Full per-cell feature table ────────────────────────────────────────────
keep_cols = ['_class', 'source_x', patient_col] + avail_rings + avail_rings_pos + avail_rings_neg + avail_summary + [c for c in MORPH_COLS if c in df.columns]
df[[c for c in keep_cols if c in df.columns]].to_csv(DATA_DIR / '01_per_cell_features_all.csv', index=False)
print('Saved 01_per_cell_features_all.csv')

# ── 2. Ring attribution profiles — mean ± SEM + MWU per class ────────────────
# Patient-median MWU for AML vs Control ring attribution
ring_rows = []
for ring_set, label in [(avail_rings, 'total'), (avail_rings_pos, 'positive'), (avail_rings_neg, 'negative')]:
    aml_pat_r  = df[df['_class']=='AML'    ].groupby(patient_col)[ring_set].median()
    ctrl_pat_r = df[df['_class']=='Control'].groupby(patient_col)[ring_set].median()
    for i, col in enumerate(ring_set):
        star, p = mwu_stars(aml_pat_r[col].dropna(), ctrl_pat_r[col].dropna())
        for cls in CLASSES:
            sub = df[df['_class'] == cls]
            vals = sub[col].dropna()
            ring_rows.append({
                'class': cls, 'attribution_type': label, 'ring': i,
                'cell_mean': vals.mean(), 'cell_sem': vals.sem(), 'cell_median': vals.median(),
                'n_cells': len(vals), 'mwu_p_pat': p, 'mwu_sig_pat': star,
            })
pd.DataFrame(ring_rows).to_csv(DATA_DIR / '02_ring_attribution_profiles_all.csv', index=False)
print('Saved 02_ring_attribution_profiles_all.csv')

# ── 3. Attribution summary stats — mean/median/MWU per feature per class ──────
sum_rows = []
for col in avail_summary:
    aml_pat_s  = df[df['_class']=='AML'    ].groupby(patient_col)[col].median()
    ctrl_pat_s = df[df['_class']=='Control'].groupby(patient_col)[col].median()
    star, p = mwu_stars(aml_pat_s.dropna(), ctrl_pat_s.dropna())
    for cls in CLASSES:
        vals = df[df['_class'] == cls][col].dropna()
        sum_rows.append({
            'feature': col, 'class': cls,
            'cell_mean': vals.mean(), 'cell_median': vals.median(), 'cell_std': vals.std(),
            'n_cells': len(vals), 'mwu_p_pat': p, 'mwu_sig_pat': star,
        })
pd.DataFrame(sum_rows).to_csv(DATA_DIR / '03_attribution_summary_stats_all.csv', index=False)
print('Saved 03_attribution_summary_stats_all.csv')

# ── 4. Morphology × attribution correlation matrix ────────────────────────────
avail_morph_exp = [c for c in MORPH_COLS if c in df.columns]
attr_for_corr_exp = [c for c in avail_rings + avail_summary if c in df.columns]
from scipy.stats import spearmanr as _sp
corr_rows = []
for mc in avail_morph_exp:
    for ac in attr_for_corr_exp:
        valid = df[[mc, ac]].dropna()
        if len(valid) > 10:
            r, p = _sp(valid[mc], valid[ac])
            corr_rows.append({'morph_feature': mc.replace('_x',''), 'attr_feature': ac, 'spearman_r': r, 'p_value': p})
pd.DataFrame(corr_rows).to_csv(DATA_DIR / '04_morph_attr_correlations_all.csv', index=False)
print('Saved 04_morph_attr_correlations_all.csv')

# ── 5. DAPI ring intensity × attribution correlations (Fig 05) ────────────────
int_rings_exp = [c for c in INT_RINGS if c in df.columns]
ring_int_rows = []
for i, (ir, ar) in enumerate(zip(int_rings_exp, avail_rings)):
    for cls in CLASSES:
        sub = df[df['_class'] == cls][[ir, ar]].dropna()
        if len(sub) < 5: continue
        r, p = _sp(sub[ir], sub[ar])
        ring_int_rows.append({'ring': i, 'class': cls, 'spearman_r': r, 'p_value': p,
                               'dapi_mean': sub[ir].mean(), 'attr_mean': sub[ar].mean()})
pd.DataFrame(ring_int_rows).to_csv(DATA_DIR / '05_dapi_ring_attr_correlations_all.csv', index=False)
print('Saved 05_dapi_ring_attr_correlations_all.csv')

# ── 6. Patient-level ring attribution ─────────────────────────────────────────
if patient_col in df.columns:
    pat_exp = (df.groupby([patient_col, '_class'])[avail_rings].mean().reset_index())
    pat_exp.to_csv(DATA_DIR / '06_patient_ring_attribution_all.csv', index=False)
    print('Saved 06_patient_ring_attribution_all.csv')

# ── 7. Mutation group summary (AML only) ──────────────────────────────────────
avail_mut_exp = [c for c in MUT_COLS if c in df.columns]
mut_rows = []
for mc in avail_mut_exp:
    flag = pos[mc].fillna(0).astype(bool)
    wt, mut_cells = pos[~flag], pos[flag]
    for col in avail_rings + avail_summary:
        if col not in pos.columns: continue
        star, p = mwu_stars(wt[col], mut_cells[col])
        mut_rows.append({
            'mutation': mc, 'feature': col,
            'WT_mean': wt[col].mean(), 'MUT_mean': mut_cells[col].mean(),
            'delta': mut_cells[col].mean() - wt[col].mean(),
            'n_WT': len(wt[col].dropna()), 'n_MUT': len(mut_cells[col].dropna()),
            'mwu_p': p, 'mwu_sig': star,
        })
pd.DataFrame(mut_rows).to_csv(DATA_DIR / '07_mutation_group_stats_all.csv', index=False)
print('Saved 07_mutation_group_stats_all.csv')

# ── 8. BM vs PB attribution stats (AML only) ──────────────────────────────────
source_col_exp = 'source_x' if 'source_x' in df.columns else 'source'
if source_col_exp in df.columns:
    pos_bm_exp = pos[pos[source_col_exp] == 'BM']
    pos_pb_exp = pos[pos[source_col_exp] == 'PB']
    src_rows = []
    for col in avail_rings + avail_summary:
        if col not in pos.columns: continue
        star, p = mwu_stars(pos_bm_exp[col], pos_pb_exp[col])
        src_rows.append({
            'feature': col,
            'BM_mean': pos_bm_exp[col].mean(), 'PB_mean': pos_pb_exp[col].mean(),
            'delta_BM_minus_PB': pos_bm_exp[col].mean() - pos_pb_exp[col].mean(),
            'n_BM': len(pos_bm_exp[col].dropna()), 'n_PB': len(pos_pb_exp[col].dropna()),
            'mwu_p': p, 'mwu_sig': star,
        })
    pd.DataFrame(src_rows).to_csv(DATA_DIR / '08_bm_vs_pb_stats_all.csv', index=False)
    print('Saved 08_bm_vs_pb_stats_all.csv')

# ── 9. High vs low confidence stats ───────────────────────────────────────────
conf_rows = []
for cls in CLASSES:
    sub = df[df['_class'] == cls]
    hi, lo = sub[sub['_high_conf']], sub[~sub['_high_conf']]
    for col in avail_rings + avail_summary + [c for c in MORPH_COLS if c in df.columns]:
        if col not in sub.columns: continue
        star, p = mwu_stars(hi[col], lo[col])
        conf_rows.append({
            'class': cls, 'feature': col,
            'high_conf_mean': hi[col].mean(), 'low_conf_mean': lo[col].mean(),
            'delta': hi[col].mean() - lo[col].mean(),
            'n_high': len(hi[col].dropna()), 'n_low': len(lo[col].dropna()),
            'mwu_p': p, 'mwu_sig': star,
        })
pd.DataFrame(conf_rows).to_csv(DATA_DIR / '09_high_vs_low_conf_stats_all.csv', index=False)
print('Saved 09_high_vs_low_conf_stats_all.csv')

# ── 10. DAPI at positive vs negative occlusion pixels (pixel-level, Fig 13) ───
if 'px_df' in dir():
    px_df.drop(columns=[c for c in px_df.columns if c.startswith('dapi_raw')], errors='ignore') \
         .to_csv(DATA_DIR / '10_dapi_at_occlusion_pixels_all.csv', index=False)
    print('Saved 10_dapi_at_occlusion_pixels_all.csv')
else:
    print('px_df not found — run Section 13 first')

print(f'\nAll exports written to: {DATA_DIR}')
